# 🧪 Automated Psychometric Scale Development Pipeline

**Single-notebook master pipeline.** Each phase reads the previous phase's Excel output and writes a richer Excel file — creating a complete, auditable chain from construct definition to a human-review-ready item pool.

---

## Background

This pipeline was originally conceived and developed under the name **Project Majdal** by **Tariq Shaban (Miraai Ltd)** as a practical implementation of emerging methods in AI-assisted psychometric scale development. The architecture draws directly on the pseudo-factor analysis (PFA) framework introduced by Guenole et al. (2024), the methodological evaluation of PFA by Suárez-Álvarez et al. (2026), the forced-choice measurement survey by Lee et al. (2025a), and the AI-powered item generation framework by Lee et al. (2025b).

> Guenole, N., Samo, A., & Sun, T. (2024). *Pseudo-discrimination parameters from language embeddings*. Open Science Framework. https://doi.org/10.31234/osf.io/9a4qx

> Guenole, N., D'Urso, E. D., Samo, A., & Sun, T. (2024). Pseudo factor analysis of language embedding similarity matrices: New ways to model latent constructs.

> Suárez-Álvarez, J., He, Q., Guenole, N., & D'Urso, D. (2026). Using Artificial Intelligence in Test Construction: A Practical Guide. *Psicothema*, *38*(1), 1-12.

> Milano, N., Luongo, M., Ponticorvo, M., & Marocco, D. (2025). Semantic analysis of test items through large language model embeddings predicts a-priori factorial structure of personality tests. *Current Research in Behavioral Sciences*, *8*, 100168.

> Lee, P., Son, M., & Jia, Z. (2025). AI-powered automatic item generation for psychological tests: A conceptual framework for an LLM-based multi-agent AIG system. *Journal of Business and Psychology*, 1-29.

> Lee, P., Son, M., Zhou, S., Joo, S., Jia, Z., & Cheng, V. (2025). The journey of forced choice measurement over 80 years: Past, present, and future. *Organizational Research Methods*, *28*(4), 680-722.

---

## Pipeline Architecture

```
scales.json (input)
    │
    ├── PHASE 1 │ Behavioral Domain Generation and Validation → 01_behavioral_domains.xlsx
    ├── PHASE 2 │ Item Generation (dominance items)           → 02_generated_items.xlsx
    ├── PHASE 3 │ Readability & Bias Analysis                 → 03_readability_bias.xlsx
    ├── PHASE 4 │ Content Validity (LLM SMEs)                 → 04_content_validity.xlsx
    └── PHASE 5 │ Pseudo-Factor Analysis and Final Review     → 05_pseudo_factor_analysis.xlsx
```

---

## Human Review Integration

Each phase that applies an automated pass/fail system also writes a **`human_review_pass`** column (default `True`) and a **`human_comments`** column. Reviewers open the Excel output after each phase and:

1. Change any `human_review_pass` value to `False` to override a system decision.
2. Add free-text reasoning in `human_comments`.

The **next phase always reads `human_review_pass`** as its processing filter — overriding the system `process_flag` where a human has intervened. This gives human experts full veto power at every stage without requiring any code changes.

---

## Before Running

1. Copy `.env_template` → `.env` and fill in `OPENROUTER_API_KEY`
2. Ensure `pipeline_settings.json` and `scales.json` are in the same folder as this notebook
3. Install requirements (Cell 1 does this automatically)
4. Run **Section 0 — Setup** before any other section
5. Each subsequent phase can be re-run independently provided its input Excel file exists

---

*Pipeline version: Revised April 2026 — dominance item framework*  
*Author: Tariq Shaban — Miraai Ltd & reviewed by Dr. Philseok Lee — George Mason University*


In [ ]:
import subprocess, sys

_packages = [
    "textstat", "openai", "pandas", "openpyxl",
    "scikit-learn", "sentence-transformers",
    "factor-analyzer", "python-dotenv", "tqdm"
]

process = subprocess.Popen(
    [sys.executable, "-m", "pip", "install"] + _packages,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in process.stdout:
    print(line, end="")

process.wait()
print(f"\nReturn code: {process.returncode}")

# ─── SECTION 0: Setup & Shared Utilities ───────────────────────────────────

In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import os, json, time, re, logging, warnings
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional
from dataclasses import dataclass, asdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from sentence_transformers import SentenceTransformer

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from dotenv import load_dotenv
import textstat
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine
from openai import OpenAI

warnings.filterwarnings("ignore")

# ── Working directory — always the folder containing this notebook ─────────────
def _get_notebook_dir() -> Path:
    # VS Code sets this variable automatically
    if "__vsc_ipynb_file__" in globals():
        return Path(__vsc_ipynb_file__).parent
    # Jupyter Lab / Notebook classic — try ipynbname
    try:
        import ipynbname
        return ipynbname.path().parent
    except Exception:
        pass
    # Final fallback — current working directory
    return Path.cwd()

NOTEBOOK_DIR = _get_notebook_dir()
os.chdir(NOTEBOOK_DIR)
print(f"✓ Working directory: {Path.cwd()}")

# ── Load .env ─────────────────────────────────────────────────────────────────
load_dotenv()

# ── Load pipeline_settings.json ───────────────────────────────────────────────
SETTINGS_FILE = Path("pipeline_settings.json")
assert SETTINGS_FILE.exists(), (
    f"Cannot find pipeline_settings.json in {Path.cwd()}. "
    "Make sure it is in the same folder as the notebook."
)
with open(SETTINGS_FILE, "r", encoding="utf-8") as f:
    CFG = json.load(f)

# ── Output directory ──────────────────────────────────────────────────────────
BASE_OUTPUT_DIR = Path(os.getenv("BASE_OUTPUT_DIR", CFG["project"]["base_output_dir"]))
BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Logging ───────────────────────────────────────────────────────────────────
log_level = getattr(logging, CFG["project"].get("log_level", "INFO"))
logging.basicConfig(
    level=log_level,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger("majdal")

# ── OpenRouter client ─────────────────────────────────────────────────────────
OPENROUTER_API_KEY  = os.getenv("OPENROUTER_API_KEY", "")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
assert OPENROUTER_API_KEY, (
    "OPENROUTER_API_KEY is not set. "
    "Add it to your .env file: OPENROUTER_API_KEY=sk-or-..."
)

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
    default_headers={
        "HTTP-Referer": "https://inubilum.io",
        "X-Title": os.getenv("OPENROUTER_APP_NAME", "Project Majdal")
    }
)

# ── Shared LLM call with exponential-backoff retry ────────────────────────────
def llm_call(
    prompt: str,
    model: str,
    max_tokens: int = 1000,
    temperature: float = 0.7,
    retries: int = 3,
    system: str = "You are a psychometric expert assistant."
) -> str:
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user",   "content": prompt}
                ],
                max_tokens=max_tokens,
                temperature=temperature
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            wait = (2 ** attempt) + np.random.uniform(0, 1)
            logger.warning(
                f"LLM call failed (attempt {attempt+1}/{retries}): {e}. "
                f"Retrying in {wait:.1f}s"
            )
            time.sleep(wait)
    raise RuntimeError(f"LLM call failed after {retries} attempts")

# ── Text cleaning helper ──────────────────────────────────────────────────────
def clean_text(text: str) -> str:
    if not text:
        return text
    text = text.replace("**", "").replace("*", "")
    text = text.strip("[]")           # remove wrapping square brackets
    text = re.sub(r'^\[|\]$', '', text.strip())  # catch any remaining
    text = " ".join(text.split())
    return text.strip(".,;:!?- ")

# ── Scale normalisation ───────────────────────────────────────────────────────
SCALE_DEFAULTS = CFG["scale_defaults"]

def normalise_scale(s: Dict) -> Dict:
    """Map any scale JSON format to the standard field names used across all phases."""
    return {
        "scale_name":               s.get("scale_name")           or s.get("name", ""),
        "construct_definition":     s.get("construct_definition") or s.get("definition", ""),
        "high_anchor":              s.get("high_anchor")          or s.get("high_behaviors", ""),
        "low_anchor":               s.get("low_anchor")           or s.get("low_behaviors", ""),
        "measure_type":             s.get("measure_type")         or s.get("measure", "Trait"),
        "discriminant_description": s.get("discriminant_description", ""),
        "target_population":        s.get("target_population",    SCALE_DEFAULTS["target_population"]),
        "target_environment":       s.get("target_environment",   SCALE_DEFAULTS["target_environment"]),
        "reading_level_target":     s.get("reading_level_target", SCALE_DEFAULTS["reading_level_target"]),
        "response_format":          s.get("response_format",      SCALE_DEFAULTS["response_format"]),
        **{k: v for k, v in s.items() if k not in {
            "scale_name", "name", "construct_definition", "definition",
            "high_anchor", "high_behaviors", "low_anchor", "low_behaviors",
            "measure_type", "measure", "discriminant_description",
            "target_population", "target_environment",
            "reading_level_target", "response_format"
        }}
    }

# ── Load and normalise scales ─────────────────────────────────────────────────
scales_json_path     = Path(os.getenv("SCALES_JSON",     CFG["project"]["scales_json"]))
all_scales_json_path = Path(os.getenv("ALL_SCALES_JSON", CFG["project"]["all_scales_reference_json"]))

assert scales_json_path.exists(),     f"scales JSON not found: {scales_json_path}"
assert all_scales_json_path.exists(), f"all_scales JSON not found: {all_scales_json_path}"

with open(scales_json_path, "r", encoding="utf-8") as f:
    SCALES_DATA = json.load(f)
with open(all_scales_json_path, "r", encoding="utf-8") as f:
    ALL_SCALES_DATA = json.load(f)

TARGET_SCALES: List[Dict]  = [normalise_scale(s) for s in SCALES_DATA["scales"]]
ALL_SCALES_REF: List[Dict] = ALL_SCALES_DATA["scales"]

ALL_SCALES_LOOKUP: Dict[str, str] = {}
for s in ALL_SCALES_REF:
    name = s.get("name") or s.get("scale_name", "")
    defn = s.get("definition") or s.get("construct_definition", "")
    if name:
        ALL_SCALES_LOOKUP[name] = defn

ALL_SCALES_LIST_STR = "\n".join(
    f"- {name}: {defn[:120]}..." if len(defn) > 120 else f"- {name}: {defn}"
    for name, defn in ALL_SCALES_LOOKUP.items()
)

scale_params_lookup: Dict[str, Dict] = {s["scale_name"]: s for s in TARGET_SCALES}

# ── Shared sentence-transformer loader (used by Phase 2 dedup and Phase 5 PFA) ──
_st_model_cache: Dict[str, "SentenceTransformer"] = {}

def get_st_model(model_name: str) -> "SentenceTransformer":
    """Load a SentenceTransformer once per kernel session, cache thereafter."""
    if model_name not in _st_model_cache:
        logger.info(f"Loading sentence-transformer: {model_name}")
        _st_model_cache[model_name] = SentenceTransformer(model_name)
    return _st_model_cache[model_name]

# ── Summary ───────────────────────────────────────────────────────────────────
logger.info(f"✓ Loaded {len(TARGET_SCALES)} target scales from {scales_json_path.name}")
logger.info(f"✓ Loaded {len(ALL_SCALES_LOOKUP)} reference scales from {all_scales_json_path.name}")
logger.info(f"✓ Output directory: {BASE_OUTPUT_DIR.resolve()}")

print("\n🟢 Setup complete. Ready to run pipeline phases.")
print(f"\nTarget scales ({len(TARGET_SCALES)}):")
for s in TARGET_SCALES:
    print(f"  • {s['scale_name']}  |  env: {s['target_environment']}  |  "
          f"pop: {s['target_population'].get('age_range','?')} / "
          f"{s['target_population'].get('education_level','?')}")

# ─── PHASE 1: Behavioral Domain Generation and Validation ──────────────────

**Input:** `scales.json`  
**Output:** `01_behavioral_domains.xlsx`  
**Sheets:** `Behavioral_Domains`, `Validation_Summary`, `Scale_Parameters`, `Run_Log`

---

### What this phase does

Phase 1 reads each construct definition from `scales.json` and performs two operations in sequence:

1. **Generation** — the LLM generates observable behavioral examples — concrete descriptions of what someone high or low on the trait actually *does* in context. These examples form the generative substrate from which all downstream items are produced. The `occurrence_likelihood` field (low / moderate / high) reflects how frequently the behavior typically occurs in the target environment and population — **not** item extremity in the psychometric sense.

2. **Validation** — a second LLM pass evaluates each generated example for three validity criteria:
   - **Construct validity** — does the example genuinely reflect the target construct?
   - **Face validity** — would it appear relevant to target respondents?
   - **Discriminant validity** — is it distinct from the excluded constructs listed in `discriminant_description`?

Examples that fail any criterion have `process_flag` set to `False`. Only examples with a valid flag proceed to item writing (Phase 2).

### System pass/fail rules (from `pipeline_settings.json`)

| Setting | Role |
|---|---|
| `examples_per_construct` | Target number of examples per construct |
| `min_valid_examples_threshold` | Minimum valid examples required per scale before raising a warning |
| `validation_criteria` | The three criteria evaluated per example |
| `retry_attempts` | Retries on API failure |

### Human review columns added to output

| Column | Default | Purpose |
|---|---|---|
| `process_flag` | System-set | LLM validation result |
| `human_review_pass` | `True` | Override — set `False` to exclude an example the system passed, or correct a false failure |
| `human_comments` | *(blank)* | Record reasoning for any override |

**Phase 2 reads `human_review_pass`.** Review `01_behavioral_domains.xlsx → Behavioral_Domains` before running Phase 2. Pay particular attention to `validation_issues` for context on why the system flagged an example.


In [ ]:
P1            = CFG["phase_01_behavioral_domains"]
P1_GEN_MODEL  = P1["generation_model"]
P1_VAL_MODEL  = P1["validation_model"]
P1_OUT        = BASE_OUTPUT_DIR / P1["output_file"]

# ── Phase 1A: Behavioral Domain Generation ────────────────────────────────────

def generate_behavioral_examples(scale: Dict) -> List[Dict]:
    """Generate stratified behavioral domain examples for one scale."""

    prompt = f"""You are a psychometric expert mapping the behavioral domain structure of a personality construct.

Construct: {scale['scale_name']}
Definition: {scale['construct_definition']}
High pole: {scale.get('high_anchor', 'N/A')}
Low pole: {scale.get('low_anchor', 'N/A')}
What this construct is NOT: {scale.get('discriminant_description', 'N/A')}
Target population: {json.dumps(scale.get('target_population', {}))}
Target environment: {scale.get('target_environment', 'General')}

Your task is to identify {P1['examples_per_construct']} distinct BEHAVIORAL DOMAINS for this construct.

A behavioral domain is a broad category or theme of behavior through which the construct expresses itself —
NOT a specific incident, story, or scenario. Think of domains as the chapter headings of the construct,
each representing a different facet or arena where the trait manifests.

GOOD domain examples for Conscientiousness:
- Following rules and procedures
- Maintaining routines and schedules
- Attention to detail and accuracy
- Planning and organizing tasks
- Persisting through difficulty to complete work
- Meeting commitments and deadlines

BAD domain examples (too specific — these are incidents, not domains):
- An employee updates their project tracker daily
- A worker checks their email first thing every morning
- A person rewrites their report three times before submitting

Each domain should be:
- A short noun phrase (3–7 words), not a sentence
- Broad enough that multiple distinct survey items could be written from it
- Specific enough to produce items that are meaningfully different from other domains listed
- Grounded in observable behavior in {scale.get('target_environment', 'the target environment')}
- Clearly an expression of {scale['scale_name']} and not an adjacent construct

Assign an occurrence_likelihood level (high / moderate / low) reflecting
how commonly behaviors in this domain occur in the target environment. Distribute across levels.

Format each domain EXACTLY like this (one per block, blocks separated by ---):
EXAMPLE_ID: 1
BEHAVIOR: [domain name as a short noun phrase, 3–7 words]
LEVEL: [high / moderate / low]
RATIONALE: [why this domain is a valid expression of {scale['scale_name']}]
ENVIRONMENT_FIT: [why behaviors in this domain occur in {scale.get('target_environment', 'the target environment')}]
POPULATION_FIT: [why this domain is relevant to the target population]
---"""

    raw = llm_call(
        prompt, P1_GEN_MODEL,
        max_tokens  = P1["generation_max_tokens"],
        temperature = P1["generation_temperature"],
        retries     = P1["retry_attempts"]
    )

    examples = []
    blocks   = [b.strip() for b in raw.split("---") if b.strip()]

    for i, section in enumerate(blocks):
        data = {
            "scale_name":       scale["scale_name"],
            "construct":        scale.get("construct_definition", ""),
            "target_environment": scale.get("target_environment", ""),
            "target_population":  json.dumps(scale["target_population"]),
            "process_flag":       True,
            "validation_issues":  ""
        }

        for line in section.split("\n"):
            line = line.strip()
            for key, field in [
                ("EXAMPLE_ID:",      "example_id"),
                ("BEHAVIOR:",        "behavior_description"),
                ("LEVEL:",           "occurrence_likelihood"),
                ("RATIONALE:",       "rationale"),
                ("ENVIRONMENT_FIT:", "environment_relevance"),
                ("POPULATION_FIT:",  "population_relevance"),
            ]:
                if line.upper().startswith(key):
                    data[field] = clean_text(line[len(key):].strip())

        # Build ID from scale's id field + sequential domain number
        scale_id = scale.get("id", scale["scale_name"][:3].upper())
        if "example_id" not in data:
            data["example_id"] = f"{scale_id}_D{i+1:02d}"
        else:
            # Reformat whatever the LLM returned to match our convention
            data["example_id"] = f"{scale_id}_D{len(examples)+1:02d}"

        if "behavior_description" in data and data["behavior_description"]:
            examples.append(data)

    return examples


# ── Phase 1B: Domain Validation ───────────────────────────────────────────────

def validate_examples_for_scale(scale_name: str, examples: List[Dict], scale: Dict) -> Dict[str, Dict]:
    """Batch-validate up to 10 examples at once to reduce API calls."""

    # Build a numbered list so we can match by position if ID matching fails
    numbered = []
    for idx, e in enumerate(examples):
        numbered.append({
            "n":          idx + 1,
            "example_id": e.get("example_id", f"item_{idx+1}"),
            "behavior":   e.get("behavior_description", ""),
            "level":      e.get("occurrence_likelihood", "")
        })

    batch_str = json.dumps(numbered, indent=2)

    prompt = f"""Review these behavioral examples for the construct: {scale_name}

Construct Definition: {scale.get('construct_definition', '')}
What it is NOT: {scale.get('discriminant_description', 'N/A')}

For each example evaluate:
1. construct_validity: Does it truly measure the target construct?
2. face_validity: Does it appear relevant to target users?
3. discriminant_validity: Is it distinct from excluded constructs?

Examples to evaluate:
{batch_str}

You MUST return exactly {len(numbered)} JSON objects, one per example, separated by ---.
Use the EXACT example_id values provided above — do not shorten or modify them.

{{"example_id": "<exact id from above>", "valid": true/false, "issues": "<brief reason or empty string>"}}
---"""

    raw = llm_call(
        prompt, P1_VAL_MODEL,
        max_tokens  = P1["validation_max_tokens"],
        temperature = P1["validation_temperature"],
        retries     = P1["retry_attempts"]
    )

    # Parse results — try ID match first, fall back to positional match
    results = {}
    parsed_blocks = []
    for block in raw.split("---"):
        block = block.strip()
        if not block:
            continue
        try:
            cleaned = re.sub(r'```json|```', '', block).strip()
            obj = json.loads(cleaned)
            parsed_blocks.append(obj)
            # Primary match: exact example_id
            eid = str(obj.get("example_id", ""))
            if eid:
                results[eid] = {
                    "valid":  bool(obj.get("valid", True)),
                    "issues": obj.get("issues", "")
                }
        except Exception:
            pass

    # Fallback: positional match for any examples still unmatched
    for idx, e in enumerate(examples):
        eid = str(e.get("example_id", ""))
        if eid not in results and idx < len(parsed_blocks):
            obj = parsed_blocks[idx]
            results[eid] = {
                "valid":  bool(obj.get("valid", True)),
                "issues": obj.get("issues", "") + " [matched positionally]"
            }

    return results


# ── Run Phase 1 ───────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"PHASE 1: Behavioral Domain Generation and Validation")
print(f"Generation model : {P1_GEN_MODEL}")
print(f"Validation model : {P1_VAL_MODEL}")
print(f"Scales           : {len(TARGET_SCALES)}")
print(f"{'='*60}\n")

# ── Step 1A: Generate behavioral examples ─────────────────────────────────────
all_examples: List[Dict] = []
run_log_p1:   List[Dict] = []

for scale in tqdm(TARGET_SCALES, desc="Generating examples"):
    scale_name = scale["scale_name"]
    t0 = time.time()
    try:
        examples = generate_behavioral_examples(scale)
        all_examples.extend(examples)
        status = "success"
        n      = len(examples)
    except Exception as e:
        logger.error(f"Phase 1 generation failed for '{scale_name}': {e}")
        examples = []
        status   = f"error: {e}"
        n        = 0

    run_log_p1.append({
        "scale_name":         scale_name,
        "phase":              "generation",
        "examples_generated": n,
        "status":             status,
        "duration_s":         round(time.time() - t0, 2),
        "timestamp":          datetime.now().isoformat()
    })
    print(f"  {'✓' if n > 0 else '✗'} {scale_name}: {n} examples")

print(f"\n  Total examples generated: {len(all_examples)}")

# ── Step 1B: Validate behavioral examples ─────────────────────────────────────
print(f"\n{'─'*60}")
print(f"Validating examples...")
print(f"{'─'*60}")

df_examples = pd.DataFrame(all_examples)
scale_params_lookup = {s["scale_name"]: s for s in TARGET_SCALES}

validation_results_all: Dict[str, Dict] = {}
for scale_name, group in tqdm(df_examples.groupby("scale_name"), desc="Validating"):
    scale = scale_params_lookup.get(scale_name, {})
    examples = group.to_dict(orient="records")
    t0 = time.time()
    try:
        batch_size = 10
        for i in range(0, len(examples), batch_size):
            batch = examples[i:i+batch_size]
            batch_results = validate_examples_for_scale(scale_name, batch, scale)
            validation_results_all.update(batch_results)
        status = "success"
    except Exception as e:
        logger.error(f"Phase 1 validation failed for {scale_name}: {e}")
        status = f"error: {e}"

    run_log_p1.append({
        "scale_name": scale_name,
        "phase":      "validation",
        "examples":   len(examples),
        "status":     status,
        "duration_s": round(time.time()-t0, 2),
        "timestamp":  datetime.now().isoformat()
    })

# Apply validation results back to dataframe
def apply_validation(row):
    eid = str(row.get("example_id", ""))
    result = validation_results_all.get(eid, {"valid": True, "issues": "not reviewed"})
    row["process_flag"]      = result["valid"]
    row["validation_issues"] = result.get("issues", "")
    return row

df_examples = df_examples.apply(apply_validation, axis=1)

# Diagnostic — show any still not reviewed
not_reviewed = df_examples[df_examples["validation_issues"] == "not reviewed"]
if len(not_reviewed) > 0:
    print(f"⚠️  {len(not_reviewed)} examples still not reviewed:")
    print(not_reviewed[["scale_name", "example_id"]].to_string())
else:
    print("✓ All examples reviewed")

# Summary stats
summary_rows = []
for scale_name, group in df_examples.groupby("scale_name"):
    total = len(group)
    valid = group["process_flag"].sum()
    summary_rows.append({
        "scale_name":      scale_name,
        "total_examples":  total,
        "valid_examples":  int(valid),
        "invalid_examples": total - int(valid),
        "pass_rate_pct":   round(100*valid/total, 1) if total else 0
    })

df_summary = pd.DataFrame(summary_rows)
df_params  = pd.DataFrame(TARGET_SCALES)
df_log     = pd.DataFrame(run_log_p1)

# ── Human review columns ──────────────────────────────────────────────────────
df_examples["human_review_pass"] = True   # set False to exclude an example
df_examples["human_comments"]    = ""     # free-text rationale for any override

# ── Write Excel ───────────────────────────────────────────────────────────────
with pd.ExcelWriter(P1_OUT, engine="openpyxl") as writer:
    df_examples.to_excel(writer, sheet_name="Behavioral_Domains", index=False)
    df_summary .to_excel(writer, sheet_name="Validation_Summary", index=False)
    df_params  .to_excel(writer, sheet_name="Scale_Parameters",   index=False)
    df_log     .to_excel(writer, sheet_name="Run_Log",            index=False)

total_valid = df_examples["process_flag"].sum()
print(f"\n✅ Phase 1 complete.")
print(f"   {len(all_examples)} examples generated, {int(total_valid)} passed validation.")
print(f"📄 Output: {P1_OUT}")


# ─── PHASE 2: Item Generation ───────────────────────────────────────────────

**Input:** `01_behavioral_domains.xlsx` → sheet `Behavioral_Domains` (respects `human_review_pass`)  
**Output:** `02_generated_items.xlsx`  
**Sheets:** `Generated_Items`, `Generation_Summary`, `Run_Log`

---

### What this phase does

Phase 2 generates survey items — first-person "I" statements — from each validated behavioral domain. Items are written as standard **dominance** items following Goldberg item-writing principles: short, specific, present tense, plain vocabulary, and single idea. Items are stratified by **keying direction** rather than by location on a trait continuum.

**Dominance framework.** In a dominance item, higher trait = higher endorsement probability monotonically. This is the standard Likert-format design used in almost all operational personality inventories (IPIP-NEO, IPIP-HEXACO, Big Five Inventory, DASS, RIASEC, HSQ) and is the model on which the Pseudo-Factor Analysis validation literature rests (Guenole et al., 2024; Milano et al., 2025; Suárez-Álvarez et al., 2026).

**Two keying directions**

- **Positively keyed items** — standard monotone dominance items. Higher trait → stronger endorsement. These express the high pole of the construct clearly and behaviorally. No negations.
- **Negatively keyed items** — genuine semantic reversals of the construct. Lower trait → stronger endorsement. These express the opposite or absence of the construct, not simply low-intensity expressions of it. Negations and reversal language (`rarely`, `never`, `seldom`, `avoid`) are permitted and expected.

The distinction matters. A genuine negatively keyed item for Conscientiousness is `"I often look for ways around workplace rules"` — it describes the opposite behavior. `"I sometimes forget to check my work"` is a low-intensity positive item, not a true reversal. The prompt in this phase enforces the stronger criterion.

**Keying ratio.** A roughly 2:1 positive to negative ratio is recommended (e.g., 10 positive and 5 negative items per domain) rather than 50:50. This is a conservative balance between the two main concerns: negatively keyed items are needed to control **acquiescence bias** — the tendency of some respondents to agree with items regardless of content (Clark & Watson, 1995; DeVellis, 2017) — but too many negatively keyed items can introduce a method factor. The default ratio favours positive items while retaining enough negatively keyed items to detect acquiescence. The ratio is configurable via `items_per_keying` in `pipeline_settings.json`.

**Column added:** `item_keying` with values `positive` or `negative`. This column drives keying-aware behaviour in every downstream phase — simplification preservation in Phase 3, reversal for embedding in Phase 5, and within-keying factor analysis in Phase 5.

**Social desirability** is addressed at the prompt level. Positive items are written as specific, concrete behaviors rather than virtuous generalizations. Negative items are written as neutral counter-indicators rather than transparent admissions of failure. This reduces — but does not eliminate — social desirability bias; empirical social desirability ratings on the final item pool are recommended before operational use.

**Domain specificity** is enforced by requiring every item to be tightly anchored to its source behavioral domain, not to the construct in general. The prompt provides a concrete contrast example using the actual domain text to illustrate the difference between domain-specific and generic items. This is the primary mechanism for reducing cross-domain duplication.

**Near-duplicate detection.** Near-duplicate items within a scale are detected using TF-IDF cosine similarity and flagged — not removed. Detection is applied **separately within each keying direction**, because a positive item and its intentional negative counterpart will legitimately appear similar on a lexical measure. Both items in a near-duplicate pair are flagged with `duplicate_flag = True`, `duplicate_of`, and `duplicate_similarity` columns so the reviewer can decide which to keep.

### Settings (from `pipeline_settings.json`)

| Setting | Role |
|---|---|
| `items_per_keying` | `{"positive": N, "negative": N}` — items generated per domain per keying direction. Set to `0` to skip. |
| `duplicate_removal_threshold` | Cosine similarity above which near-duplicate items are flagged (0–1, default 0.85) |
| `retry_attempts` | Retries on API failure |

### Human review columns added to output

No automated pass/fail is applied in Phase 2 — all generated items proceed to Phase 3 for readability screening. However:

| Column | Default | Purpose |
|---|---|---|
| `item_keying` | System-set | `positive` or `negative` — drives downstream keying-aware logic |
| `duplicate_flag` | System-set | `True` if item is a near-duplicate of another within the same scale and keying direction |
| `duplicate_of` | System-set | `item_id` of the item it most closely resembles |
| `duplicate_similarity` | System-set | Cosine similarity score of the duplicate pair (0–1) |
| `human_review_pass` | `True` | Set `False` to exclude an item before readability analysis |
| `human_comments` | *(blank)* | Notes — especially useful for duplicate decisions and obvious construct contamination |

**Phase 3 reads `human_review_pass` and `item_text`.** Any edits you make to item wording in the Excel file are carried forward to Phase 3. Review duplicate-flagged items first — sort by `duplicate_similarity` descending and keep whichever item in each pair is more tightly anchored to the source behavioral domain.


In [ ]:
# ── Reload settings ───────────────────────────────────────────────────────────
with open(SETTINGS_FILE, "r", encoding="utf-8") as f:
    CFG = json.load(f)
print("✓ Settings reloaded")

P2       = CFG["phase_02_item_generation"]
P2_MODEL = P2["model"]
P2_IN    = BASE_OUTPUT_DIR / CFG["phase_01_behavioral_domains"]["output_file"]
P2_OUT   = BASE_OUTPUT_DIR / P2["output_file"]

assert P2_IN.exists(), f"Input not found: {P2_IN}. Run Phase 1 first."
with pd.ExcelFile(P2_IN) as xls:
    df_p2_in = pd.read_excel(xls, sheet_name="Behavioral_Domains")

# Respect human overrides from Phase 1 and carry forward any edits to example text
if "human_review_pass" in df_p2_in.columns:
    mask = (df_p2_in["human_review_pass"] != False)
    if "process_flag" in df_p2_in.columns:
        mask = mask & (df_p2_in["process_flag"] != False)
    df_p2_valid = df_p2_in[mask].copy()
    n_excl = len(df_p2_in) - len(df_p2_valid)
    print(f"✓ Phase 1 human review: {n_excl} examples excluded by human_review_pass")
else:
    df_p2_valid = df_p2_in.copy()
print(f"Using {len(df_p2_valid)} valid examples (of {len(df_p2_in)} total)")

# ==============================================================================
# KEYING DIRECTIONS — Dominance Item Framework
# ==============================================================================
# Items are stratified by keying direction, not by location on a trait continuum.
# This aligns with the standard Likert / dominance item framework used in
# IPIP-NEO-300, IPIP-HEXACO-240, Big Five Inventory, DASS, RIASEC, HSQ, and
# every other operational personality inventory on which the PFA validation
# literature was developed (Guenole et al. 2024; Milano et al. 2025).
#
#   positive — standard monotone dominance items where higher trait =
#              stronger endorsement. These express the HIGH pole of the
#              construct clearly and behaviorally. No negations.
#
#   negative — genuine semantic reversals where lower trait = stronger
#              endorsement. These express the OPPOSITE or ABSENCE of the
#              construct. Negations and reversal language (rarely, never,
#              seldom, avoid) are permitted and expected. A negatively keyed
#              item is NOT the same as a low-intensity positive item —
#              "I rarely check my work" (negative) is a true counter-indicator
#              of Conscientiousness, while "I sometimes forget to check my work"
#              would be a low-intensity positive item.
#
# The default 2:1 positive-to-negative ratio controls acquiescence bias
# (Clark & Watson 1995; DeVellis 2017) without over-weighting method-factor
# variance from too many negatively keyed items.
# ==============================================================================

items_per_keying = P2["items_per_keying"]
# Ignore internal comment keys if present
items_per_keying = {k: v for k, v in items_per_keying.items() if not k.startswith("_")}

# Keying directions actually requested (count > 0)
ACTIVE_KEYING = [
    k for k in ("positive", "negative")
    if int(items_per_keying.get(k, 0)) > 0
]
assert ACTIVE_KEYING, (
    "At least one keying direction in items_per_keying must be > 0. "
    "Edit pipeline_settings.json → phase_02_item_generation.items_per_keying."
)

# Duplicate-detection threshold — single source of truth in pass_rules
DUP_THRESHOLD = P2["pass_rules"]["duplicate_removal"]["threshold"]

# Human-readable labels and descriptors used in the prompt
KEYING_META = {
    "positive": {
        "label":      "POSITIVELY KEYED",
        "descriptor": (
            "standard dominance items written so that higher trait = stronger "
            "endorsement. These express the HIGH pole of the construct clearly "
            "and behaviorally — the kind of behaviour someone genuinely high on "
            "this trait would show. Must be positively worded — no negations, "
            "no avoidance language, no 'rarely' or 'seldom'."
        )
    },
    "negative": {
        "label":      "NEGATIVELY KEYED",
        "descriptor": (
            "GENUINE SEMANTIC REVERSALS of the construct, written so that lower "
            "trait = stronger endorsement. These describe the OPPOSITE or "
            "ABSENCE of the construct — not low-intensity versions of the "
            "positive items. Negations ('I don't...'), reversal adverbs "
            "('rarely', 'seldom', 'never'), and counter-indicator language "
            "('avoid', 'look for ways around') are permitted and expected. "
            "A negatively keyed item must be something that a HIGH scorer "
            "would clearly DISAGREE with and a LOW scorer would clearly AGREE "
            "with — it is a true reversal, not a milder form of the positive item."
        )
    }
}


def generate_items_for_example(example: Dict, scale: Dict) -> List[Dict]:
    """
    Generate dominance 'I' statement items for one behavioral domain,
    stratified by keying direction (positive vs. negative).
    """

    # Build keying request and format block from active directions only
    keying_request_lines = []
    format_lines         = []
    for k in ACTIVE_KEYING:
        meta  = KEYING_META[k]
        label = meta["label"]
        n     = int(items_per_keying[k])
        keying_request_lines.append(
            f"- {n} {label} items: {meta['descriptor']}"
        )
        format_lines.append(f"{label}\n* I [item — max 9 words]\n...")

    keying_request = "\n".join(keying_request_lines)
    format_block   = "\n".join(format_lines)

    example_id = (
        str(example.get("example_id", ""))
        .replace("*", "").replace("/", "").replace(":", "")
    )

    prompt = f"""You are writing personality survey items for a standard Likert-format dominance assessment of: {scale['scale_name']}

Construct Definition: {scale['construct_definition']}
High pole: {scale.get('high_anchor', 'N/A')}
Low pole: {scale.get('low_anchor', 'N/A')}
Target Population: {json.dumps(scale.get('target_population', {}))}
Setting: {scale.get('target_environment', 'General')}
Behavioral domain: "{example.get('behavior_description', '')}"
Domain likelihood in target environment: {example.get('occurrence_likelihood', 'moderate')}

ASSESSMENT MODEL — IMPORTANT:
These items will be used in a standard Likert (dominance) assessment. In this
model, items are monotone: higher trait = higher endorsement probability for
positively keyed items, and the reverse for negatively keyed items. Mixing
positively and negatively keyed items is standard practice to control
acquiescence bias (Clark & Watson 1995; DeVellis 2017).

Create exactly:
{keying_request}

CRITICAL RULE — DOMAIN SPECIFICITY:
Every item MUST be rooted in the specific behavioral domain above.
If you replaced this domain with a different one, the item should no longer fit.
Generic items that could apply to ANY domain of {scale['scale_name']} are not acceptable.

Example — if the domain is "Following workplace rules and procedures":
GOOD (POSITIVELY KEYED): "I consistently follow the rules at work"     ← high scorers strongly agree
GOOD (NEGATIVELY KEYED): "I often look for ways around workplace rules" ← low scorers strongly agree
BAD (low-intensity positive, NOT a true reversal): "I sometimes forget workplace rules"
BAD (too generic): "I am a conscientious person"

ITEM WRITING RULES — follow all strictly:
1. Write as "I" statements in the present tense
2. MAXIMUM 9 words per item — count every word including "I"
3. One idea only — never use "and", "but", or "or" to join two thoughts
4. POSITIVELY KEYED items: no negations, no avoidance language, no "rarely/seldom/never"
5. NEGATIVELY KEYED items: negations and reversal language are permitted and expected.
   The item must be a GENUINE REVERSAL — something a HIGH scorer would clearly
   disagree with — not a low-intensity version of a positive item.
6. Plain everyday words only — if a 16-year-old might not know it, replace it
7. No abstract nouns — avoid words like "efficiency", "integrity", "competence"
8. Describe observable behavior or a clear personal tendency, not beliefs or values
9. Tightly anchored to the specific behavioral domain above — not the construct generally
10. Each item on its own line starting with "* "
11. No brackets [], quotes, or formatting in the statements
12. US spelling only
13. Each item must measure {scale['scale_name']} ONLY — if a neutral observer
    could plausibly attribute it to a different construct, rewrite it
14. No two items may express the same idea with different words — each must
    describe a meaningfully distinct behavior within this domain
15. POSITIVELY KEYED items must NOT sound like the obviously right answer.
    Rephrase them as specific, concrete behaviors rather than virtuous
    generalizations. Compare:
    BAD (too desirable):  "I always make sure my work meets high standards"
    GOOD (neutral):       "I redo work until the details are exactly right"
16. NEGATIVELY KEYED items must NOT sound like obvious admissions of failure
    or laziness. Rephrase them as neutral counter-indicator statements.
    Compare:
    BAD (too undesirable): "I rarely bother checking my work"
    GOOD (neutral):        "I move on once a task feels done"
17. Negatively keyed items are TRUE REVERSALS, not low-intensity positives.
    Before finalising a negatively keyed item, ask: "Would a HIGH scorer
    clearly DISAGREE with this?" If the answer is just 'probably', rewrite
    it to make the reversal clearer.

GOOD item examples (short, plain, domain-specific):
Positively keyed:
* I keep my desk tidy
* I finish tasks ahead of time
* I speak up in meetings
Negatively keyed:
* I rarely plan my day in advance
* I avoid social gatherings at work
* I let details slide when finishing tasks

BAD item examples (never write these):
* I consistently demonstrate a strong commitment to maintaining high standards  [TOO LONG AND ABSTRACT]
* I connect ideas from different fields together  [TOO GENERIC]
* I help my team stay organized  [CONTAMINATED — measures both Altruism and Conscientiousness]

Format EXACTLY as shown below. Include ONLY the keying sections listed:
{format_block}"""

    raw = llm_call(
        prompt, P2_MODEL,
        max_tokens  = P2["max_tokens"],
        temperature = P2["temperature"],
        retries     = P2["retry_attempts"]
    )

    items        = []
    current_key  = None
    item_counter = 1

    # Map label strings back to keying keys for parsing. The tolerant parser
    # strips common LLM decorators (##, **, --, :) before matching, and accepts
    # prefix matches so that labels like "POSITIVELY KEYED ITEMS" or
    # "POSITIVELY KEYED:" still parse correctly.
    label_to_key = {
        KEYING_META[k]["label"]: k for k in ACTIVE_KEYING
    }

    for line in raw.split("\n"):
        line     = line.strip()
        lu_clean = (
            line.upper()
            .strip("#").strip("*").strip("-").strip(":").strip()
        )

        # Check for section header — exact or prefix match
        matched_key = None
        for label, k in label_to_key.items():
            if lu_clean == label or lu_clean.startswith(label):
                matched_key = k
                break
        if matched_key is not None:
            current_key = matched_key
            continue

        if current_key and line.startswith("* "):
            already = sum(
                1 for i in items if i["item_keying"] == current_key
            )
            if already >= int(items_per_keying[current_key]):
                continue
            text = clean_text(line[2:].strip())
            if text.startswith("I ") and len(text) > 8:
                items.append({
                    "item_id":               f"{example_id}_I{item_counter:02d}",
                    "scale_name":            example.get("scale_name", ""),
                    "item_text":             text,
                    "item_keying":           current_key,
                    "source_example":        example.get("example_id", ""),
                    "example_behavior":      example.get("behavior_description", ""),
                    "occurrence_likelihood": example.get("occurrence_likelihood", ""),
                    "process_flag":          True
                })
                item_counter += 1

    return items

def flag_near_duplicates(items: List[Dict], threshold: float = 0.80) -> List[Dict]:
    """
    Flag near-duplicate items for human review using semantic similarity.

    Uses Sentence-BERT embeddings rather than TF-IDF because lexical overlap
    misses paraphrase duplicates — e.g., "I move on once a task feels done"
    and "I move forward once a task feels complete" are semantically identical
    but share few distinctive tokens (Lee, personal communication; aligned with
    Milano et al. 2025 and Guenole et al. 2025 use of SBERT for psychometric
    similarity).

    Detection is applied SEPARATELY within each keying direction, because a
    positive item and its intentional negative counterpart will legitimately
    appear similar on semantic measures and should NOT be flagged against
    each other.
    """
    if len(items) < 2:
        return items

    model_name = P2.get("duplicate_detection_model", "all-MiniLM-L6-v2")
    model      = get_st_model(model_name)

    from collections import defaultdict
    by_keying = defaultdict(list)
    for item in items:
        by_keying[item.get("item_keying", "positive")].append(item)

    for keying_group in by_keying.values():
        if len(keying_group) < 2:
            continue
        texts = [i["item_text"] for i in keying_group]
        try:
            embs = model.encode(
                texts,
                show_progress_bar      = False,
                normalize_embeddings   = True,   # unit-norm so dot product = cosine
            )
            sim = embs @ embs.T                   # N × N cosine similarity matrix
            for i in range(len(keying_group)):
                for j in range(i + 1, len(keying_group)):
                    if sim[i, j] > threshold:
                        keying_group[i]["duplicate_flag"]       = True
                        keying_group[i]["duplicate_of"]         = keying_group[j]["item_id"]
                        keying_group[j]["duplicate_flag"]       = True
                        keying_group[j]["duplicate_of"]         = keying_group[i]["item_id"]
                        keying_group[i]["duplicate_similarity"] = round(float(sim[i, j]), 3)
                        keying_group[j]["duplicate_similarity"] = round(float(sim[i, j]), 3)
        except Exception as e:
            logger.warning(f"Semantic duplicate flagging failed: {e}")

    return items

# ==============================================================================
# MAIN RUN
# ==============================================================================
print(f"\n{'='*60}")
print(f"PHASE 2: Item Generation (Dominance / standard Likert)")
print(f"Model               : {P2_MODEL}")
print(f"Valid examples      : {len(df_p2_valid)}")
print(f"Active keying dirs  : {ACTIVE_KEYING}")
print(f"Items per keying    : { {k: int(items_per_keying[k]) for k in ACTIVE_KEYING} }")
print(f"Duplicate threshold : {DUP_THRESHOLD} (SBERT cosine, flagged within each keying direction)")
print(f"{'='*60}\n")

all_items:    List[Dict] = []
run_log_p2:   List[Dict] = []
scale_totals: Dict[str, int] = {}

for _, row in tqdm(df_p2_valid.iterrows(), total=len(df_p2_valid), desc="Examples"):
    example    = row.to_dict()
    scale_name = example.get("scale_name", "")
    scale      = scale_params_lookup.get(scale_name, {})
    t0         = time.time()

    try:
        items  = generate_items_for_example(example, scale)
        all_items.extend(items)
        status = "success"
        n      = len(items)
    except Exception as e:
        logger.error(f"Phase 2 failed for {example.get('example_id')}: {e}")
        status = f"error: {e}"
        n      = 0

    scale_totals[scale_name] = scale_totals.get(scale_name, 0) + n
    print(
        f"  {'✓' if n > 0 else '✗'} {example.get('example_id')} "
        f"→ {n} items  "
        f"[{scale_name} running total: {scale_totals[scale_name]}]"
    )

    run_log_p2.append({
        "example_id":      example.get("example_id"),
        "scale_name":      scale_name,
        "items_generated": n,
        "status":          status,
        "duration_s":      round(time.time() - t0, 2),
        "timestamp":       datetime.now().isoformat()
    })
    time.sleep(P2["inter_example_delay_seconds"])

# ── Flag near-duplicates within each scale and each keying direction ──────────
print(f"\n{'='*60}")
print("Duplicate Flagging (no items removed — flagged for human review)")
print("Detection applied SEPARATELY within each keying direction")
print(f"{'='*60}")

flagged_items: List[Dict] = []
for scale_name in df_p2_valid["scale_name"].unique():
    scale_items = [i for i in all_items if i["scale_name"] == scale_name]
    flagged     = flag_near_duplicates(scale_items, DUP_THRESHOLD)
    flagged_items.extend(flagged)
    n_flagged   = sum(1 for i in flagged if i.get("duplicate_flag"))
    print(f"  {scale_name}: {len(flagged)} items, {n_flagged} flagged as near-duplicates")

# ── Build dataframe ───────────────────────────────────────────────────────────
df_items = pd.DataFrame(flagged_items)
if not df_items.empty:
    df_items["duplicate_flag"]       = df_items.get(
        "duplicate_flag",       pd.Series(False, index=df_items.index)
    ).fillna(False)
    df_items["duplicate_of"]         = df_items.get(
        "duplicate_of",         pd.Series("", index=df_items.index)
    ).fillna("")
    df_items["duplicate_similarity"] = df_items.get(
        "duplicate_similarity", pd.Series("", index=df_items.index)
    ).fillna("")

    gen_summary = df_items.groupby(
        ["scale_name", "item_keying"]
    ).size().reset_index(name="item_count")
    print(f"\nFinal item counts by scale and keying direction:")
    print(gen_summary.to_string(index=False))
else:
    gen_summary = pd.DataFrame()

# ── Human review columns ──────────────────────────────────────────────────────
df_items["human_review_pass"] = True
df_items["human_comments"]    = ""

with pd.ExcelWriter(P2_OUT, engine="openpyxl") as writer:
    df_items                 .to_excel(writer, sheet_name="Generated_Items",    index=False)
    gen_summary              .to_excel(writer, sheet_name="Generation_Summary", index=False)
    pd.DataFrame(run_log_p2) .to_excel(writer, sheet_name="Run_Log",            index=False)

n_dup = sum(1 for i in flagged_items if i.get("duplicate_flag"))
print(f"\n✅ Phase 2 complete.")
print(f"   {len(flagged_items)} items across {df_p2_valid['scale_name'].nunique()} scales")
print(f"   {n_dup} items flagged as near-duplicates for human review")
print(f"   Keying directions: {ACTIVE_KEYING}")
print(f"📄 Output: {P2_OUT}")

# ─── PHASE 3: Readability & Bias Analysis ───────────────────────────────────

**Input:** `02_generated_items.xlsx` → sheet `Generated_Items` (respects `human_review_pass`)  
**Output:** `03_readability_bias.xlsx`  
**Sheets:** `Items_With_Readability`, `Readability_Summary`, `Bias_Summary`, `Bias_Detail`, `Simplified_Items_Log`

---

### What this phase does

Phase 3 subjects every item to two quality filters: readability screening and LLM-based bias review. Items are carried forward from Phase 2 including the `item_keying` column (`positive` / `negative`), which determines the simplification rules applied in this phase.

**Readability screening** computes six metrics per item: Flesch-Kincaid Grade, Flesch Reading Ease, Gunning Fog, Coleman-Liau Index, SMOG Index, and Automated Readability Index. Items whose Flesch-Kincaid grade exceeds `reading_level_hard_cap` are automatically simplified by LLM up to `max_simplification_attempts` times. Each simplification attempt rewrites the current item text using simpler vocabulary while preserving the original meaning — critically, **negatively keyed items must retain their negations and reversal language during simplification**. The original wording is preserved in `original_item_text` for audit purposes.

**LLM bias review** screens each item for **construct-irrelevant bias** — bias arising from the item's surface content, not from the trait being measured. Multiple LLM reviewers independently evaluate each item against six dimensions: gender, ethnicity and race, age, religion and culture, socioeconomic, and sexual orientation and disability.

The review is structured around a critical distinction. Every personality item is, by design, harder to endorse for people at one end of the trait continuum than the other — a Conscientiousness item will be harder for disorganised respondents, an Extraversion item harder for introverts, and so on. That is not bias. That is the item discriminating on the construct exactly as intended. Bias exists only when the item's surface content is inaccessible, misleading, or insulting to a specific identifiable group for reasons unrelated to their standing on the construct — for example, an item that assumes the respondent drives, owns a home office, or practices a specific cultural ritual.

The rubric is asymmetric. A rating of 5 (no construct-irrelevant bias identified) is the default. To rate any dimension below 5, a reviewer must name both a specific affected group and a specific construct-irrelevant mechanism — vague concerns like "might be harder for some people" are explicitly rejected. Ratings of 4, 3, 2, and 1 correspond to increasing severity of a concretely-named concern, not to the reviewer's general level of worry. This design is a defence against the well-documented LLM-reviewer tendency to over-flag generic workplace or behavioural content as potentially disadvantaging some hypothetical group.

An item is flagged if its mean rating across reviewers falls below the `flag_threshold` (default 3.5) on any dimension. Flagged items are not automatically excluded — they are passed forward with concern notes for human decision. The reviewer concern notes for flagged items will name the specific group and the specific surface-content mechanism, making the flag auditable rather than impressionistic.

This approach is grounded in Zieky (2013) and aligns with the bias reviewer agent framework of Lee, Son & Jia (2025), with the asymmetric-rubric modification to address LLM-reviewer over-flagging patterns observed in practice. It produces defensible, auditable bias screening evidence suitable for inclusion in a technical report.

### Settings (from `pipeline_settings.json`)

| Setting | Role |
|---|---|
| `reading_level_target` | Target FK grade — soft threshold, items above this are flagged |
| `reading_level_hard_cap` | Hard cap — items above this are simplified automatically |
| `max_simplification_attempts` | Maximum LLM simplification retries per item |
| `bias_review.models` | LLM models used as independent bias reviewers |
| `bias_review.dimensions` | Bias dimensions evaluated per item |
| `bias_review.flag_threshold` | Mean rating below this value on any dimension triggers a flag |
| `bias_review.max_workers` | Parallel threads for bias review calls |

### Output sheets

`Items_With_Readability` is the primary output — one row per item with all readability metrics, simplification history, per-dimension bias mean ratings, and flag columns. `Bias_Detail` provides a full audit trail with per-dimension mean ratings and reviewer concern notes for every item. `Bias_Summary` aggregates flag counts and mean ratings per dimension across the full item pool. `Simplified_Items_Log` records every simplification attempt with before and after text and FK grade.

### Human review columns added to output

| Column | Default | Purpose |
|---|---|---|
| `system_readability_pass` | System-set | `False` if item remains above hard cap after all simplification attempts |
| `bias_any_flagged` | System-set | `True` if mean reviewer rating fell below threshold on any dimension |
| `bias_flagged_dims` | System-set | Comma-separated list of flagged dimensions |
| `bias_concerns` | System-set | Aggregated reviewer concern notes across all flagged dimensions |
| `human_review_pass` | `True` | Set `False` to exclude; set `True` to accept a flagged item with justification |
| `human_comments` | *(blank)* | Required for any accepted bias-flagged item — record reasoning here |

**Phase 4 reads `human_review_pass` and `item_text`.** Any edits you make to item wording in the Excel file are carried forward to Phase 4. Before running Phase 4, open `03_readability_bias.xlsx` and review `Bias_Detail` for all flagged items, paying particular attention to items where multiple dimensions are flagged or where reviewer concern notes identify a specific construct-irrelevant cue. Items where `above_reading_target = True` but `system_readability_pass = True` passed the hard cap but remain above the soft target — review these for scales targeting lower-literacy populations. For negatively keyed items (`item_keying = negative`), verify that the simplification has retained the reversal direction.

In [ ]:
# ── Reload settings ───────────────────────────────────────────────────────────
from concurrent.futures import ThreadPoolExecutor, as_completed
import re as _re
with open(SETTINGS_FILE, "r", encoding="utf-8") as f:
    CFG = json.load(f)
print("✓ Settings reloaded")

P3     = CFG["phase_03_readability_bias"]
P3_IN  = BASE_OUTPUT_DIR / CFG["phase_02_item_generation"]["output_file"]
P3_OUT = BASE_OUTPUT_DIR / P3["output_file"]

assert P3_IN.exists(), f"Input not found: {P3_IN}. Run Phase 2 first."
with pd.ExcelFile(P3_IN) as xls:
    df_p3_raw = pd.read_excel(xls, sheet_name="Generated_Items")

# Respect human overrides from Phase 2 — also carries forward any edits to item_text
if "human_review_pass" in df_p3_raw.columns:
    df_p3 = df_p3_raw[df_p3_raw["human_review_pass"] != False].copy()
    n_excl = len(df_p3_raw) - len(df_p3)
    print(f"✓ Phase 2 human review: {n_excl} items excluded by human_review_pass")
else:
    df_p3 = df_p3_raw.copy()

print(f"Items to process: {len(df_p3)}")

BIAS_CFG        = P3["bias_review"]
BIAS_MODELS     = BIAS_CFG["models"]
BIAS_DIMS       = BIAS_CFG["dimensions"]
BIAS_THRESHOLD  = BIAS_CFG["flag_threshold"]
BIAS_MAX_TOKENS = BIAS_CFG["max_tokens"]
BIAS_TEMP       = BIAS_CFG["temperature"]
BIAS_RETRIES    = BIAS_CFG["retry_attempts"]
BIAS_WORKERS    = BIAS_CFG.get("max_workers", 3)


# ==============================================================================
# READABILITY
# ==============================================================================

def analyze_readability(text: str) -> Dict[str, float]:
    if not text or not text.strip():
        return {m: 0.0 for m in P3["readability_metrics"]}
    return {
        "flesch_kincaid_grade":        round(textstat.flesch_kincaid_grade(text), 3),
        "flesch_reading_ease":         round(textstat.flesch_reading_ease(text), 3),
        "gunning_fog":                 round(textstat.gunning_fog(text), 3),
        "coleman_liau_index":          round(textstat.coleman_liau_index(text), 3),
        "smog_index":                  round(textstat.smog_index(text), 3),
        "automated_readability_index": round(textstat.automated_readability_index(text), 3)
    }


def simplify_item(text: str, scale_name: str, target_grade: int, item_keying: str = "positive") -> str:
    """
    Simplify item wording while preserving meaning and keying direction.

    Critically, negatively keyed items MUST retain their negations and reversal
    language during simplification. Stripping a "rarely" or "don't" from a
    negatively keyed item would flip its semantic direction and invalidate it.
    """
    if item_keying == "negative":
        keying_rule = (
            "5. THIS IS A NEGATIVELY KEYED ITEM. It MUST retain its negation "
            "and/or reversal language (e.g., 'I don't...', 'I rarely...', "
            "'I seldom...', 'I avoid...', 'I look for ways around...'). "
            "Stripping the negation or reversal language would flip the item's "
            "semantic direction and is NOT permitted. Simplify vocabulary only."
        )
    else:
        keying_rule = (
            "5. This is a POSITIVELY KEYED item. It must remain positively "
            "worded — no negations, no 'rarely/seldom/never', no avoidance "
            "language. If simpler wording would introduce a negation, keep "
            "the original wording."
        )

    prompt = f"""Simplify the wording of this personality survey item.

Construct being measured: {scale_name}
Keying direction: {item_keying}
Original item: {text}

Your task is ONLY to simplify the words — do NOT change the meaning, content,
intent, or keying direction of the original item. Every core idea in the
original must be preserved.

GOOD simplifications (same meaning, simpler words):
* "I maintain a well-organized and structured workspace" → "I keep my workspace tidy"
* "I consistently follow through on my commitments" → "I do what I say I will do"
* "I demonstrate persistence when faced with obstacles" → "I keep going when things get hard"
* "I seldom review my work for errors" → "I rarely check my work for mistakes"  (keying preserved)

BAD simplifications (meaning or keying changed — never do this):
* "I maintain a well-organized and structured workspace" → "I plan my day carefully" [WRONG — drifted]
* "I consistently follow through on my commitments" → "I am a reliable person" [WRONG — too vague]
* "I rarely check my work for errors" → "I check my work for errors" [WRONG — keying flipped]

Rules:
1. Keep the "I" statement format in present tense
2. MAXIMUM 9 words — count every word including "I"
3. Replace long or complex words with shorter everyday equivalents
4. Preserve the original keying direction EXACTLY
{keying_rule}
6. No abstract nouns, brackets, quotes, or formatting
7. US spelling only
8. Must still clearly measure {scale_name} and nothing else
9. Reading level at or below Grade {target_grade}
10. Do NOT introduce new ideas or behaviours not in the original

Respond with ONLY the simplified item — no explanation, no punctuation at the end.

Original item to simplify: {text}"""

    resp = llm_call(
        prompt,
        P3["simplification_model"],
        max_tokens  = P3["simplification_max_tokens"],
        temperature = P3["simplification_temperature"]
    )
    for line in resp.split("\n"):
        line = clean_text(line)
        if line.startswith("I ") and 8 < len(line) < 200:
            return line
    return clean_text(resp.split("\n")[0])[:200]


# ==============================================================================
# LLM BIAS REVIEW
# ==============================================================================
# Each item is independently reviewed by multiple LLM models on six
# construct-irrelevant bias dimensions. The rubric is asymmetric: 5 is the
# default, and ratings below 5 require the reviewer to name BOTH a specific
# affected group AND a construct-irrelevant mechanism. "Construct-irrelevant"
# means bias coming from the item's surface content, not from the trait being
# measured — a Conscientiousness item being harder for disorganised people is
# the scale working, not bias.
#
#   5 = no construct-irrelevant bias identified (default)
#   4 = minor — specific group and mechanism named, usable as written
#   3 = moderate — specific group and mechanism named, minor revision resolves
#   2 = serious — specific group and mechanism named, substantial revision
#   1 = severe — specific group and mechanism named, item cannot be used
#
# An item is flagged if its mean rating across reviewers falls below
# flag_threshold (default 3.5) on ANY dimension.
#
# Methodology: Zieky (2013); Lee, Son & Jia (2025, J. Business & Psychology).
# The asymmetric rubric is a defence against the LLM-reviewer over-flagging
# pattern in which generic workplace or behavioural content is flagged because
# "someone, somewhere might find it harder" — a pattern documented across
# multiple LLM-as-judge fairness studies.
# ==============================================================================

BIAS_PROMPT_TEMPLATE = """You are a fairness reviewer for a personality assessment.
Your task is to evaluate a single survey item for CONSTRUCT-IRRELEVANT bias
— bias that comes from the item's surface content, not from the trait it measures.

Item: "{item_text}"
Construct being measured: {scale_name}
Target population: {target_population}

══════════════════════════════════════════════════════════════════════════════
WHAT BIAS IS, AND WHAT IT IS NOT
══════════════════════════════════════════════════════════════════════════════

Every personality item is, by design, harder to endorse for people at one end
of the trait continuum than the other. That is not bias — that is the item
doing its job. A Conscientiousness item that mentions deadlines, schedules,
organisation, or written records is NOT biased against people who struggle
with executive function; those people are simply lower on Conscientiousness,
and the item is discriminating on the target trait exactly as intended. The
same logic applies to every construct: Extraversion items will be harder for
introverts, Openness items harder for concrete thinkers, and so on. None of
that is bias.

Bias exists when the item's SURFACE CONTENT is inaccessible, misleading, or
insulting to a specific identifiable group for reasons UNRELATED to their
standing on the construct. The canonical example: an item about driving a
car would disadvantage people who do not drive, regardless of their trait
level — that is construct-irrelevant bias. An item assuming the respondent
has a fixed workplace, a spouse, a particular religious practice, or familiarity
with a culturally specific idiom would also be construct-irrelevant bias.

══════════════════════════════════════════════════════════════════════════════
HOW TO RATE
══════════════════════════════════════════════════════════════════════════════

For each dimension below, first decide: is there a SPECIFIC, IDENTIFIABLE
group for whom the ITEM CONTENT (not the construct) is inaccessible,
misleading, or insulting, for reasons UNRELATED to their standing on the
construct?

If you cannot name both a specific group AND a specific construct-irrelevant
mechanism, the correct rating is 5. Do NOT rate below 5 to hedge, to cover
hypothetical edge cases, or because the item might be slightly harder for
some person somewhere. "Someone might find this harder" is never sufficient
grounds to rate below 5.

Only if you have identified a real construct-irrelevant mechanism AND can
name the specific affected group, use:

  1 = severe — the item cannot be used as written; specific group is
      meaningfully excluded or demeaned by surface content
  2 = serious — substantial revision needed; content clearly disadvantages
      an identifiable group for construct-irrelevant reasons
  3 = moderate — minor revision would resolve; the content has a specific
      construct-irrelevant cue but the disadvantage is limited
  4 = minor — the item is usable but the concern is worth noting for the
      record

Do NOT use ratings 1–4 unless you can complete BOTH of these sentences in
your concern note:
  "This item disadvantages [specific group] because..."
  "This disadvantage is unrelated to the construct because..."

If you cannot complete both sentences with specific content, rate 5.

══════════════════════════════════════════════════════════════════════════════
DIMENSIONS
══════════════════════════════════════════════════════════════════════════════

For each dimension, ask: "Is there a specific, identifiable group for whom
the item content is inaccessible, misleading, or insulting — for reasons
unrelated to their standing on the construct?" If not, rate 5.

1. gender — e.g., item assumes a gendered role, uses a gendered example,
   or describes a behaviour tied to one gender's typical experience.
2. ethnicity_race — e.g., item uses a culturally specific reference,
   idiom, or context that does not translate across racial/ethnic groups.
3. age — e.g., item assumes familiarity with a technology, activity, or
   life stage specific to one age cohort.
4. religion_culture — e.g., item assumes a specific religious practice,
   dietary norm, holiday, or cultural ritual.
5. socioeconomic — e.g., item assumes access to specific resources
   (a car, a home office, paid travel, disposable income) that some
   respondents genuinely cannot access. NOTE: generic workplace references
   (deadlines, schedules, colleagues, tasks, meetings, projects, emails,
   documents) are NOT socioeconomic bias — they are standard features of
   the target environment for a workplace assessment.
6. sexual_orientation_disability — e.g., item assumes a specific
   relationship structure or physical capability unrelated to the
   construct. NOTE: items describing cognitive, behavioural, or
   organisational behaviours are NOT disability bias merely because some
   disabled people may find them harder — that is construct-relevant
   difficulty, not construct-irrelevant bias.

══════════════════════════════════════════════════════════════════════════════
OUTPUT
══════════════════════════════════════════════════════════════════════════════

For any dimension rated below 5, the concern note MUST name the specific
group AND the specific construct-irrelevant mechanism. Notes like "may
disadvantage some people" or "could be harder for certain groups" are NOT
acceptable — either name the group and mechanism concretely, or rate 5.

Respond in JSON only — no preamble, no markdown:
{{
  "gender": {{"rating": <1-5>, "concern": "<specific group + mechanism, or null>"}},
  "ethnicity_race": {{"rating": <1-5>, "concern": "<specific group + mechanism, or null>"}},
  "age": {{"rating": <1-5>, "concern": "<specific group + mechanism, or null>"}},
  "religion_culture": {{"rating": <1-5>, "concern": "<specific group + mechanism, or null>"}},
  "socioeconomic": {{"rating": <1-5>, "concern": "<specific group + mechanism, or null>"}},
  "sexual_orientation_disability": {{"rating": <1-5>, "concern": "<specific group + mechanism, or null>"}}
}}"""


def review_item_bias_single(
    item_text: str,
    scale_name: str,
    target_population: str,
    model: str
) -> Dict:
    """Run bias review for one item using one model. Returns dimension ratings."""
    prompt = BIAS_PROMPT_TEMPLATE.format(
        item_text         = item_text,
        scale_name        = scale_name,
        target_population = target_population
    )
    try:
        raw     = llm_call(prompt, model,
                           max_tokens=BIAS_MAX_TOKENS,
                           temperature=BIAS_TEMP,
                           retries=BIAS_RETRIES)
        cleaned = _re.sub(r'```json|```', '', raw).strip()
        match   = _re.search(r'\{.*\}', cleaned, _re.DOTALL)
        if not match:
            raise ValueError("No JSON found in response")
        return json.loads(match.group())
    except Exception as e:
        logger.warning(f"Bias review failed ({model}): {e}")
        return {}


def review_item_bias(
    item_id: str,
    item_text: str,
    scale_name: str,
    target_population: str
) -> Dict:
    """
    Run bias review across all configured models in parallel.
    Returns aggregated ratings and flags per dimension.
    """
    model_results = {}

    with ThreadPoolExecutor(max_workers=BIAS_WORKERS) as executor:
        futures = {
            executor.submit(
                review_item_bias_single,
                item_text, scale_name, target_population, model
            ): model
            for model in BIAS_MODELS
        }
        for future in as_completed(futures):
            model = futures[future]
            try:
                model_results[model] = future.result()
            except Exception as e:
                logger.warning(f"Bias future failed for {item_id} ({model}): {e}")
                model_results[model] = {}

    # Aggregate ratings and concerns across models per dimension
    agg = {}
    all_flags   = []
    all_concerns = []

    for dim in BIAS_DIMS:
        ratings  = []
        concerns = []
        for model, result in model_results.items():
            dim_data = result.get(dim, {})
            if isinstance(dim_data, dict):
                r = dim_data.get("rating")
                c = dim_data.get("concern")
                if r is not None:
                    try:
                        ratings.append(float(r))
                    except (ValueError, TypeError):
                        pass
                if c and str(c).lower() not in ("null", "none", ""):
                    concerns.append(f"{dim}: {c}")

        mean_rating = round(sum(ratings) / len(ratings), 2) if ratings else None
        flagged     = mean_rating is not None and mean_rating < BIAS_THRESHOLD

        agg[dim] = {
            "mean_rating": mean_rating,
            "n_reviewers": len(ratings),
            "flagged":     flagged,
            "concerns":    "; ".join(concerns) if concerns else ""
        }

        if flagged:
            all_flags.append(dim)
        if concerns:
            all_concerns.extend(concerns)

    return {
        "bias_dimensions":    agg,
        "bias_flagged_dims":  all_flags,
        "bias_flag_count":    len(all_flags),
        "bias_any_flagged":   len(all_flags) > 0,
        "bias_concerns":      " | ".join(all_concerns),
        "bias_reviewer_raw":  json.dumps(model_results)
    }


# ==============================================================================
# MAIN RUN
# ==============================================================================
print(f"\n{'='*60}")
print(f"PHASE 3: Readability & Bias Analysis")
print(f"Items to process   : {len(df_p3)}")
print(f"Reading target     : Grade {P3['reading_level_target']}")
print(f"Hard cap           : Grade {P3['reading_level_hard_cap']}")
print(f"Max simplifications: {P3.get('max_simplification_attempts', 3)}")
print(f"Bias reviewers     : {len(BIAS_MODELS)} models × {len(BIAS_DIMS)} dimensions")
print(f"Bias flag threshold: mean < {BIAS_THRESHOLD}")
print(f"{'='*60}\n")

readability_cols = P3["readability_metrics"]
simplified_log:  List[Dict] = []
bias_detail_rows: List[Dict] = []

# Initialise columns
for col in readability_cols:
    df_p3[col] = 0.0

df_p3["bias_flagged_dims"]        = ""
df_p3["bias_flag_count"]          = 0
df_p3["bias_any_flagged"]         = False
df_p3["bias_concerns"]            = ""
df_p3["was_simplified"]           = False
df_p3["simplification_attempts"]  = 0
df_p3["original_item_text"]       = ""
df_p3["above_reading_target"]     = False
df_p3["system_readability_pass"]  = True

# Per-dimension mean rating columns
for dim in BIAS_DIMS:
    df_p3[f"bias_{dim}_mean"] = None

max_attempts = P3.get("max_simplification_attempts", 3)

for idx, row in tqdm(df_p3.iterrows(), total=len(df_p3), desc="Items"):
    item_id     = row.get("item_id", f"item_{idx}")
    text        = str(row.get("item_text", ""))
    scale_name  = str(row.get("scale_name", ""))
    item_keying = str(row.get("item_keying", "positive"))
    scale       = scale_params_lookup.get(scale_name, {})
    target_grade = int(scale.get("reading_level_target", P3["reading_level_target"]))
    target_pop   = json.dumps(scale.get("target_population", {}))

    # ── Readability ───────────────────────────────────────────────────────────
    scores   = analyze_readability(text)
    fk_grade = scores["flesch_kincaid_grade"]
    for col, val in scores.items():
        df_p3.at[idx, col] = val

    # ── Simplification retry loop ─────────────────────────────────────────────
    attempt = 0
    while fk_grade > P3["reading_level_hard_cap"] and attempt < max_attempts:
        attempt += 1
        try:
            current_text = str(df_p3.at[idx, "item_text"])
            new_text     = simplify_item(current_text, scale_name, target_grade, item_keying)
            new_scores   = analyze_readability(new_text)

            simplified_log.append({
                "item_id":         item_id,
                "scale_name":      scale_name,
                "item_keying":     item_keying,
                "source_example":  row.get("source_example", ""),
                "attempt":         attempt,
                "input_text":      current_text,
                "simplified_text": new_text,
                "input_fk_grade":  fk_grade,
                "new_fk_grade":    new_scores["flesch_kincaid_grade"]
            })

            if not df_p3.at[idx, "original_item_text"]:
                df_p3.at[idx, "original_item_text"] = text

            df_p3.at[idx, "item_text"]              = new_text
            df_p3.at[idx, "was_simplified"]          = True
            df_p3.at[idx, "simplification_attempts"] = attempt
            for col, val in new_scores.items():
                df_p3.at[idx, col] = val

            print(
                f"  ✂ {item_id} attempt {attempt}/{max_attempts}  "
                f"FK: {fk_grade:.1f} → {new_scores['flesch_kincaid_grade']:.1f}  "
                f"| {current_text[:40]}... → {new_text[:40]}"
            )

            fk_grade = new_scores["flesch_kincaid_grade"]
            scores   = new_scores

        except Exception as e:
            logger.warning(f"Simplification attempt {attempt} failed for {item_id}: {e}")
            break

    if fk_grade > P3["reading_level_hard_cap"]:
        logger.warning(
            f"  ⚠ {item_id} still above hard cap (FK: {fk_grade:.1f}) "
            f"after {attempt} attempt(s) — kept as-is for human review"
        )
        df_p3.at[idx, "system_readability_pass"] = False

    df_p3.at[idx, "above_reading_target"] = fk_grade > target_grade

    # ── LLM Bias Review — always on final item_text ───────────────────────────
    final_text  = str(df_p3.at[idx, "item_text"])
    bias_result = review_item_bias(item_id, final_text, scale_name, target_pop)

    df_p3.at[idx, "bias_flagged_dims"] = ", ".join(bias_result["bias_flagged_dims"])
    df_p3.at[idx, "bias_flag_count"]   = bias_result["bias_flag_count"]
    df_p3.at[idx, "bias_any_flagged"]  = bias_result["bias_any_flagged"]
    df_p3.at[idx, "bias_concerns"]     = bias_result["bias_concerns"]

    for dim in BIAS_DIMS:
        dim_data = bias_result["bias_dimensions"].get(dim, {})
        df_p3.at[idx, f"bias_{dim}_mean"] = dim_data.get("mean_rating")

    # Bias detail row for audit sheet
    detail_row = {
        "item_id":           item_id,
        "scale_name":        scale_name,
        "item_keying":       item_keying,
        "item_text":         final_text,
        "bias_any_flagged":  bias_result["bias_any_flagged"],
        "bias_flagged_dims": ", ".join(bias_result["bias_flagged_dims"]),
        "bias_concerns":     bias_result["bias_concerns"],
    }
    for dim in BIAS_DIMS:
        dim_data = bias_result["bias_dimensions"].get(dim, {})
        detail_row[f"{dim}_mean_rating"] = dim_data.get("mean_rating")
        detail_row[f"{dim}_concerns"]    = dim_data.get("concerns", "")
    bias_detail_rows.append(detail_row)

    if bias_result["bias_any_flagged"]:
        print(
            f"  ⚠ {item_id} bias flags: {', '.join(bias_result['bias_flagged_dims'])}  "
            f"| {final_text[:60]}"
        )

# ==============================================================================
# SUMMARY
# ==============================================================================
read_summary = df_p3.groupby("scale_name").agg(
    total_items        = ("item_id",             "count"),
    mean_fk_grade      = ("flesch_kincaid_grade", "mean"),
    items_above_target = ("above_reading_target", "sum"),
    items_simplified   = ("was_simplified",       "sum")
).reset_index()
read_summary["mean_fk_grade"] = read_summary["mean_fk_grade"].round(2)

bias_summary_rows = []
for dim in BIAS_DIMS:
    count = df_p3["bias_flagged_dims"].str.contains(dim, na=False).sum()
    mean_col = f"bias_{dim}_mean"
    mean_rating = round(df_p3[mean_col].mean(), 2) if mean_col in df_p3.columns else None
    bias_summary_rows.append({
        "bias_dimension":   dim,
        "items_flagged":    int(count),
        "pct_of_total":     round(100 * count / len(df_p3), 1) if len(df_p3) else 0,
        "mean_rating_all":  mean_rating,
        "flag_threshold":   BIAS_THRESHOLD,
        "n_reviewers":      len(BIAS_MODELS)
    })

print(f"\n{'='*60}")
print("Readability Summary by Scale")
print(f"{'='*60}")
print(read_summary.to_string(index=False))

print(f"\n{'='*60}")
print("Bias Summary by Dimension")
print(f"{'='*60}")
print(pd.DataFrame(bias_summary_rows).to_string(index=False))

# ── Human review columns ──────────────────────────────────────────────────────
df_p3["human_review_pass"] = True
df_p3["human_comments"]    = ""

# ==============================================================================
# WRITE OUTPUT
# ==============================================================================
with pd.ExcelWriter(P3_OUT, engine="openpyxl") as writer:
    df_p3.to_excel(writer,
                   sheet_name="Items_With_Readability", index=False)
    read_summary.to_excel(writer,
                          sheet_name="Readability_Summary",    index=False)
    pd.DataFrame(bias_summary_rows).to_excel(writer,
                                             sheet_name="Bias_Summary",     index=False)
    pd.DataFrame(bias_detail_rows).to_excel(writer,
                                            sheet_name="Bias_Detail",       index=False)
    if simplified_log:
        pd.DataFrame(simplified_log).to_excel(writer,
                                              sheet_name="Simplified_Items_Log", index=False)

print(f"\n✅ Phase 3 complete.")
print(f"   {int(df_p3['was_simplified'].sum())} items simplified")
print(f"   {int(df_p3['bias_any_flagged'].sum())} items with bias flags")
print(f"   {int(df_p3['above_reading_target'].sum())} items still above reading target")
print(f"   Bias reviewed by {len(BIAS_MODELS)} models across {len(BIAS_DIMS)} dimensions")
print(f"📄 Output: {P3_OUT}")


# ─── PHASE 4: Content Validity – LLM SME Review ─────────────────────────────

**Input:** `03_readability_bias.xlsx` → sheet `Items_With_Readability` (respects `human_review_pass`)  
**Output:** `04_content_validity.xlsx`  
**Sheets:** `CV_All_SME_Ratings`, `CV_Item_Summary`, `CV_Scale_Summary`, `Run_Log`

---

### What this phase does

Five LLM models act as independent subject matter experts (SMEs). Each rates every item's correspondence with **every scale** in your assessment on a 1–7 Likert scale (Hinkin & Tracey, 1999). From these multi-scale ratings, two content validity indices are computed per item following the procedures in Colquitt et al. (2019) and the LM-AIG framework (Lee, Son, & Jia, 2025):

- **c-value (Correspondence)** — the average correspondence rating for the item's *intended* (target) scale, normalized to 0–1. Captures how well the item measures what it is supposed to measure.
- **d-value (Distinctiveness)** — the average of (target rating − orbiting rating) / (a − 1), where *a* is the number of anchors on the rating scale. Captures how clearly the item separates from non-target constructs.

An item **passes** if it meets both thresholds set in `pipeline_settings.json`:

- `min_c_value` ≥ 0.88
- `min_d_value` ≥ 0.35

Scale mapping agreement (the proportion of SMEs whose highest-rated scale matches the target) is retained as a diagnostic metric. The `item_keying` column is carried forward from Phase 2, recording whether each item was written as positively or negatively keyed. The CV prompt itself does not reference keying — distinctiveness is evaluated on semantic correspondence to the construct definition, not on item direction.

Checkpointing is applied every 100 items — if the run is interrupted, it resumes from where it left off.

> **Methodological basis:** The correspondence and distinctiveness indices were introduced by Hinkin and Tracey (1999), operationalized by Colquitt et al. (2019), and implemented in an automated LLM context by Lee, Son, and Jia (2025) in the LM-AIG multi-agent framework. Suárez-Álvarez et al. (2026) provide the broader practical guide within which these techniques operate.

### System pass/fail rules (from `pipeline_settings.json`)

| Setting | Value / Role |
|---|---|
| `min_c_value` | Default 0.88 — minimum correspondence index |
| `min_d_value` | Default 0.35 — minimum distinctiveness index |
| `min_scale_mapping_agreement` | Default 0.6 — diagnostic metric (proportion of SMEs mapping to target) |
| `sme_models` | Dict of LLM models used as SMEs |
| `sme_max_workers` | Parallel threads for SME calls |

### Human review columns added to output

| Column | Default | Purpose |
|---|---|---|
| `cv_pass` | System-set | Automated CV verdict based on c-value and d-value thresholds |
| `human_review_pass` | `True` | Override CV verdict — accept marginal items or reject borderline passers |
| `human_comments` | *(blank)* | Rationale — particularly useful when c-value or d-value is borderline |

**Phase 5 reads `human_review_pass` and `item_text`.** Any edits you make to item wording in the Excel file are carried forward to Phase 5. Review `04_content_validity.xlsx → CV_Item_Summary` before running Phase 5 — focus on items where c-value or d-value is just below threshold, items where `cv_majority_mapped_scale` differs from the target, and negatively keyed items (`item_keying = negative`) where distinctiveness may be harder to achieve because SMEs may rate the negative paraphrase lower on the target scale.


In [ ]:
# ==============================================================================
# PHASE 4 — Content Validity: LLM SME Review
# ==============================================================================
# Reads:  output/03_readability_bias.xlsx
# Writes: output/04_content_validity.xlsx
#   Sheets: CV_All_SME_Ratings | CV_Item_Summary | CV_Scale_Summary | Run_Log
#
# Checkpoint: saves progress every 100 items to output/04_checkpoint.json
#   - If a checkpoint exists, the run resumes from where it left off
#   - Checkpoint is deleted automatically on successful completion
#
# c-value = average definitional correspondence rating / n_anchors
# d-value = average of (target - orbiting) / (n_anchors - 1) across SMEs
# Both follow Hinkin & Tracey (1999) as implemented by Colquitt et al. (2019)
# and used in Lee, Son & Jia (2025).
# ==============================================================================

logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)

# ── Reload settings ───────────────────────────────────────────────────────────
with open(SETTINGS_FILE, "r", encoding="utf-8") as f:
    CFG = json.load(f)
print("✓ Settings reloaded")

from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

P4     = CFG["phase_04_content_validity"]
P4_IN  = BASE_OUTPUT_DIR / CFG["phase_03_readability_bias"]["output_file"]
P4_OUT = BASE_OUTPUT_DIR / P4["output_file"]
P4_CHK = BASE_OUTPUT_DIR / "04_checkpoint.json"

CHECKPOINT_INTERVAL = 100
MIN_C_VALUE = P4["pass_thresholds"]["min_c_value"]
MIN_D_VALUE = P4["pass_thresholds"]["min_d_value"]
MIN_MAPPING = P4["pass_thresholds"].get("min_scale_mapping_agreement", 0.6)

assert P4_IN.exists(), f"Input not found: {P4_IN}. Run Phase 3 first."
with pd.ExcelFile(P4_IN) as xls:
    df_p4_raw = pd.read_excel(xls, sheet_name="Items_With_Readability")

# Respect human overrides from Phase 3 — also carries forward any edits to item_text
if "human_review_pass" in df_p4_raw.columns:
    df_p4 = df_p4_raw[df_p4_raw["human_review_pass"] != False].copy()
    n_excl = len(df_p4_raw) - len(df_p4)
    print(f"✓ Phase 3 human review: {n_excl} items excluded by human_review_pass")
else:
    df_p4 = df_p4_raw.copy()

SME_MODELS: Dict[str, Dict] = P4["sme_models"]
SME_KEYS        = list(SME_MODELS.keys())
SME_MAX_WORKERS = P4.get("sme_max_workers", len(SME_KEYS))

# ── Diagnostic: verify scales list is populated ───────────────────────────────
print(f"✓ ALL_SCALES_LIST_STR length: {len(ALL_SCALES_LIST_STR)} chars")
if not ALL_SCALES_LIST_STR:
    raise ValueError(
        "ALL_SCALES_LIST_STR is empty — re-run Section 0 to reload scales."
    )


# ── Checkpoint helpers ────────────────────────────────────────────────────────
def save_checkpoint(all_sme_rows: List[Dict], run_log: List[Dict], completed_ids: List[str]) -> None:
    payload = {
        "completed_item_ids": completed_ids,
        "all_sme_rows":       all_sme_rows,
        "run_log":            run_log,
        "saved_at":           datetime.now().isoformat()
    }
    tmp = str(P4_CHK) + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(payload, f)
    os.replace(tmp, P4_CHK)
    print(f"  💾 Checkpoint saved — {len(completed_ids)} items complete")

def load_checkpoint() -> tuple:
    if not P4_CHK.exists():
        return [], [], []
    with open(P4_CHK, "r", encoding="utf-8") as f:
        payload = json.load(f)
    completed_ids = payload.get("completed_item_ids", [])
    all_sme_rows  = payload.get("all_sme_rows",       [])
    run_log       = payload.get("run_log",             [])
    print(f"✓ Checkpoint found — resuming from item {len(completed_ids) + 1} "
          f"({len(completed_ids)} already complete)")
    return all_sme_rows, run_log, completed_ids


# ── Thread-safe LLM helpers ───────────────────────────────────────────────────
def make_thread_client() -> OpenAI:
    return OpenAI(
        api_key      = OPENROUTER_API_KEY,
        base_url     = OPENROUTER_BASE_URL,
        default_headers={
            "HTTP-Referer": "https://inubilum.io",
            "X-Title":      os.getenv("OPENROUTER_APP_NAME", "Project Majdal")
        }
    )

def llm_call_threaded(
    prompt:      str,
    model:       str,
    max_tokens:  int   = 500,
    temperature: float = 0.2,
    retries:     int   = 3,
    system:      str   = "You are a psychometric expert assistant."
) -> str:
    thread_client = make_thread_client()
    for attempt in range(retries):
        try:
            resp = thread_client.chat.completions.create(
                model       = model,
                messages    = [
                    {"role": "system", "content": system},
                    {"role": "user",   "content": prompt}
                ],
                max_tokens  = max_tokens,
                temperature = temperature
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            wait = (2 ** attempt) + np.random.uniform(0, 1)
            logger.warning(
                f"Threaded LLM call failed (attempt {attempt+1}/{retries}): {e}. "
                f"Retrying in {wait:.1f}s"
            )
            time.sleep(wait)
    raise RuntimeError(f"Threaded LLM call failed after {retries} attempts")

def build_cv_prompt(item_text: str, scale_name: str, scale_definition: str) -> str:
    scales_block = "\n".join(
        f"- {name}: {defn[:200]}..." if len(defn) > 200 else f"- {name}: {defn}"
        for name, defn in ALL_SCALES_LOOKUP.items()
    )
    template = P4["prompt_template"]
    return template.format(
        scale_name       = scale_name,
        all_scales_block = scales_block,
        item_text        = item_text
    )

def parse_cv_response(raw: str) -> Dict:
    try:
        cleaned = re.sub(r'```json|```', '', raw).strip()
        brace_depth = 0
        start = None
        for i, ch in enumerate(cleaned):
            if ch == '{':
                if start is None:
                    start = i
                brace_depth += 1
            elif ch == '}':
                brace_depth -= 1
                if brace_depth == 0 and start is not None:
                    obj = json.loads(cleaned[start:i+1])
                    ratings = obj.get("ratings", {})
                    ratings = {k.strip(): float(v) for k, v in ratings.items()}
                    return {
                        "scale_ratings": ratings,
                        "rationale":     str(obj.get("rationale", ""))
                    }
    except Exception:
        pass
    return {
        "scale_ratings": {},
        "rationale":     raw[:200]
    }

def review_item_with_sme(
    item_text:  str,
    scale_name: str,
    scale_def:  str,
    sme_key:    str
) -> Dict:
    model_id = SME_MODELS[sme_key]["model_id"]
    prompt   = build_cv_prompt(item_text, scale_name, scale_def)
    try:
        raw    = llm_call_threaded(
            prompt, model_id,
            max_tokens  = P4["max_tokens"],
            temperature = P4["temperature"],
            retries     = P4["retry_attempts"],
            system      = "You are a psychometric subject matter expert. Respond only in valid JSON."
        )
        result = parse_cv_response(raw)
        result["sme"]          = sme_key
        result["raw_response"] = raw[:400]
        result["error"]        = ""
    except Exception as e:
        result = {
            "sme":               sme_key,
            "scale_ratings":     {},
            "rationale":         "",
            "raw_response":      "",
            "error":             str(e)
        }
    return result


# ── Quick single-call diagnostic before full run ──────────────────────────────
print("\nRunning single-call diagnostic...")
_test = review_item_with_sme(
    item_text  = "I enjoy solving complex problems",
    scale_name = "Abstract Thinking",
    scale_def  = "The ability to think conceptually and recognize patterns",
    sme_key    = SME_KEYS[0]
)
if _test["error"]:
    raise RuntimeError(
        f"Diagnostic call failed for {SME_KEYS[0]}: {_test['error']}\n"
        "Fix this before running the full phase."
    )
print(f"✓ Diagnostic passed — {SME_KEYS[0]} returned ratings: {_test.get('scale_ratings', {})}")


# ── Load checkpoint if available ──────────────────────────────────────────────
all_sme_rows, run_log_p4, completed_ids = load_checkpoint()
completed_set = set(completed_ids)

df_remaining = df_p4[~df_p4["item_id"].isin(completed_set)].copy()

print(f"\n{'='*60}")
print(f"PHASE 4: Content Validity – LLM SME Review")
print(f"Total items     : {len(df_p4)}")
print(f"Already done    : {len(completed_set)}")
print(f"Remaining       : {len(df_remaining)}")
print(f"SMEs            : {len(SME_KEYS)}")
print(f"Parallel workers: {SME_MAX_WORKERS}")
print(f"Remaining calls : {len(df_remaining) * len(SME_KEYS)}")
print(f"SME models      : {', '.join(SME_KEYS)}")
print(f"Pass threshold  : c-value >= {MIN_C_VALUE} AND d-value >= {MIN_D_VALUE}")
print(f"Checkpoint every: {CHECKPOINT_INTERVAL} items  →  {P4_CHK.name}")
print(f"{'='*60}\n")

# ── Main processing loop ──────────────────────────────────────────────────────
items_since_checkpoint = 0

for idx, row in tqdm(df_remaining.iterrows(), total=len(df_remaining), desc="Items"):
    item_id    = row.get("item_id",    f"item_{idx}")
    item_text  = str(row.get("item_text",  ""))
    scale_name = str(row.get("scale_name", ""))
    scale      = scale_params_lookup.get(scale_name, {})
    scale_def  = scale.get("construct_definition", ALL_SCALES_LOOKUP.get(scale_name, ""))

    item_ratings:     Dict[str, float]          = {}
    item_mappings:    Dict[str, str]             = {}
    item_all_ratings: Dict[str, Dict[str, float]] = {}
    t0_item = time.time()

    # ── Run all SMEs in parallel ──────────────────────────────────────────────
    with ThreadPoolExecutor(max_workers=SME_MAX_WORKERS) as executor:
        futures = {
            executor.submit(
                review_item_with_sme, item_text, scale_name, scale_def, sme_key
            ): sme_key
            for sme_key in SME_KEYS
        }
        for future in as_completed(futures):
            sme_key = futures[future]
            try:
                result = future.result()
            except Exception as e:
                result = {
                    "sme":               sme_key,
                    "scale_ratings":     {},
                    "rationale":         "",
                    "raw_response":      "",
                    "error":             str(e)
                }
            result.update({
                "item_id":               item_id,
                "scale_name":            scale_name,
                "item_text":             item_text,
                "source_example":        row.get("source_example",        ""),
                "occurrence_likelihood": row.get("occurrence_likelihood",  ""),
                "item_keying":           row.get("item_keying",            ""),
                "flesch_kincaid_grade":  row.get("flesch_kincaid_grade",   0),
                "was_simplified":        row.get("was_simplified",         False),
                "bias_flagged_dims":     row.get("bias_flagged_dims",      ""),
                "bias_any_flagged":      row.get("bias_any_flagged",       False),
                "duration_s":            round(time.time() - t0_item, 2)
            })
            all_sme_rows.append(result)
            scale_ratings = result.get("scale_ratings", {})
            target_rating = scale_ratings.get(scale_name, 0.0)
            item_ratings[sme_key]     = target_rating
            best_scale                = max(scale_ratings, key=scale_ratings.get) if scale_ratings else "unknown"
            item_mappings[sme_key]    = best_scale
            item_all_ratings[sme_key] = scale_ratings

    ratings_str = "  ".join(f"{k}: {v:.1f}" for k, v in item_ratings.items())
    mean_rating = round(float(np.mean(list(item_ratings.values()))), 2) if item_ratings else 0.0
    print(
        f"  {item_id}  |  target_mean: {mean_rating}  |  "
        f"{ratings_str}  |  best: {list(item_mappings.values())}"
    )

    run_log_p4.append({
        "item_id":     item_id,
        "scale_name":  scale_name,
        "sme_count":   len(SME_KEYS),
        "mean_rating": mean_rating,
        "duration_s":  round(time.time() - t0_item, 2),
        "timestamp":   datetime.now().isoformat()
    })

    completed_ids.append(item_id)
    items_since_checkpoint += 1

    if items_since_checkpoint >= CHECKPOINT_INTERVAL:
        save_checkpoint(all_sme_rows, run_log_p4, completed_ids)
        items_since_checkpoint = 0

    time.sleep(P4["inter_item_delay_seconds"])

# Serialize scale_ratings dicts for Excel storage
for row in all_sme_rows:
    sr = row.get("scale_ratings", {})
    row["scale_ratings_str"] = str(sr)

df_sme_raw = pd.DataFrame(all_sme_rows)

# ── Compute agreement metrics per item ────────────────────────────────────────
print(f"\n{'='*60}")
print("Computing agreement metrics...")
print(f"{'='*60}")

item_summary_rows: List[Dict] = []
N_ANCHORS = 7

for item_id, grp in df_sme_raw.groupby("item_id"):
    first_row    = grp.iloc[0]
    target_scale = first_row["scale_name"]

    sme_scale_ratings: List[Dict[str, float]] = []
    for _, sme_row in grp.iterrows():
        sr = sme_row.get("scale_ratings", {})
        if isinstance(sr, str):
            try:
                sr = json.loads(sr.replace("'", '"'))
            except Exception:
                sr = {}
        if sr:
            sme_scale_ratings.append(sr)

    # ── c-value and d-value (Hinkin & Tracey, 1999; Colquitt et al., 2019) ───
    if sme_scale_ratings:
        target_ratings  = [sr.get(target_scale, 0.0) for sr in sme_scale_ratings]
        mean_target_raw = float(np.mean(target_ratings))
        c_value         = round(mean_target_raw / N_ANCHORS, 3)

        orbiting_scales = [s for s in ALL_SCALES_LOOKUP.keys() if s != target_scale]
        if orbiting_scales:
            d_values_per_sme = []
            for sr in sme_scale_ratings:
                t_rating  = sr.get(target_scale, 0.0)
                orb_diffs = [(t_rating - sr.get(os, 0.0)) for os in orbiting_scales]
                mean_diff = float(np.mean(orb_diffs))
                d_values_per_sme.append(mean_diff / (N_ANCHORS - 1))
            d_value = round(float(np.mean(d_values_per_sme)), 3)
        else:
            d_value = 1.0

        mean_target_rating = round(mean_target_raw, 3)
        std_target_rating  = round(float(np.std(target_ratings)), 3)
        min_rating         = min(target_ratings)
        max_rating         = max(target_ratings)
    else:
        c_value = d_value = mean_target_rating = std_target_rating = 0.0
        min_rating = max_rating = 0

    # ── Scale mapping ─────────────────────────────────────────────────────────
    mapped_scales = []
    for sr in sme_scale_ratings:
        if sr:
            mapped_scales.append(max(sr, key=sr.get))
        else:
            mapped_scales.append("unknown")

    valid_scales            = [s for s in mapped_scales if s != "unknown"]
    target_count            = valid_scales.count(target_scale)
    scale_mapping_agreement = round(target_count / len(SME_KEYS), 3) if SME_KEYS else 0.0

    if valid_scales:
        most_common_scale, most_common_count = Counter(valid_scales).most_common(1)[0]
        majority_agreement = round(most_common_count / len(SME_KEYS), 3)
    else:
        most_common_scale  = "unknown"
        majority_agreement = 0.0

    cv_pass = (c_value >= MIN_C_VALUE and d_value >= MIN_D_VALUE)

    item_summary_rows.append({
        "item_id":                    item_id,
        "scale_name":                 target_scale,
        "item_text":                  first_row["item_text"],
        "source_example":             first_row.get("source_example",        ""),
        "occurrence_likelihood":      first_row.get("occurrence_likelihood",  ""),
        "item_keying":                first_row.get("item_keying",            ""),
        "flesch_kincaid_grade":       first_row.get("flesch_kincaid_grade",   0),
        "was_simplified":             first_row.get("was_simplified",         False),
        "bias_flagged_dims":          first_row.get("bias_flagged_dims",      ""),
        "bias_any_flagged":           first_row.get("bias_any_flagged",       False),
        "cv_c_value":                 c_value,
        "cv_d_value":                 d_value,
        "cv_mean_target_rating":      mean_target_rating,
        "cv_std_target_rating":       std_target_rating,
        "cv_min_rating":              min_rating,
        "cv_max_rating":              max_rating,
        "cv_scale_mapping_agreement": scale_mapping_agreement,
        "cv_maps_to_target_count":    target_count,
        "cv_maps_to_target":          target_count == len(SME_KEYS),
        "cv_majority_mapped_scale":   most_common_scale,
        "cv_majority_agreement":      majority_agreement,
        "cv_majority_is_target":      most_common_scale == target_scale,
        "cv_sme_scale_ratings":       str({
                                          sme: sr
                                          for sme, sr in zip(grp["sme"], sme_scale_ratings)
                                      }) if sme_scale_ratings else "{}",
        "cv_pass":                    cv_pass
    })

df_cv_summary = pd.DataFrame(item_summary_rows)

# ── Scale-level CV summary ────────────────────────────────────────────────────
cv_scale_summary = df_cv_summary.groupby("scale_name").agg(
    total_items            = ("item_id",                   "count"),
    cv_pass_count          = ("cv_pass",                   "sum"),
    mean_c_value           = ("cv_c_value",                "mean"),
    mean_d_value           = ("cv_d_value",                "mean"),
    mean_target_rating     = ("cv_mean_target_rating",     "mean"),
    mean_agreement         = ("cv_scale_mapping_agreement","mean"),
    unanimous_target_pct   = ("cv_maps_to_target",         "mean"),
    majority_is_target_pct = ("cv_majority_is_target",     "mean")
).reset_index()

cv_scale_summary["cv_pass_rate_pct"]       = round(
    100 * cv_scale_summary["cv_pass_count"] / cv_scale_summary["total_items"], 1
)
cv_scale_summary["mean_c_value"]           = cv_scale_summary["mean_c_value"].round(3)
cv_scale_summary["mean_d_value"]           = cv_scale_summary["mean_d_value"].round(3)
cv_scale_summary["mean_target_rating"]     = cv_scale_summary["mean_target_rating"].round(2)
cv_scale_summary["mean_agreement"]         = cv_scale_summary["mean_agreement"].round(2)
cv_scale_summary["unanimous_target_pct"]   = (cv_scale_summary["unanimous_target_pct"] * 100).round(1)
cv_scale_summary["majority_is_target_pct"] = (cv_scale_summary["majority_is_target_pct"] * 100).round(1)

print(f"\n{'='*60}")
print("CV Scale Summary")
print(f"{'='*60}")
print(cv_scale_summary[[
    "scale_name", "total_items", "cv_pass_count",
    "cv_pass_rate_pct", "mean_c_value", "mean_d_value",
    "mean_target_rating", "mean_agreement"
]].to_string(index=False))

# ── Human review columns ──────────────────────────────────────────────────────
df_cv_summary["human_review_pass"] = True
df_cv_summary["human_comments"]    = ""

# ── Write Excel ───────────────────────────────────────────────────────────────
with pd.ExcelWriter(P4_OUT, engine="openpyxl") as writer:
    df_sme_raw      .to_excel(writer, sheet_name="CV_All_SME_Ratings", index=False)
    df_cv_summary   .to_excel(writer, sheet_name="CV_Item_Summary",    index=False)
    cv_scale_summary.to_excel(writer, sheet_name="CV_Scale_Summary",   index=False)
    pd.DataFrame(run_log_p4).to_excel(writer, sheet_name="Run_Log",    index=False)

# ── Delete checkpoint on successful completion ────────────────────────────────
if P4_CHK.exists():
    P4_CHK.unlink()
    print("🗑️  Checkpoint deleted — run completed successfully.")

cv_pass_total = df_cv_summary["cv_pass"].sum()
print(f"\n✅ Phase 4 complete.")
print(f"   {int(cv_pass_total)}/{len(df_cv_summary)} items passed CV thresholds "
      f"(c≥{MIN_C_VALUE}, d≥{MIN_D_VALUE}).")
print(f"📄 Output: {P4_OUT}")


# ─── PHASE 5: Pseudo-Factor Analysis and Final Review ────────────────────────────────────────

**Input:** `04_content_validity.xlsx` → sheet `CV_Item_Summary` (respects `human_review_pass`)  
**Output:** `05_pseudo_factor_analysis.xlsx`  
**Sheets:** `PFA_Item_Results`, `PFA_Scale_Summary`, `Discriminant_Matrix`, `Item_Scale_Heatmap`, `Loading_Matrix_All`, `Loading_Matrix_CV_Pass`, `Fit_Statistics`, `Eigenvalues`, `High_Pass_Loadings`, `All_Pass_Loadings`, `Pass_Rules_Definition`

---

### What this phase does

Phase 5 is the statistical core of the pipeline. Using sentence-transformer embeddings as a semantic proxy for item inter-correlations, it applies factor analysis to evaluate each item's psychometric properties *before* collecting any real human response data — the Pseudo-Factor Analysis (PFA) technique introduced by Guenole et al. (2025) and independently evaluated by Suárez-Álvarez et al. (2026).

**Methodological basis:**

> Guenole et al. (2025) introduced PFA and demonstrated that the *substitutability assumption* — that embedding similarity can approximate empirical item correlations — holds well enough for early-stage item selection across two well-established personality frameworks (NEO and HEXACO). Suárez-Álvarez et al. (2026) conducted the first independent evaluation of PFA, confirming its utility while identifying boundary conditions. Milano et al. (2025) provide converging evidence that LLM-derived embeddings capture a-priori factorial structure with moderate-to-high correspondence to human-response-based factor solutions across four validated personality tests. All of this validation work was conducted on **standard Likert / dominance instruments**, which is why this pipeline generates dominance items in Phase 2.

---

### Key design decisions

**1. Multi-transformer aggregation (Guenole et al., 2024)**

One or more sentence transformer models can be specified in `pipeline_settings.json` under `transformer_models`. If multiple models are listed, their cosine similarity matrices are averaged before factor analysis — Guenole et al. (2025) found that aggregated multi-transformer matrices consistently outperform any single transformer in factor structure recovery. The default is `all-MiniLM-L6-v2` only. To use the dual-model approach from the original Guenole et al. (2025) design, add `all-mpnet-base-v2` to the list. The `Fit_Statistics` sheet records which models were used. The heatmap and projection calculations use the first model in the list (the primary model) only, to ensure consistent embedding dimensionality.

**2. Factor extraction method**

The `factor_extraction_method` setting in `pipeline_settings.json` controls how the factor analysis extracts loadings from the similarity matrix. Three values are supported:

- **`principal`** *(default)* — principal-axis extraction. PCA-style decomposition that accounts for total variance. This is the convention used in the foundational PFA literature (Guenole et al., 2024; Milano et al., 2025) and is recommended when benchmarking against published PFA results.
- **`minres`** — minimum residual EFA. An OLS-style common-factor extraction that accounts for shared variance only, treating item uniqueness as separate. Loading patterns are typically very similar to principal extraction on high-quality similarity matrices.
- **`ml`** — maximum-likelihood EFA. Common-factor extraction with a likelihood-based loss. Useful for researchers who want EFA-style estimation, though the standard ML diagnostics (CFI, TLI, RMSEA) remain unreported because the input is a similarity matrix rather than a sample correlation matrix (see Decision 7 — Model fit).

The choice does not affect the rotation strategy, DAAL identity logic, discriminant ratio, or the reported RMSR fit metric. Pass-rule thresholds may behave slightly differently across methods on the same matrix; if you change the extraction method, validate pass rates on a known-good run before relying on the existing thresholds.

**3. Flag-driven reversal for embedding**

In the dominance framework the keying direction of every item is already known from Phase 2 (`item_keying = positive` or `item_keying = negative`). There is no need for heuristic avoidance-pattern detection or LLM classification — a negatively keyed item is, by construction, the item that requires reversal before embedding under the atomic-reversed strategy. The pipeline uses the `item_keying` flag directly and calls the LLM only for the paraphrase rewrite when `atomic_reversed` encoding is requested. This makes the behaviour deterministic and auditable.

**4. Encoding strategy — atomic vs atomic-reversed (Guenole et al., 2024)**

Two encoding strategies are supported:

- **`atomic`** — items are embedded exactly as written. Positively keyed items carry the full construct meaning; negatively keyed items carry their reversed meaning. This is the standard approach that performed best on HEXACO in Guenole et al. (2025) (98% factor recovery).
- **`atomic_reversed`** — negatively keyed items are paraphrased to their positive-pole equivalents by an LLM before embedding; positively keyed items are embedded unchanged. All items then share the same semantic direction. This performed best on NEO-type instruments in Guenole et al. (2025) (94% factor recovery vs. 71% for standard atomic).

The `encoding_strategy` setting in `pipeline_settings.json` accepts `atomic`, `atomic_reversed`, or `both`. When `both` is selected, the pipeline runs each scale under both strategies, reports loadings for each, and the reviewer can select the better-performing approach per scale. The recommendation is:

- NEO-type / Big Five instruments → `atomic_reversed`
- HEXACO instruments → `atomic`
- Unknown frameworks → `both`

The `encoding_strategy_primary` column on each item records which strategy was used to evaluate that scale's pass rules. For `atomic` and `atomic_reversed`, this is that single strategy; for `both`, it defaults to whichever achieved higher mean communality on that scale (see `PFA_Scale_Summary → primary_encoding_chosen`).

**5. Dual encoding report (atomic + macro)**

Independently of the atomic/atomic-reversed choice, the macro encoding (concatenate all items in a scale; compute item-to-scale cosine similarity) is always computed and reported as a convergent-validity check. Tucker's congruence between the atomic single-factor loading vector and the macro item-to-scale similarity vector provides a scale-level diagnostic (see Decision 5).


**6. DAAL factor identity**

In the joint multi-scale factor analysis, each factor is labelled using the **Dominant Average Absolute Loading (DAAL)** approach (Guenole et al., 2024). For each extracted factor, the mean of the absolute loadings of every item belonging to a given scale is computed — this is the *average absolute loading* for that scale on that factor. This is done for every scale on every factor. The scale with the *dominant* (highest) average absolute loading on a given factor is then assigned as that factor's label. A scale's factor identity is confirmed if the factor labelled with its name is also the factor on which that scale's items load most strongly. Where two or more scales produce the same dominant average absolute loading on the same factor, the label cannot be uniquely assigned and the factor is marked `Unassigned_F{n}`. These unassigned factors still appear in the `Discriminant_Matrix` for completeness but do not contribute to DAAL identity confirmation. DAAL results are reported as flags for human review — not as hard pass/fail criteria — because heterogeneous scale frameworks may produce uninterpretable joint solutions.

**7. Model fit — RMSR only**

RMSR (Root Mean Square Residual) is the sole reported fit metric. CFI, TLI, and RMSEA are not computed — they require a known sample size and are unreliable in PFA contexts (Guenole et al., 2024; Suárez-Álvarez et al., 2026). Values below 0.08 indicate acceptable fit; values around 0.03–0.05 indicate good fit.

---

### Output sheets — what each one contains and when to use it

**`PFA_Item_Results`** is the primary per-item results sheet. Every item that entered PFA has its full set of metrics here — atomic loadings (and atomic-reversed where relevant), macro similarity, pass rules, discriminant validity flags, and DAAL identity. This is the sheet to use when making item-level decisions.

**`PFA_Scale_Summary`** provides one row per scale with aggregated statistics, DAAL identity, and discriminant flag counts. Start here for a scale-level overview before drilling into individual items.

**`Discriminant_Matrix`** contains the raw FA loadings from the joint multi-scale analysis — one row per item, one column per extracted factor in rotation order. Columns are labelled with the scale name where DAAL resolved cleanly, or `Unassigned_F{n}` where it did not. This is the most psychometrically rigorous view of inter-scale relationships, but it only covers scales where the joint FA ran successfully, and the column order follows FA rotation rather than your scale definitions. Use this sheet when you want to examine the precise factor-analytic structure or investigate a specific cross-loading value.

**`Item_Scale_Heatmap`** is the most human-readable inter-scale view. It contains one row per item and one column per scale in `scales.json` order, with values representing cosine similarity between each item's embedding and each scale's macro embedding. No factor analysis or DAAL is involved — every item gets a value against every scale regardless of whether DAAL resolved that scale. Values typically range from roughly 0.3 to 0.8. Scanning across a row, the highest value should fall on the item's own scale column; items where the peak falls elsewhere are discriminant validity concerns. Use this sheet for visual inspection and for reviewing scales that DAAL could not assign.

**`Loading_Matrix_All`** restructures the joint FA loadings as an item × scale matrix (rather than item × factor), covering all Phase 4 items including those that failed CV. Items that did not enter the PFA solution have *projected* loadings estimated via similarity-weighted projection onto the joint factor space — these are approximations and are flagged `in_pfa_solution = False`.

**`Loading_Matrix_CV_Pass`** is identical in structure to `Loading_Matrix_All` but contains only items that passed CV. All loadings here are exact FA values with no projections.

**`Fit_Statistics`** reports RMSR and residual distribution statistics per scale and per encoding strategy, along with the transformer models used. Check `fit_acceptable` here before interpreting item-level results for any scale.

**`High_Pass_Loadings`** and **`All_Pass_Loadings`** are filtered subsets of `PFA_Item_Results` containing only HIGH_PASS items and all trial-ready items respectively, with a reduced column set for easier review.

---

### Six pass/fail rules (configurable in `pipeline_settings.json`)

| Rule | Criterion | Default threshold |
|---|---|---|
| Rule 1 – Strong Single Factor | Atomic single-factor loading ≥ threshold | 0.40 |
| Rule 2 – Adequate Single Factor | Atomic single-factor loading ≥ threshold | 0.30 |
| Rule 3 – Adequate Communality | Communality ≥ threshold | 0.10 |
| Rule 4 – Clean Factor Structure | Max secondary loading < threshold | 0.35 |
| Rule 5 – Cross-Sample Viable | All-items loading ≥ threshold | 0.20 |
| Rule 6 – Keying-Viable | Within-keying loading ≥ threshold | 0.25 |

Rule 5 is the former `Cross_Location_Viable` rule. It now checks whether the item loads at a usable level on the all-items (pooled positive + negative) factor — i.e., whether it is cross-keying viable.

Rule 6 is the former `Within_Location_Viable` rule. It now checks whether the item loads adequately within its own keying stratum, analysing positively keyed items and negatively keyed items separately. An item can fail Rule 6 even when it passes Rule 5 if the positive-only or negative-only sub-factor is weaker than the all-items factor — a useful within-direction diagnostic.

Rules 1, 2, 5, and 6 use absolute values to handle factor sign flips, which are expected in PFA and do not reflect item quality.

An item is **HIGH_PASS** if it meets Rules 1, 3, and 4. It is **STANDARD_PASS** if it meets at least `min_rules_to_pass` rules (default 3). Items below this threshold are **FAIL**.

---

### Human review columns added to output

| Column | Default | Purpose |
|---|---|---|
| `encoding_strategy_primary` | System-set | `atomic`, `atomic_reversed`, or the chosen one when `both` is configured |
| `item_reversed_for_embedding` | System-set | `True` when atomic-reversed encoding was applied and the item was paraphrased for embedding; original `item_text` always preserved |
| `pfa_pass_level` | System-set | `HIGH_PASS`, `STANDARD_PASS`, or `FAIL` |
| `human_review_pass` | `True` | Override PFA verdict — set `False` to exclude a passing item; set `True` to reinstate a failed one |
| `human_comments` | *(blank)* | Notes — especially important for DAAL failures, discriminant flags, and reversed items |

**Review the output after this phase completes.** A suggested review sequence:

1. `PFA_Scale_Summary` — scales where `daal_identity_confirmed = False` and high `disc_flag_count`
2. `Item_Scale_Heatmap` — visual scan for items whose peak similarity falls on the wrong scale column
3. `PFA_Item_Results` — item-level decisions; filter on `discriminant_flag_review = True` and `item_keying = negative` first

---

### Final Review: Item Pool Assembly

After PFA completes, Phase 5 assembles the final human-review-ready item pool. Only items that passed **both** Phase 4 (Content Validity) and PFA — as determined by the `human_review_pass` columns from each — are included in `Review_Pool`.

**Additional output sheets:**

- **`Review_Pool`** — all passing items with full quality metadata. Two blank columns for reviewer decisions.
- **`Rejected_Items`** — excluded items, preserved for transparency.
- **`Pool_Statistics`** — one row per scale: item counts by keying direction (`positive_keyed_items`, `negative_keyed_items`), mean CV rating, mean atomic loading, bias flag counts, reversal counts.
- **`Quality_Flags`** — every item with at least one automated concern: readability, bias, low CV rating, mapping failure, standard pass only, discriminant flag, DAAL failure, or negatively keyed items with weak within-keying loadings. Review these first.
- **`Recommendations`** — scale-level guidance with HIGH / MEDIUM priority: scales with too few items, skewed keying balance, high mean FK grade, multiple discriminant flags, DAAL failures, and reversal counts.

**Reviewer workflow:**

1. Open `05_pseudo_factor_analysis.xlsx`
2. Start with `Recommendations` and `Quality_Flags` sheets
3. Work through `Review_Pool` scale by scale
4. Record decisions in `human_decision` and rationale in `human_notes`
5. Items marked `reject` or `modify` can be flagged for regeneration by re-running earlier phases with adjusted settings

> **This is a pre-trial item pool.** Real human response data and full psychometric validation are required before any items are used operationally. The pipeline replaces weeks of manual drafting and initial screening — it does not replace empirical validation (Guenole et al., 2024; Suárez-Álvarez et al., 2026).


In [ ]:
# ==============================================================================
# PHASE 5 — Pseudo-Factor Analysis (PFA) + Item Pool Assembly
# ==============================================================================
# Reads:  output/04_content_validity.xlsx  (CV_Item_Summary sheet)
#         output/03_readability_bias.xlsx   (Items_With_Readability sheet)
# Writes: output/05_pseudo_factor_analysis.xlsx
#
# Sheets (PFA):
#   PFA_Item_Results        — per-item PFA metrics for both encoding strategies
#   PFA_Scale_Summary       — scale-level summary with DAAL identity
#   Discriminant_Matrix     — full item × factor loading matrix (all factors,
#                             all items, factor-index order including Unassigned)
#   Item_Scale_Heatmap      — item × scale cosine similarity matrix in scales.json
#                             order; every item against every scale, no DAAL dependency
#   Loading_Matrix_All      — item × scale loading matrix, ALL Phase 4 items
#   Loading_Matrix_CV_Pass  — item × scale loading matrix, cv_pass items only
#   Fit_Statistics          — RMSR + residual stats per scale
#   Eigenvalues             — eigenvalue tables per scale
#   High_Pass_Loadings      — items meeting HIGH_PASS criteria
#   All_Pass_Loadings       — all items ready for trial
#   Pass_Rules_Definition   — threshold documentation
#
# Sheets (Pool Assembly, appended to same file):
#   Review_Pool             — passing items with human review columns
#   Rejected_Items          — excluded items for audit
#   Pool_Statistics         — scale-level summary
#   Quality_Flags           — items with automated concerns
#   Recommendations         — scale-level HIGH/MEDIUM priority guidance
#
# KEY DESIGN DECISIONS (dominance-item framework):
#   1. Multi-transformer aggregation — similarity matrices averaged across
#      configured transformer models before FA (Guenole et al. 2025)
#   2. Keying-driven reversal — item_keying == 'negative' deterministically
#      triggers reversal for embedding under atomic_reversed encoding. No
#      heuristic detection. LLM is called only to paraphrase the reversal.
#   3. Encoding strategy — atomic | atomic_reversed | both. Guenole et al.
#      (2025) reported atomic-reversed 94% factor recovery on NEO vs 71% for
#      standard atomic; standard atomic 98% on HEXACO. The pipeline can run
#      either or both.
#   4. DAAL factor identity (Guenole et al. 2025)
#   5. RMSR only — CFI/TLI/RMSEA not computed (require sample size)
#   6. Item_Scale_Heatmap uses primary model (first in list) for both item
#      and scale embeddings to ensure consistent dimensionality
#   7. item_keying carries keying direction (positive / negative) from Phase 2
# ==============================================================================

# ── Reload settings ───────────────────────────────────────────────────────────
with open(SETTINGS_FILE, "r", encoding="utf-8") as f:
    CFG = json.load(f)
print("✓ Settings reloaded")

import re as _re
from factor_analyzer import FactorAnalyzer

P5      = CFG["phase_05_pseudo_factor_analysis"]
P3_CFG  = CFG["phase_03_readability_bias"]
P4_CFG  = CFG["phase_04_content_validity"]

P5_IN   = BASE_OUTPUT_DIR / CFG["phase_04_content_validity"]["output_file"]
P5_OUT  = BASE_OUTPUT_DIR / P5["output_file"]

assert P5_IN.exists(), f"Input not found: {P5_IN}. Run Phase 4 first."

# ── Settings ──────────────────────────────────────────────────────────────────
PASS_RULES_CFG    = P5["pass_rules"]
MIN_RULES         = P5["min_rules_to_pass"]
RMSR_THRESHOLD    = P5["model_fit_thresholds"]["rmsr_max"]
DISC_RATIO_MIN    = P5.get("discriminant_validity", {}).get("target_to_max_ratio_min", 1.5)
DISC_CEILING      = P5.get("discriminant_validity", {}).get("max_cross_loading_ceiling", 0.45)
DISC_CEIL_ENABLED = P5.get("discriminant_validity", {}).get("ceiling_enabled", True)
MIN_ITEMS_PFA     = P5.get("min_items_for_pfa", 4)
MIN_SCALES_JOINT  = P5.get("min_scales_for_joint_analysis", 3)
TRANSFORMER_MODELS = P5.get("transformer_models", [
    "all-MiniLM-L6-v2",
    "all-mpnet-base-v2"
])
ENCODING_STRATEGY = P5.get("encoding_strategy", "both").lower().strip()
assert ENCODING_STRATEGY in ("atomic", "atomic_reversed", "both"), (
    f"encoding_strategy must be 'atomic', 'atomic_reversed', or 'both'; got {ENCODING_STRATEGY!r}"
)

# ── Load Phase 4 data ─────────────────────────────────────────────────────────
with pd.ExcelFile(P5_IN) as xls:
    df_cv_all = pd.read_excel(xls, sheet_name="CV_Item_Summary")

if "human_review_pass" in df_cv_all.columns:
    df_cv = df_cv_all[df_cv_all["human_review_pass"] != False].copy().reset_index(drop=True)
    n_excluded = len(df_cv_all) - len(df_cv)
    print(f"✓ Phase 4 human review: {n_excluded} items excluded by human_review_pass")
else:
    df_cv = df_cv_all[df_cv_all["cv_pass"] == True].copy().reset_index(drop=True)
    n_excluded = len(df_cv_all) - len(df_cv)

print(f"✓ Phase 4 items loaded : {len(df_cv_all)}")
print(f"  Entering PFA         : {len(df_cv)}")
print(f"  Excluded             : {n_excluded}")

# Defensive: ensure item_keying column is present (carried forward from Phase 2)
if "item_keying" not in df_cv.columns:
    logger.warning(
        "item_keying column not found in CV_Item_Summary — assuming all items "
        "are positively keyed. Re-run Phase 2 onward if this is unexpected."
    )
    df_cv["item_keying"]     = "positive"
    df_cv_all["item_keying"] = "positive"

# ── Merge Phase 3 readability columns ─────────────────────────────────────────
with pd.ExcelFile(BASE_OUTPUT_DIR / P3_CFG["output_file"]) as xls:
    df_read = pd.read_excel(xls, sheet_name="Items_With_Readability")[
        ["item_id", "flesch_kincaid_grade", "bias_flagged_dims",
         "bias_flag_count", "bias_any_flagged", "was_simplified",
         "simplification_attempts"]
    ].copy()

df_p5 = df_cv.merge(df_read, on="item_id", how="left")

# ── Load transformer models ───────────────────────────────────────────────────
print(f"\nLoading {len(TRANSFORMER_MODELS)} sentence transformer(s)...")
st_models = []
for model_name in TRANSFORMER_MODELS:
    print(f"  Loading: {model_name}")
    st_models.append(SentenceTransformer(model_name))
print(f"✓ {len(st_models)} transformer(s) loaded")

# Primary model (index 0) used for heatmap and projection — consistent dims.
PRIMARY_MODEL = st_models[0]


# ==============================================================================
# KEYING-DRIVEN REVERSAL FOR ATOMIC_REVERSED ENCODING
# ==============================================================================
# In the dominance framework, the keying direction of every item is known from
# Phase 2 (item_keying = 'positive' or 'negative'). Under atomic_reversed
# encoding, negatively keyed items are paraphrased to their positive-pole
# equivalents before embedding. This paraphrase uses an LLM call but is
# triggered DETERMINISTICALLY by the item_keying flag — there is NO heuristic
# detection step. This makes the process auditable and reproducible.
#
# The original item_text is always preserved in all output columns.
# ==============================================================================
_reversal_cache: Dict[str, str] = {}


def paraphrase_to_positive(item_id: str, item_text: str, scale_name: str) -> str:
    """
    Rewrite a negatively keyed item as its positive-pole equivalent for embedding.
    Only called when atomic_reversed encoding is active AND item_keying == 'negative'.
    The returned string is used ONLY for embedding; item_text is preserved.
    """
    if item_id in _reversal_cache:
        return _reversal_cache[item_id]

    rcfg   = P5.get("reversal_rewrite", {})
    prompt = f"""You are rewriting a negatively keyed personality survey item as a
positive-pole equivalent, for use in a semantic embedding step only.

Item: "{item_text}"
Construct: {scale_name}

The item above is NEGATIVELY KEYED — meaning that people who are LOW on the
target construct endorse it more strongly. Rewrite it as a POSITIVELY KEYED
equivalent with the same underlying semantic content, so that people who are
HIGH on the construct would endorse the rewritten form.

Requirements for the rewrite:
- Natural first-person "I" statement in present tense
- Maximum 9 words
- No negations, no avoidance language, no "rarely/seldom/never"
- Express the high pole of the construct clearly
- Preserve the specific behaviour — do not generalise

Respond in JSON only:
{{"embedding_text": "<rewritten positive-pole item>"}}"""

    try:
        raw = llm_call(
            prompt,
            model       = rcfg.get("model", "anthropic/claude-haiku-4.5"),
            max_tokens  = rcfg.get("max_tokens", 100),
            temperature = rcfg.get("temperature", 0.0)
        )
        cleaned = _re.sub(r'```json|```', '', raw).strip()
        match   = _re.search(r'\{.*?\}', cleaned, _re.DOTALL)
        if not match:
            raise ValueError("No JSON object found in response")
        result  = json.loads(match.group())
        emb_txt = str(result.get("embedding_text", item_text)).strip()
        if not emb_txt.startswith("I "):
            emb_txt = item_text
    except Exception as e:
        logger.warning(f"Reversal rewrite failed for {item_id}: {e} — using original")
        emb_txt = item_text

    _reversal_cache[item_id] = emb_txt
    return emb_txt


def embedding_text_for(item_id: str, item_text: str, item_keying: str,
                       scale_name: str, strategy: str) -> tuple:
    """
    Return (embedding_text, was_reversed_for_embedding) for the given strategy.

    strategy = 'atomic'          -> every item embedded as written
    strategy = 'atomic_reversed' -> negatively keyed items paraphrased to positive
    """
    if strategy == "atomic_reversed" and str(item_keying).lower() == "negative":
        return paraphrase_to_positive(item_id, item_text, scale_name), True
    return item_text, False


def get_embedding_texts(items_df: pd.DataFrame, scale_name: str,
                        strategy: str) -> tuple:
    emb_texts, rev_flags = [], []
    for _, row in items_df.iterrows():
        txt, rev = embedding_text_for(
            row["item_id"], row["item_text"],
            row.get("item_keying", "positive"),
            scale_name, strategy
        )
        emb_texts.append(txt)
        rev_flags.append(rev)
    return emb_texts, rev_flags


# ==============================================================================
# PREPROCESSING
# ==============================================================================
def preprocess_for_embedding(text: str) -> str:
    t = str(text).strip().lower()
    t = _re.sub(r'^i\s+', '', t)
    t = _re.sub(r'[.!?,;:]+$', '', t)
    t = _re.sub(r'\s+', ' ', t).strip()
    return t


# ==============================================================================
# ENCODING
# ==============================================================================
def encode_atomic_aggregate(texts: List[str]) -> np.ndarray:
    """Aggregate cosine similarity matrix averaged across all transformers."""
    preprocessed = [preprocess_for_embedding(t) for t in texts]
    sim_matrices = []
    for model in st_models:
        embs = model.encode(preprocessed, show_progress_bar=False)
        sim  = np.array(sk_cosine(embs))
        np.fill_diagonal(sim, 1.0)
        sim_matrices.append(sim)
    agg = np.mean(sim_matrices, axis=0)
    np.fill_diagonal(agg, 1.0)
    return agg

def encode_atomic_raw(texts: List[str]) -> np.ndarray:
    """Raw embeddings from primary model only — used for projection and heatmap."""
    preprocessed = [preprocess_for_embedding(t) for t in texts]
    return PRIMARY_MODEL.encode(preprocessed, show_progress_bar=False)

def encode_macro_aggregate(texts: List[str]) -> np.ndarray:
    """
    Item-to-scale cosine similarities averaged across all transformers.
    Returns shape: (n_items,)
    """
    preprocessed  = [preprocess_for_embedding(t) for t in texts]
    concatenated  = " ".join(preprocessed)
    sims_per_model = []
    for model in st_models:
        scale_vec  = model.encode([concatenated],  show_progress_bar=False)[0]
        item_vecs  = model.encode(preprocessed,    show_progress_bar=False)
        scale_norm = scale_vec  / (np.linalg.norm(scale_vec)                        + 1e-9)
        items_norm = item_vecs  / (np.linalg.norm(item_vecs, axis=1, keepdims=True) + 1e-9)
        sims_per_model.append(items_norm @ scale_norm)
    return np.mean(sims_per_model, axis=0)

def encode_macro_primary(texts: List[str]) -> np.ndarray:
    """
    Scale macro embedding using primary model only.
    Returns a single vector of shape (primary_model_dim,).
    Used for heatmap — must match encode_atomic_raw dimensionality.
    """
    preprocessed = [preprocess_for_embedding(t) for t in texts]
    concatenated  = " ".join(preprocessed)
    return PRIMARY_MODEL.encode([concatenated], show_progress_bar=False)[0]

def embeddings_to_sim_matrix(embeddings: np.ndarray) -> np.ndarray:
    sim = np.array(sk_cosine(embeddings))
    np.fill_diagonal(sim, 1.0)
    return sim


# ==============================================================================
# FACTOR ANALYSIS
# ==============================================================================
def run_fa(sim_matrix: np.ndarray, n_factors: int):
    n = sim_matrix.shape[0]
    if n < n_factors + 2:
        return None, None
    try:
        fa = FactorAnalyzer(
            n_factors = n_factors,
            rotation  = None if n_factors == 1 else "oblimin",
            method    = FA_METHOD
        )
        fa.fit(sim_matrix)
        loadings     = fa.loadings_
        eigenvals, _ = fa.get_eigenvalues()
        return loadings, eigenvals
    except Exception as e:
        logger.warning(f"FA failed (n={n}, k={n_factors}): {e}")
        return None, None

def residual_stats(sim_matrix: np.ndarray, loadings: np.ndarray) -> Dict:
    if loadings is None:
        return {"rmsr": 1.0, "residual_mean": 0, "residual_std": 0,
                "residual_max_abs": 0, "n_large_residuals": 0, "fit_acceptable": False}
    n_factors  = loadings.shape[1] if loadings.ndim > 1 else 1
    reproduced = (
        loadings @ loadings.T if n_factors > 1
        else np.outer(loadings.flatten(), loadings.flatten())
    )
    residuals = sim_matrix - reproduced
    np.fill_diagonal(residuals, 0)
    off  = residuals[np.triu_indices_from(residuals, k=1)]
    rmsr = float(np.sqrt(np.mean(off**2)))
    return {
        "rmsr":               round(rmsr,                       4),
        "residual_mean":      round(float(np.mean(off)),        4),
        "residual_std":       round(float(np.std(off)),         4),
        "residual_max_abs":   round(float(np.max(np.abs(off))), 4),
        "n_large_residuals":  int(np.sum(np.abs(off) > 0.10)),
        "fit_acceptable":     rmsr <= RMSR_THRESHOLD
    }


# ==============================================================================
# ITEM-LEVEL RULE EVALUATION
# ==============================================================================
# Rules 5 and 6 are the former Cross_Location_Viable / Within_Location_Viable
# rules, renamed for the dominance framework:
#   Rule 5 — Cross_Sample_Viable: item loads on the all-items (pooled positive
#            + negative) single factor at a usable level
#   Rule 6 — Keying_Viable:       item loads on the within-keying single factor
#            (positive items analysed separately from negative items)
# Rules 1, 2, 5, 6 use absolute values to handle factor sign flips.
# ==============================================================================
def evaluate_item_rules(
    atomic_single_load: float,
    communality:        float,
    max_sec_load:       float,
    within_load:        float
) -> tuple:
    rules_passed = []
    pass_results = {}

    def _thr(key, default):
        entry = PASS_RULES_CFG.get(key, {})
        return entry.get("threshold", default)

    rules_def = {
        "Rule_1_Strong_Single_Factor":   (abs(atomic_single_load), _thr("Rule_1_Strong_Single_Factor",   0.40), ">="),
        "Rule_2_Adequate_Single_Factor": (abs(atomic_single_load), _thr("Rule_2_Adequate_Single_Factor", 0.30), ">="),
        "Rule_3_Adequate_Communality":   (communality,             _thr("Rule_3_Adequate_Communality",   0.10), ">="),
        "Rule_4_Clean_Factor_Structure": (max_sec_load,            _thr("Rule_4_Clean_Factor_Structure", 0.35), "<"),
        "Rule_5_Cross_Sample_Viable":    (abs(atomic_single_load), _thr("Rule_5_Cross_Sample_Viable",    0.20), ">="),
        "Rule_6_Keying_Viable":          (abs(within_load),        _thr("Rule_6_Keying_Viable",          0.25), ">="),
    }
    for rule, (val, thresh, op) in rules_def.items():
        if val is None:
            passed = False
        elif op == ">=":
            passed = val >= thresh
        else:
            passed = val < thresh
        pass_results[rule] = "PASS" if passed else "FAIL"
        if passed:
            rules_passed.append(rule)
    return rules_passed, pass_results


# ==============================================================================
# JOINT MULTI-SCALE ANALYSIS  (DAAL + Discriminant Validity)
# ==============================================================================
def run_joint_analysis(df: pd.DataFrame, encoding: str = "atomic",
                       reversal_strategy: str = "atomic") -> Dict:
    """
    encoding          : 'atomic' (item-level) or 'macro' (scale-level)
    reversal_strategy : 'atomic' or 'atomic_reversed' — controls whether
                        negatively keyed items are paraphrased before embedding.
    """
    scale_counts = df.groupby("scale_name")["item_id"].count()
    valid_scales = sorted(scale_counts[scale_counts >= MIN_ITEMS_PFA].index.tolist())
    n_scales     = len(valid_scales)

    if n_scales < MIN_SCALES_JOINT:
        logger.warning(
            f"Joint analysis skipped: only {n_scales} scales with >= {MIN_ITEMS_PFA} items "
            f"(need {MIN_SCALES_JOINT})."
        )
        return {"available": False, "scales_included": valid_scales}

    df_joint     = df[df["scale_name"].isin(valid_scales)].copy().reset_index(drop=True)
    item_ids     = df_joint["item_id"].tolist()
    scale_labels = df_joint["scale_name"].tolist()

    if encoding == "macro":
        scale_vecs = {}
        for sn in valid_scales:
            sn_df = df_joint[df_joint["scale_name"] == sn]
            emb_texts, _ = get_embedding_texts(sn_df, sn, reversal_strategy)
            scale_vecs[sn] = encode_macro_primary(emb_texts)
        emb_matrix = np.array([scale_vecs[sn] for sn in valid_scales])
        sim_matrix = embeddings_to_sim_matrix(emb_matrix)
        item_level = False
    else:
        emb_texts_list = []
        for _, row in df_joint.iterrows():
            txt, _ = embedding_text_for(
                row["item_id"], row["item_text"],
                row.get("item_keying", "positive"),
                row["scale_name"], reversal_strategy
            )
            emb_texts_list.append(txt)
        sim_matrix = encode_atomic_aggregate(emb_texts_list)
        raw_embs   = encode_atomic_raw(emb_texts_list)
        item_level = True

    loadings, eigenvals = run_fa(sim_matrix, n_scales)
    if loadings is None:
        logger.warning(f"Joint {encoding} FA failed.")
        return {"available": False, "scales_included": valid_scales}

    # DAAL factor labelling
    factor_labels = {}
    scale_daal    = {}
    if item_level:
        for sn in valid_scales:
            mask = [i for i, s in enumerate(scale_labels) if s == sn]
            scale_daal[sn] = {
                f: float(np.mean(np.abs(loadings[mask, f])))
                for f in range(n_scales)
            }
        for f in range(n_scales):
            daal_scores      = {sn: scale_daal[sn][f] for sn in valid_scales}
            factor_labels[f] = max(daal_scores, key=daal_scores.get)
    else:
        for si, sn in enumerate(valid_scales):
            scale_daal[sn] = {f: float(np.abs(loadings[si, f])) for f in range(n_scales)}
        for f in range(n_scales):
            daal_scores      = {sn: scale_daal[sn][f] for sn in valid_scales}
            factor_labels[f] = max(daal_scores, key=daal_scores.get)

    from collections import Counter as _Counter
    label_counts = _Counter(factor_labels.values())
    for f in list(factor_labels):
        if label_counts[factor_labels[f]] > 1:
            factor_labels[f] = f"Unassigned_F{f+1}"

    scale_to_factor = {v: k for k, v in factor_labels.items()
                       if not v.startswith("Unassigned")}

    daal_identity = {}
    for sn in valid_scales:
        if sn not in scale_to_factor:
            daal_identity[sn] = False
            continue
        expected_f = scale_to_factor[sn]
        if item_level:
            mask             = [i for i, s in enumerate(scale_labels) if s == sn]
            mean_on_expected = float(np.mean(np.abs(loadings[mask, expected_f])))
            all_means        = [float(np.mean(np.abs(loadings[mask, f]))) for f in range(n_scales)]
            daal_identity[sn] = (mean_on_expected == max(all_means))
        else:
            si = valid_scales.index(sn)
            daal_identity[sn] = (np.argmax(np.abs(loadings[si, :])) == expected_f)

    item_cross_loadings: Dict[str, Dict] = {}
    if item_level:
        for i, iid in enumerate(item_ids):
            item_cross_loadings[iid] = {
                factor_labels.get(f, f"F{f+1}"): round(float(loadings[i, f]), 4)
                for f in range(n_scales)
            }

    fit = residual_stats(sim_matrix, loadings)

    result = {
        "available":           True,
        "scales_included":     valid_scales,
        "factor_labels":       factor_labels,
        "scale_to_factor":     scale_to_factor,
        "scale_daal":          scale_daal,
        "daal_identity":       daal_identity,
        "item_cross_loadings": item_cross_loadings,
        "loadings":            loadings,
        "eigenvalues":         eigenvals,
        "rmsr":                fit["rmsr"],
        "fit_acceptable":      fit["fit_acceptable"],
        "item_level":          item_level,
        "item_ids":            item_ids,
        "scale_labels":        scale_labels,
        "reversal_strategy":   reversal_strategy,
    }
    if item_level:
        result["embeddings"] = raw_embs
    return result


def compute_discriminant_metrics(
    item_id:      str,
    target_scale: str,
    joint_atomic: Dict,
) -> Dict:
    empty = {
        "discriminant_available":           False,
        "discriminant_target_loading":      None,
        "discriminant_max_cross_loading":   None,
        "discriminant_target_to_max_ratio": None,
        "discriminant_closest_scale":       None,
        "discriminant_ceiling_exceeded":    None,
        "discriminant_pass":                None,
        "discriminant_flag_review":         False,
    }
    if not joint_atomic.get("available") or not joint_atomic.get("item_level"):
        return empty
    cross = joint_atomic["item_cross_loadings"].get(item_id)
    if cross is None:
        return empty
    target_loading = cross.get(target_scale)
    if target_loading is None:
        return empty

    other = {k: abs(v) for k, v in cross.items()
             if k != target_scale and not k.startswith("Unassigned")}
    if not other:
        return {**empty,
                "discriminant_available":           True,
                "discriminant_target_loading":      round(target_loading, 4),
                "discriminant_max_cross_loading":   0.0,
                "discriminant_target_to_max_ratio": None,
                "discriminant_closest_scale":       None,
                "discriminant_ceiling_exceeded":    False,
                "discriminant_pass":                True,
                "discriminant_flag_review":         False}

    max_cross_scale   = max(other, key=other.get)
    max_cross_loading = other[max_cross_scale]
    abs_target        = abs(target_loading)
    ratio             = round(abs_target / max_cross_loading, 3) if max_cross_loading > 0 else None
    ceiling_exceeded  = (max_cross_loading > DISC_CEILING) if DISC_CEIL_ENABLED else False
    disc_pass         = (ratio is not None and ratio >= DISC_RATIO_MIN) and (not ceiling_exceeded)

    return {
        "discriminant_available":           True,
        "discriminant_target_loading":      round(target_loading, 4),
        "discriminant_max_cross_loading":   round(max_cross_loading, 4),
        "discriminant_target_to_max_ratio": ratio,
        "discriminant_closest_scale":       max_cross_scale,
        "discriminant_ceiling_exceeded":    ceiling_exceeded,
        "discriminant_pass":                disc_pass,
        "discriminant_flag_review":         not disc_pass,
    }


# ==============================================================================
# LOADING MATRIX BUILDER
# ==============================================================================
def build_loading_matrix(df_source: pd.DataFrame, joint_atomic: Dict) -> pd.DataFrame:
    if not joint_atomic.get("available") or not joint_atomic.get("item_level"):
        return pd.DataFrame()

    factor_labels   = joint_atomic["factor_labels"]
    n_factors       = len(factor_labels)
    loadings_matrix = joint_atomic["loadings"]
    joint_embs      = joint_atomic["embeddings"]
    id_to_idx       = {iid: i for i, iid in enumerate(joint_atomic["item_ids"])}
    existing_ids    = set(joint_atomic["item_ids"])

    df_excl        = df_source[~df_source["item_id"].isin(existing_ids)].copy()
    excl_projected = {}
    if len(df_excl) > 0:
        excl_embs  = encode_atomic_raw(df_excl["item_text"].tolist())
        excl_norm  = excl_embs  / (np.linalg.norm(excl_embs,  axis=1, keepdims=True) + 1e-9)
        joint_norm = joint_embs / (np.linalg.norm(joint_embs, axis=1, keepdims=True) + 1e-9)
        projected  = (excl_norm @ joint_norm.T) @ loadings_matrix
        for i, iid in enumerate(df_excl["item_id"].tolist()):
            excl_projected[iid] = projected[i]

    rows = []
    for _, row in df_source.iterrows():
        iid  = row["item_id"]
        base = {
            "item_id":         iid,
            "scale_name":      row.get("scale_name", ""),
            "item_text":       row.get("item_text", ""),
            "item_keying":     row.get("item_keying", ""),
            "cv_pass":         row.get("cv_pass", False),
            "in_pfa_solution": iid in existing_ids,
        }
        vec = (loadings_matrix[id_to_idx[iid]] if iid in id_to_idx
               else excl_projected.get(iid, np.zeros(n_factors)))
        for f in range(n_factors):
            label = factor_labels.get(f, f"F{f+1}")
            base[f"loading_{label}"] = round(float(vec[f]), 4)
        rows.append(base)

    return pd.DataFrame(rows)


# ==============================================================================
# MAIN PFA PIPELINE
# ==============================================================================
# Which reversal strategies to run. For 'both' we run each strategy separately
# and report per-strategy loadings + Tucker's congruence. The 'primary' strategy
# (first in STRATEGIES_TO_RUN) drives the per-item output columns and the
# pass-rule evaluation; the secondary strategy adds parallel columns and
# per-strategy rows in Fit_Statistics.
# ==============================================================================
if ENCODING_STRATEGY == "atomic":
    STRATEGIES_TO_RUN = ["atomic"]
elif ENCODING_STRATEGY == "atomic_reversed":
    STRATEGIES_TO_RUN = ["atomic_reversed"]
else:
    STRATEGIES_TO_RUN = ["atomic", "atomic_reversed"]
PRIMARY_STRATEGY = STRATEGIES_TO_RUN[0]

print(f"\n{'='*60}")
print(f"PHASE 5: Pseudo-Factor Analysis (dominance-item framework)")
print(f"Transformers       : {TRANSFORMER_MODELS}")
print(f"Encoding strategy  : {ENCODING_STRATEGY}  (runs: {STRATEGIES_TO_RUN})")
print(f"Min rules to pass  : {MIN_RULES}")
print(f"Items entering PFA : {len(df_p5)}")
print(f"Discriminant ratio : >= {DISC_RATIO_MIN}")
print(f"Discriminant ceil  : {'<= ' + str(DISC_CEILING) if DISC_CEIL_ENABLED else 'disabled'}")
print(f"RMSR threshold     : <= {RMSR_THRESHOLD}")

n_neg_items = int((df_p5["item_keying"].astype(str).str.lower() == "negative").sum())
n_pos_items = int((df_p5["item_keying"].astype(str).str.lower() == "positive").sum())
print(f"Keying breakdown   : {n_pos_items} positive / {n_neg_items} negative")
print(f"{'='*60}\n")

# ── Step 0: Paraphrase pre-compute (only if atomic_reversed is active) ────────
if "atomic_reversed" in STRATEGIES_TO_RUN and n_neg_items > 0:
    print(f"Pre-computing positive-pole paraphrases for {n_neg_items} negatively keyed items...")
    neg_rows = df_p5[df_p5["item_keying"].astype(str).str.lower() == "negative"]
    for _, row in tqdm(neg_rows.iterrows(), total=len(neg_rows), desc="Reversal rewrites"):
        paraphrase_to_positive(row["item_id"], row["item_text"],
                               row.get("scale_name", ""))
    print(f"  ✓ {len(_reversal_cache)} paraphrases cached")
else:
    print("Skipping reversal rewrites (atomic-only encoding or no negatively keyed items).")

# ── Step 1: Joint multi-scale analyses (one run per strategy) ────────────────
print("\nRunning joint multi-scale analyses...")
joint_by_strategy: Dict[str, Dict] = {}
for strat in STRATEGIES_TO_RUN:
    j_atomic = run_joint_analysis(df_p5, encoding="atomic",
                                  reversal_strategy=strat)
    j_macro  = run_joint_analysis(df_p5, encoding="macro",
                                  reversal_strategy=strat)
    joint_by_strategy[strat] = {"atomic": j_atomic, "macro": j_macro}
    if j_atomic.get("available"):
        n_conf = sum(j_atomic["daal_identity"].values())
        print(f"  ✓ {strat:<16} atomic: "
              f"{len(j_atomic['scales_included'])} scales, RMSR={j_atomic['rmsr']:.4f}, "
              f"DAAL confirmed {n_conf}/{len(j_atomic['scales_included'])}")
    else:
        print(f"  ⚠ {strat:<16} atomic joint analysis not available")
    if j_macro.get("available"):
        print(f"  ✓ {strat:<16} macro : "
              f"{len(j_macro['scales_included'])} scales, RMSR={j_macro['rmsr']:.4f}")

# Primary joint analysis used for discriminant columns, loading matrices,
# and DAAL identity on PFA_Item_Results.
joint_atomic = joint_by_strategy[PRIMARY_STRATEGY]["atomic"]
joint_macro  = joint_by_strategy[PRIMARY_STRATEGY]["macro"]

# ── Step 1b: Loading matrices ─────────────────────────────────────────────────
print("\nBuilding loading matrices...")
df_loading_matrix_all     = build_loading_matrix(df_cv_all, joint_atomic)
df_loading_matrix_cv_pass = build_loading_matrix(df_p5,     joint_atomic)
if not df_loading_matrix_all.empty:
    print(f"  ✓ Loading_Matrix_All     : {len(df_loading_matrix_all)} items")
    print(f"  ✓ Loading_Matrix_CV_Pass : {len(df_loading_matrix_cv_pass)} items")

# ── Step 2: Per-scale analyses ────────────────────────────────────────────────
# Per-scale loop runs under the PRIMARY strategy. If ENCODING_STRATEGY == 'both'
# we additionally run a parallel pass under the SECONDARY strategy and store
# its single-factor loadings in a per-item dict keyed by item_id, so each
# PFA_Item_Results row can show loadings under both strategies.
# ==============================================================================
print("\nRunning per-scale analyses...")

pfa_rows:        List[Dict] = []
fit_stat_rows:   List[Dict] = []
eigenvalue_rows: List[Dict] = []
congruence_rows: List[Dict] = []

# Secondary-strategy per-item loadings (only populated if ENCODING_STRATEGY='both')
SECONDARY_STRATEGY: Optional[str] = (
    STRATEGIES_TO_RUN[1] if len(STRATEGIES_TO_RUN) > 1 else None
)
secondary_item_loadings: Dict[str, float] = {}

for scale_name, scale_grp in tqdm(df_p5.groupby("scale_name"), desc="Scales"):
    items     = scale_grp.reset_index(drop=True)
    ids_all   = items["item_id"].tolist()
    scale_def = scale_params_lookup.get(scale_name, {}).get("construct_definition", "")

    if len(ids_all) < MIN_ITEMS_PFA:
        logger.warning(f"{scale_name}: only {len(ids_all)} items — skipping.")
        continue

    # ── Primary-strategy embeddings ───────────────────────────────────────────
    emb_texts, rev_flags = get_embedding_texts(items, scale_name, PRIMARY_STRATEGY)

    # Atomic aggregate sim matrix + FA (primary strategy)
    atomic_sim   = encode_atomic_aggregate(emb_texts)
    a_ld1, a_eig = run_fa(atomic_sim, 1)
    a_single     = a_ld1.flatten() if a_ld1 is not None else np.zeros(len(emb_texts))
    a_fit        = residual_stats(atomic_sim, a_ld1)

    # Two-factor for secondary loading check
    a_ld2 = None
    if len(emb_texts) >= 6:
        a_ld2, _ = run_fa(atomic_sim, 2)

    # Macro aggregate item-to-scale similarities (primary strategy)
    macro_item_sims = encode_macro_aggregate(emb_texts)

    # Within-keying loadings (positive items vs negative items analysed separately)
    keying_loadings: Dict[str, Dict] = {}
    for keying_dir in ["positive", "negative"]:
        keying_items = items[
            items["item_keying"].astype(str).str.lower() == keying_dir
        ].reset_index(drop=True)
        if len(keying_items) >= MIN_ITEMS_PFA:
            k_emb_texts, _ = get_embedding_texts(
                keying_items, scale_name, PRIMARY_STRATEGY
            )
            k_sim          = encode_atomic_aggregate(k_emb_texts)
            k_ld, _        = run_fa(k_sim, 1)
            if k_ld is not None:
                keying_loadings[keying_dir] = dict(
                    zip(keying_items["item_id"].tolist(), k_ld.flatten())
                )

    # Eigenvalues (primary strategy)
    if a_eig is not None:
        total_eig = float(np.sum(a_eig))
        for ei, ev in enumerate(a_eig[:10]):
            eigenvalue_rows.append({
                "scale_name":          scale_name,
                "encoding":            f"atomic_aggregate ({PRIMARY_STRATEGY})",
                "eigenvalue_number":   ei + 1,
                "eigenvalue":          round(float(ev), 4),
                "above_1":             "Yes" if ev > 1.0 else "No",
                "cumulative_variance": round(float(np.sum(a_eig[:ei+1])) / total_eig, 4)
                                       if total_eig > 0 else 0
            })

    # Fit stats (primary strategy)
    fit_stat_rows.append({
        "scale_name":              scale_name,
        "construct_definition":    scale_def,
        "encoding":                f"atomic_aggregate ({PRIMARY_STRATEGY})",
        "n_items":                 len(ids_all),
        "n_transformers":          len(TRANSFORMER_MODELS),
        "transformer_models":      ", ".join(TRANSFORMER_MODELS),
        "rmsr":                    a_fit["rmsr"],
        "residual_mean":           a_fit["residual_mean"],
        "residual_std":            a_fit["residual_std"],
        "residual_max_abs":        a_fit["residual_max_abs"],
        "n_large_residuals":       a_fit["n_large_residuals"],
        "fit_acceptable":          a_fit["fit_acceptable"],
        "daal_identity_confirmed": joint_atomic.get("daal_identity", {}).get(scale_name, None),
        "n_items_reversed":        sum(rev_flags),
    })

    # Per-item evaluation (primary strategy)
    scale_high = scale_std = scale_fail = 0

    for i, row in items.iterrows():
        iid         = row["item_id"]
        item_keying = str(row.get("item_keying", "positive")).lower()

        a_single_load = round(float(a_single[i]), 4) if i < len(a_single) else 0.0
        communality   = round(float(a_single_load**2), 4)
        within_load   = round(
            float(keying_loadings.get(item_keying, {}).get(iid, 0.0)), 4
        )

        if a_ld2 is not None and i < len(a_ld2):
            vec       = a_ld2[i]
            a_primary = round(float(np.max(np.abs(vec))), 4)
            a_max_sec = round(float(np.min(np.abs(vec))), 4)
        else:
            a_primary = a_single_load
            a_max_sec = 0.0

        macro_load     = round(float(macro_item_sims[i]), 4)
        disc           = compute_discriminant_metrics(iid, scale_name, joint_atomic)
        daal_confirmed = joint_atomic.get("daal_identity", {}).get(scale_name, None)

        rules_passed, pass_results = evaluate_item_rules(
            a_single_load, communality, a_max_sec, within_load
        )

        n_passed        = len(rules_passed)
        ready_for_trial = n_passed >= MIN_RULES
        high_pass       = (
            "Rule_1_Strong_Single_Factor"  in rules_passed and
            "Rule_3_Adequate_Communality"  in rules_passed and
            "Rule_4_Clean_Factor_Structure" in rules_passed
        )
        pass_level = "HIGH_PASS" if high_pass else ("STANDARD_PASS" if ready_for_trial else "FAIL")

        if pass_level == "HIGH_PASS":       scale_high += 1
        elif pass_level == "STANDARD_PASS": scale_std  += 1
        else:                               scale_fail += 1

        # was-reversed flag: item was paraphrased only if primary strategy is
        # atomic_reversed AND item is negatively keyed. When the primary
        # strategy is plain atomic this column is always False for every item.
        was_reversed_primary = (
            PRIMARY_STRATEGY == "atomic_reversed" and item_keying == "negative"
        )

        pfa_row = {
            "item_id":                         iid,
            "scale_name":                      scale_name,
            "item_text":                       row["item_text"],
            "item_keying":                     item_keying,
            "encoding_strategy_primary":       PRIMARY_STRATEGY,
            "item_reversed_for_embedding":     was_reversed_primary,
            "occurrence_likelihood":           row.get("occurrence_likelihood",       ""),
            "source_example":                  row.get("source_example",              ""),
            "flesch_kincaid_grade":            row.get("flesch_kincaid_grade",        0),
            "was_simplified":                  row.get("was_simplified",              False),
            "simplification_attempts":         row.get("simplification_attempts",     0),
            "bias_flagged_dims":               row.get("bias_flagged_dims",           ""),
            "bias_flag_count":                 row.get("bias_flag_count",             0),
            "bias_any_flagged":                row.get("bias_any_flagged",            False),
            "cv_c_value":                      row.get("cv_c_value",                  0),
            "cv_d_value":                      row.get("cv_d_value",                  0),
            "cv_mean_target_rating":           row.get("cv_mean_target_rating",       0),
            "cv_std_target_rating":            row.get("cv_std_target_rating",        0),
            "cv_scale_mapping_agreement":      row.get("cv_scale_mapping_agreement",  0),
            "cv_maps_to_target_count":         row.get("cv_maps_to_target_count",     0),
            "cv_maps_to_target":               row.get("cv_maps_to_target",           False),
            "cv_majority_mapped_scale":        row.get("cv_majority_mapped_scale",    ""),
            "cv_majority_agreement":           row.get("cv_majority_agreement",       0),
            "cv_majority_is_target":           row.get("cv_majority_is_target",       False),
            "cv_pass":                         row.get("cv_pass",                     False),
            "pfa_atomic_single_loading":       a_single_load,
            "pfa_atomic_reversed_single_loading": (
                secondary_item_loadings.get(iid) if SECONDARY_STRATEGY == "atomic_reversed"
                else (a_single_load if PRIMARY_STRATEGY == "atomic_reversed" else None)
            ),
            "pfa_atomic_communality":          communality,
            "pfa_atomic_primary_loading":      a_primary,
            "pfa_atomic_max_sec_loading":      a_max_sec,
            "pfa_atomic_within_keying_loading": within_load,
            "pfa_macro_item_to_scale_sim":     macro_load,
            "daal_identity_confirmed":         daal_confirmed,
            **disc,
            "pfa_scale_rmsr":                  a_fit["rmsr"],
            "pfa_fit_acceptable":              a_fit["fit_acceptable"],
            "pfa_rules_passed_count":          n_passed,
            "pfa_rules_failed_count":          6 - n_passed,
            "pfa_ready_for_trial":             ready_for_trial,
            "pfa_pass_level":                  pass_level,
        }
        for rule, result in pass_results.items():
            pfa_row[rule] = result

        pfa_rows.append(pfa_row)

    print(
        f"  {scale_name:<35}  n={len(items):>3}  "
        f"rev={sum(rev_flags):>2}  "
        f"HIGH={scale_high}  STD={scale_std}  FAIL={scale_fail}  "
        f"RMSR={a_fit['rmsr']:.3f}  "
        f"DAAL={'✓' if joint_atomic.get('daal_identity',{}).get(scale_name) else ('✗' if joint_atomic.get('available') else '?')}"
    )

df_pfa = pd.DataFrame(pfa_rows)


# ==============================================================================
# DISCRIMINANT MATRIX — all items, all factors, factor-index order
# Includes Unassigned factors. No filtering.
# ==============================================================================
disc_matrix_rows: List[Dict] = []
if joint_atomic.get("available") and joint_atomic.get("item_level"):
    factor_labels       = joint_atomic["factor_labels"]
    ordered_factor_cols = [
        factor_labels.get(f, f"F{f+1}")
        for f in range(len(factor_labels))
    ]
    for iid, cross in joint_atomic["item_cross_loadings"].items():
        row_data = df_pfa[df_pfa["item_id"] == iid]
        if row_data.empty:
            continue
        base_row = {
            "item_id":    iid,
            "scale_name": row_data.iloc[0]["scale_name"],
            "item_text":  row_data.iloc[0]["item_text"]
        }
        for col in ordered_factor_cols:
            base_row[f"loading_on_{col}"] = cross.get(col, None)
        disc_matrix_rows.append(base_row)
df_disc_matrix = pd.DataFrame(disc_matrix_rows)


# ==============================================================================
# ITEM-SCALE HEATMAP
# ==============================================================================
print("\nBuilding Item-Scale Heatmap...")

heatmap_scale_order = [s["scale_name"] for s in TARGET_SCALES]

scale_primary_vecs = {}
for sn in heatmap_scale_order:
    scale_items = df_p5[df_p5["scale_name"] == sn]["item_text"].tolist()
    if not scale_items:
        scale_items = df_cv_all[df_cv_all["scale_name"] == sn]["item_text"].tolist()
    if scale_items:
        preprocessed = [preprocess_for_embedding(t) for t in scale_items]
        concatenated  = " ".join(preprocessed)
        vec = PRIMARY_MODEL.encode([concatenated], show_progress_bar=False)[0]
        scale_primary_vecs[sn] = vec / (np.linalg.norm(vec) + 1e-9)
    else:
        scale_primary_vecs[sn] = None

all_item_texts = df_pfa["item_text"].tolist()
all_item_embs  = encode_atomic_raw(all_item_texts)
all_item_norms = all_item_embs / (
    np.linalg.norm(all_item_embs, axis=1, keepdims=True) + 1e-9
)

heatmap_rows = []
for i, (_, row) in enumerate(df_pfa.iterrows()):
    hmap_row = {
        "item_id":        row["item_id"],
        "scale_name":     row["scale_name"],
        "item_text":      row["item_text"],
        "item_keying":    row.get("item_keying", ""),
        "pfa_pass_level": row["pfa_pass_level"],
    }
    item_norm = all_item_norms[i]
    for sn in heatmap_scale_order:
        if scale_primary_vecs.get(sn) is not None:
            hmap_row[f"sim_{sn}"] = round(float(item_norm @ scale_primary_vecs[sn]), 4)
        else:
            hmap_row[f"sim_{sn}"] = None
    heatmap_rows.append(hmap_row)

df_heatmap = pd.DataFrame(heatmap_rows)
scale_order_map     = {sn: i for i, sn in enumerate(heatmap_scale_order)}
df_heatmap["_sort"] = df_heatmap["scale_name"].map(scale_order_map).fillna(999)
df_heatmap          = df_heatmap.sort_values("_sort").drop(columns=["_sort"]).reset_index(drop=True)

print(f"  ✓ Item-Scale Heatmap: {len(df_heatmap)} items × {len(heatmap_scale_order)} scales")


# ==============================================================================
# SCALE SUMMARY
# ==============================================================================
agg_dict = {
    "total_items":              ("item_id",                     "count"),
    "high_pass":                ("pfa_pass_level",               lambda x: (x == "HIGH_PASS").sum()),
    "standard_pass":            ("pfa_pass_level",               lambda x: (x == "STANDARD_PASS").sum()),
    "fail":                     ("pfa_pass_level",               lambda x: (x == "FAIL").sum()),
    "n_reversed_for_embedding": ("item_reversed_for_embedding",  "sum"),
    "mean_atomic_loading":      ("pfa_atomic_single_loading",    lambda x: x.abs().mean()),
    "mean_communality":         ("pfa_atomic_communality",       "mean"),
    "mean_macro_sim":           ("pfa_macro_item_to_scale_sim",  "mean"),
    "mean_c_value":             ("cv_c_value",                   "mean"),
    "mean_d_value":             ("cv_d_value",                   "mean"),
    "mean_cv_rating":           ("cv_mean_target_rating",        "mean"),
    "mean_cv_agreement":        ("cv_scale_mapping_agreement",   "mean"),
    "mean_scale_rmsr":          ("pfa_scale_rmsr",               "mean"),
}
if "discriminant_pass" in df_pfa.columns and df_pfa["discriminant_pass"].notna().any():
    agg_dict["disc_pass_count"] = ("discriminant_pass",        lambda x: x.sum() if x.notna().any() else 0)
    agg_dict["disc_flag_count"] = ("discriminant_flag_review", lambda x: x.sum() if x.notna().any() else 0)

pfa_scale_summary = df_pfa.groupby("scale_name").agg(**agg_dict).reset_index()

for col in ["mean_atomic_loading", "mean_communality", "mean_macro_sim",
            "mean_c_value", "mean_d_value", "mean_cv_rating",
            "mean_cv_agreement", "mean_scale_rmsr"]:
    if col in pfa_scale_summary.columns:
        pfa_scale_summary[col] = pfa_scale_summary[col].round(3)

pfa_scale_summary["pass_rate_pct"] = round(
    100 * (pfa_scale_summary["high_pass"] + pfa_scale_summary["standard_pass"])
    / pfa_scale_summary["total_items"], 1
)

if joint_atomic.get("available"):
    daal_df = pd.DataFrame([
        {"scale_name": k, "daal_identity_confirmed": v}
        for k, v in joint_atomic["daal_identity"].items()
    ])
    pfa_scale_summary = pfa_scale_summary.merge(daal_df, on="scale_name", how="left")

# ==============================================================================
# PASS RULES DOCUMENTATION
# ==============================================================================
rules_def_rows = [
    {"rule_id": k, "description": v["description"],
     "threshold": v["threshold"], "operator": v["operator"]}
    for k, v in PASS_RULES_CFG.items()
]
rules_def_rows += [
    {"rule_id": "min_rules_to_pass",
     "description": "Minimum rules passed to be ready for trial",
     "threshold": MIN_RULES, "operator": ">="},
    {"rule_id": "discriminant_ratio_min",
     "description": "Min target/max-cross loading ratio for discriminant pass",
     "threshold": DISC_RATIO_MIN, "operator": ">="},
    {"rule_id": "discriminant_ceiling",
     "description": "Max absolute cross-loading before ceiling flag",
     "threshold": DISC_CEILING, "operator": "<"},
    {"rule_id": "rmsr_max",
     "description": "Max RMSR for fit_acceptable (Guenole et al. 2025)",
     "threshold": RMSR_THRESHOLD, "operator": "<="},
    {"rule_id": "NOTE_reversal",
     "description": "Under atomic_reversed encoding, negatively keyed items are paraphrased to the positive pole for embedding only. item_text is always original.",
     "threshold": "N/A", "operator": "N/A"},
    {"rule_id": "NOTE_keying",
     "description": "item_keying records the keying direction of each item: positive (dominance, higher trait = stronger endorsement) or negative (genuine semantic reversal, lower trait = stronger endorsement).",
     "threshold": "N/A", "operator": "N/A"},
    {"rule_id": "NOTE_encoding_strategy",
     "description": f"Encoding strategy: {ENCODING_STRATEGY} (runs: {', '.join(STRATEGIES_TO_RUN)}). Primary strategy drives pass-rule evaluation; secondary (if any) appears as parallel columns.",
     "threshold": "N/A", "operator": "N/A"},
    {"rule_id": "NOTE_transformers",
     "description": f"Similarity matrices aggregated across: {', '.join(TRANSFORMER_MODELS)}",
     "threshold": "N/A", "operator": "N/A"},
    {"rule_id": "NOTE_heatmap",
     "description": "Item_Scale_Heatmap uses primary model only for consistent dimensionality",
     "threshold": "N/A", "operator": "N/A"},
    {"rule_id": "NOTE_projected",
     "description": "Loading_Matrix_All: in_pfa_solution=False items have projected loadings",
     "threshold": "N/A", "operator": "N/A"},
]

# ==============================================================================
# SUBSET SHEETS
# ==============================================================================
keep_cols = [
    "item_id", "scale_name", "item_text", "item_keying",
    "encoding_strategy_primary", "item_reversed_for_embedding",
    "flesch_kincaid_grade", "cv_c_value", "cv_d_value",
    "cv_mean_target_rating", "pfa_atomic_single_loading",
    "pfa_atomic_reversed_single_loading", "pfa_macro_item_to_scale_sim",
    "pfa_atomic_max_sec_loading", "daal_identity_confirmed",
    "discriminant_pass", "discriminant_flag_review"
]
keep_cols    = [c for c in keep_cols if c in df_pfa.columns]
df_high_pass = df_pfa[df_pfa["pfa_pass_level"] == "HIGH_PASS"][keep_cols].copy()
df_all_pass  = df_pfa[df_pfa["pfa_ready_for_trial"] == True][keep_cols].copy()


# ==============================================================================
# PRINT SUMMARY
# ==============================================================================
n_high = (df_pfa["pfa_pass_level"] == "HIGH_PASS").sum()
n_std  = (df_pfa["pfa_pass_level"] == "STANDARD_PASS").sum()
n_fail = (df_pfa["pfa_pass_level"] == "FAIL").sum()

print(f"\n{'='*60}")
print("PFA Scale Summary")
print(f"{'='*60}")
summary_cols = ["scale_name", "total_items"]
for c in ["n_reversed_for_embedding", "high_pass", "standard_pass", "fail",
          "pass_rate_pct", "mean_c_value", "mean_d_value",
          "mean_atomic_loading", "mean_scale_rmsr"]:
    if c in pfa_scale_summary.columns:
        summary_cols.append(c)
if "daal_identity_confirmed" in pfa_scale_summary.columns:
    summary_cols.append("daal_identity_confirmed")
print(pfa_scale_summary[summary_cols].to_string(index=False))


# ==============================================================================
# WRITE PFA SHEETS
# ==============================================================================
df_pfa["human_review_pass"] = True
df_pfa["human_comments"]    = ""

with pd.ExcelWriter(P5_OUT, engine="openpyxl") as writer:
    df_pfa.to_excel(                           writer, sheet_name="PFA_Item_Results",       index=False)
    pfa_scale_summary.to_excel(                writer, sheet_name="PFA_Scale_Summary",      index=False)
    if not df_disc_matrix.empty:
        df_disc_matrix.to_excel(               writer, sheet_name="Discriminant_Matrix",    index=False)
    if not df_heatmap.empty:
        df_heatmap.to_excel(                   writer, sheet_name="Item_Scale_Heatmap",     index=False)
    if not df_loading_matrix_all.empty:
        df_loading_matrix_all.to_excel(        writer, sheet_name="Loading_Matrix_All",     index=False)
        df_loading_matrix_cv_pass.to_excel(    writer, sheet_name="Loading_Matrix_CV_Pass", index=False)
    pd.DataFrame(fit_stat_rows).to_excel(      writer, sheet_name="Fit_Statistics",         index=False)
    pd.DataFrame(eigenvalue_rows).to_excel(    writer, sheet_name="Eigenvalues",            index=False)
    df_high_pass.to_excel(                     writer, sheet_name="High_Pass_Loadings",     index=False)
    df_all_pass.to_excel(                      writer, sheet_name="All_Pass_Loadings",      index=False)
    pd.DataFrame(rules_def_rows).to_excel(     writer, sheet_name="Pass_Rules_Definition",  index=False)

print(f"\n✅ Phase 5 PFA complete.")
print(f"   HIGH_PASS    : {n_high}")
print(f"   STANDARD_PASS: {n_std}")
print(f"   FAIL         : {n_fail}")
print(f"   Total in PFA : {len(df_pfa)} "
      f"(from {len(df_cv_all)} Phase 4 items, {n_excluded} excluded)")
n_reversed = int(df_pfa["item_reversed_for_embedding"].sum()) \
             if "item_reversed_for_embedding" in df_pfa.columns else 0
print(f"   Encoding strategy            : {ENCODING_STRATEGY} (primary: {PRIMARY_STRATEGY})")
print(f"   Items reversed for embedding : {n_reversed} "
      f"(atomic_reversed applied to negatively keyed items only)")
print(f"   Transformers aggregated      : {len(TRANSFORMER_MODELS)} ({', '.join(TRANSFORMER_MODELS)})")
if joint_atomic.get("available"):
    n_daal = sum(joint_atomic["daal_identity"].values())
    n_tot  = len(joint_atomic["daal_identity"])
    print(f"   DAAL identity: {n_daal}/{n_tot} scales confirmed")
if "discriminant_flag_review" in df_pfa.columns:
    print(f"   Discriminant flags: {int(df_pfa['discriminant_flag_review'].sum())} items flagged")
if not df_heatmap.empty:
    print(f"   Item_Scale_Heatmap: {len(df_heatmap)} items × {len(heatmap_scale_order)} scales")
if not df_loading_matrix_all.empty:
    n_proj = (~df_loading_matrix_all["in_pfa_solution"]).sum()
    print(f"   Loading_Matrix_All: {len(df_loading_matrix_all)} items ({n_proj} projected)")
print(f"📄 Output: {P5_OUT}")


# ==============================================================================
# ITEM POOL ASSEMBLY — appended to same file
# ==============================================================================
print(f"\n{'='*60}")
print(f"PHASE 5 (continued): Item Pool Assembly")
print(f"{'='*60}")

with open(SETTINGS_FILE, "r", encoding="utf-8") as f:
    CFG = json.load(f)

# Read back the PFA results we just wrote
with pd.ExcelFile(P5_OUT) as xls:
    df_pool = pd.read_excel(xls, sheet_name="PFA_Item_Results").copy()

print(f"✓ PFA items loaded: {len(df_pool)}")

# ── Pass logic ────────────────────────────────────────────────────────────────
if "human_review_pass" in df_pool.columns:
    df_pool["overall_pass"] = (df_pool["human_review_pass"] != False)
    print("✓ PFA human_review_pass found — using as primary gate")
else:
    df_pool["pfa_passed"]   = df_pool["pfa_ready_for_trial"].fillna(False).astype(bool)
    df_pool["cv_passed"]    = df_pool["cv_pass"].fillna(False).astype(bool)
    df_pool["overall_pass"] = df_pool["pfa_passed"] & df_pool["cv_passed"]
    print("⚠ human_review_pass not found — falling back to pfa_passed & cv_passed")

df_pass   = df_pool[df_pool["overall_pass"]].copy().reset_index(drop=True)
df_reject = df_pool[~df_pool["overall_pass"]].copy().reset_index(drop=True)
print(f"  Overall pass : {len(df_pass)}")
print(f"  Rejected     : {len(df_reject)}")

# ── Human review columns ──────────────────────────────────────────────────────
df_pass["human_decision"] = ""
df_pass["human_notes"]    = ""

# ── Review columns from settings ─────────────────────────────────────────────
review_cols = P5.get("include_columns", [])
if "scale_name" not in review_cols:
    review_cols = ["scale_name"] + review_cols

available_cols = [c for c in review_cols if c in df_pass.columns]
missing_cols   = [c for c in review_cols if c not in df_pass.columns]
if missing_cols:
    logger.warning(
        f"Pool assembly: {len(missing_cols)} configured columns not found: {missing_cols}"
    )
if "scale_name" not in available_cols:
    available_cols = ["scale_name"] + available_cols

df_review = df_pass[available_cols].copy()
print(f"  Review columns: {len(available_cols)}")

# ── Pool statistics ───────────────────────────────────────────────────────────
pool_stats_rows: List[Dict] = []
for scale_name, grp in df_review.groupby("scale_name"):
    keying_dist = (
        grp["item_keying"].astype(str).str.lower().value_counts().to_dict()
        if "item_keying" in grp.columns else {}
    )
    daal_col   = "daal_identity_confirmed"
    disc_col   = "discriminant_flag_review"
    rev_col    = "item_reversed_for_embedding"
    daal_confirmed = int(grp[daal_col].sum()) if daal_col in grp.columns and grp[daal_col].notna().any() else None
    disc_flags     = int(grp[disc_col].sum()) if disc_col in grp.columns and grp[disc_col].notna().any() else 0
    n_reversed     = int(grp[rev_col].sum()) if rev_col in grp.columns and grp[rev_col].notna().any() else 0

    pool_stats_rows.append({
        "scale_name":                   scale_name,
        "total_items":                  len(grp),
        "positive_keyed_items":         keying_dist.get("positive", 0),
        "negative_keyed_items":         keying_dist.get("negative", 0),
        "high_pass_items":              (grp["pfa_pass_level"] == "HIGH_PASS").sum(),
        "standard_pass_items":          (grp["pfa_pass_level"] == "STANDARD_PASS").sum(),
        "mean_c_value":                 round(grp["cv_c_value"].mean(), 3)
                                        if "cv_c_value" in grp else 0,
        "mean_d_value":                 round(grp["cv_d_value"].mean(), 3)
                                        if "cv_d_value" in grp else 0,
        "mean_cv_rating":               round(grp["cv_mean_target_rating"].mean(), 2)
                                        if "cv_mean_target_rating" in grp else 0,
        "mean_cv_agreement":            round(grp["cv_scale_mapping_agreement"].mean(), 2)
                                        if "cv_scale_mapping_agreement" in grp else 0,
        "mean_atomic_loading":          round(grp["pfa_atomic_single_loading"].abs().mean(), 3)
                                        if "pfa_atomic_single_loading" in grp else 0,
        "mean_macro_sim":               round(grp["pfa_macro_item_to_scale_sim"].mean(), 3)
                                        if "pfa_macro_item_to_scale_sim" in grp else 0,
        "mean_fk_grade":                round(grp["flesch_kincaid_grade"].mean(), 2)
                                        if "flesch_kincaid_grade" in grp else 0,
        "items_with_bias_flags":        grp["bias_flagged_dims"].fillna("").astype(str).str.strip().str.len().gt(0).sum()
                                        if "bias_flagged_dims" in grp.columns else 0,
        "items_reversed_for_embedding": n_reversed,
        "daal_identity_confirmed":      daal_confirmed,
        "discriminant_flags":           disc_flags,
    })

# ── Quality flags ─────────────────────────────────────────────────────────────
reading_target = P3_CFG.get("reading_level_target", 8)
min_c          = P4_CFG["pass_thresholds"].get("min_c_value", 0.88)
min_d          = P4_CFG["pass_thresholds"].get("min_d_value", 0.35)

quality_flag_rows: List[Dict] = []
for _, row in df_review.iterrows():
    flags = []

    if row.get("flesch_kincaid_grade", 0) > reading_target:
        flags.append(f"Above FK grade {reading_target} ({row['flesch_kincaid_grade']:.1f})")

    if str(row.get("bias_flagged_dims", "")).strip():
        flags.append(f"Bias: {row['bias_flagged_dims']}")

    if row.get("cv_c_value", 1.0) < min_c:
        flags.append(f"Low c-value ({row.get('cv_c_value', 0):.3f}, threshold {min_c})")

    if row.get("cv_d_value", 1.0) < min_d:
        flags.append(f"Low d-value ({row.get('cv_d_value', 0):.3f}, threshold {min_d})")

    if not row.get("cv_maps_to_target", True):
        flags.append(f"CV mapped to '{row.get('cv_majority_mapped_scale', '?')}' not target")

    if row.get("pfa_pass_level") == "STANDARD_PASS":
        flags.append("Standard pass only")

    if row.get("discriminant_flag_review") == True:
        closest   = row.get("discriminant_closest_scale", "?")
        ratio     = row.get("discriminant_target_to_max_ratio", None)
        ratio_str = f", ratio={ratio:.2f}" if ratio is not None else ""
        flags.append(f"Discriminant flag: closest to '{closest}'{ratio_str}")

    if row.get("daal_identity_confirmed") == False:
        flags.append("DAAL identity not confirmed")

    if row.get("item_reversed_for_embedding") == True:
        flags.append("Item reversed for embedding — verify semantic direction")

    if flags:
        quality_flag_rows.append({
            "item_id":        row.get("item_id"),
            "scale_name":     row.get("scale_name"),
            "item_text":      row.get("item_text"),
            "item_keying":    row.get("item_keying", ""),
            "pfa_pass_level": row.get("pfa_pass_level"),
            "flags":          " | ".join(flags),
            "flag_count":     len(flags),
        })

# ── Recommendations ───────────────────────────────────────────────────────────
target_per_keying = P5.get(
    "final_pool_target_per_keying",
    P5.get("final_pool_target_per_location", 20)  # legacy fallback
)
recs: List[Dict] = []

for row in pool_stats_rows:
    sn = row["scale_name"]

    total_target = target_per_keying * 2  # positive + negative combined
    if row["total_items"] < total_target:
        recs.append({"scale_name": sn, "priority": "HIGH",
                     "recommendation": f"Only {row['total_items']} items passed "
                                       f"(target: {total_target} = {target_per_keying} positive + {target_per_keying} negative). "
                                       f"Regenerate more items."})

    for keying_dir, key in [("positive", "positive_keyed_items"),
                            ("negative", "negative_keyed_items")]:
        if row[key] < target_per_keying:
            recs.append({"scale_name": sn, "priority": "MEDIUM",
                         "recommendation": f"Only {row[key]} {keying_dir}-keyed items "
                                           f"(target: {target_per_keying})."})

    # Flag extreme keying imbalance separately from raw counts
    if row["positive_keyed_items"] > 0 and row["negative_keyed_items"] == 0:
        recs.append({"scale_name": sn, "priority": "HIGH",
                     "recommendation": "No negatively keyed items passed. "
                                       "Without negative items, acquiescence bias cannot be detected."})

    if row["mean_fk_grade"] > reading_target:
        recs.append({"scale_name": sn, "priority": "MEDIUM",
                     "recommendation": f"Mean FK grade {row['mean_fk_grade']} above target {reading_target}."})

    if row.get("mean_c_value", 1.0) < min_c:
        recs.append({"scale_name": sn, "priority": "MEDIUM",
                     "recommendation": f"Mean c-value {row.get('mean_c_value', 0):.3f} below {min_c} threshold."})

    if row.get("mean_d_value", 1.0) < min_d:
        recs.append({"scale_name": sn, "priority": "MEDIUM",
                     "recommendation": f"Mean d-value {row.get('mean_d_value', 0):.3f} below {min_d} threshold."})

    if row["items_with_bias_flags"] > 0:
        recs.append({"scale_name": sn, "priority": "HIGH",
                     "recommendation": f"{row['items_with_bias_flags']} items have bias flags. Prioritize review."})

    if row.get("items_reversed_for_embedding", 0) > 0:
        recs.append({"scale_name": sn, "priority": "MEDIUM",
                     "recommendation": f"{row['items_reversed_for_embedding']} items reversed for embedding "
                                       f"(atomic_reversed paraphrase of negatively keyed items) "
                                       f"— verify semantic direction before trialling."})

    if row.get("discriminant_flags", 0) > 0:
        recs.append({"scale_name": sn, "priority": "MEDIUM",
                     "recommendation": f"{row['discriminant_flags']} items flagged for discriminant validity. "
                                       f"Check 'discriminant_closest_scale'."})

    if row.get("daal_identity_confirmed") == False:
        recs.append({"scale_name": sn, "priority": "HIGH",
                     "recommendation": "DAAL factor identity not confirmed. "
                                       "Review construct definition and Item_Scale_Heatmap."})

priority_order = {"HIGH": 0, "MEDIUM": 1, "LOW": 2}
recs.sort(key=lambda x: priority_order.get(x.get("priority", "LOW"), 2))

# ── Rejected items ────────────────────────────────────────────────────────────
reject_cols   = [c for c in available_cols if c not in ("human_decision", "human_notes")]
df_reject_out = df_reject[[c for c in reject_cols if c in df_reject.columns]].copy()

# ── Append pool assembly sheets to PFA output file ────────────────────────────
with pd.ExcelWriter(P5_OUT, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    df_review.to_excel(                        writer, sheet_name="Review_Pool",     index=False)
    df_reject_out.to_excel(                    writer, sheet_name="Rejected_Items",  index=False)
    pd.DataFrame(pool_stats_rows).to_excel(    writer, sheet_name="Pool_Statistics", index=False)
    if quality_flag_rows:
        pd.DataFrame(quality_flag_rows).to_excel(writer, sheet_name="Quality_Flags",  index=False)
    if recs:
        pd.DataFrame(recs).to_excel(           writer, sheet_name="Recommendations", index=False)

# ── Final summary ─────────────────────────────────────────────────────────────
n_high_flags = sum(1 for r in recs if r.get("priority") == "HIGH")
n_med_flags  = sum(1 for r in recs if r.get("priority") == "MEDIUM")
n_rev        = int(df_review["item_reversed_for_embedding"].sum()) \
               if "item_reversed_for_embedding" in df_review.columns else 0

print(f"\n{'='*60}")
print(f"✅ PIPELINE COMPLETE — Phase 5: PFA + Item Pool Assembly")
print(f"{'='*60}")
print(f"Items in review pool        : {len(df_review)}")
print(f"Items rejected              : {len(df_reject)}")
print(f"Items with QC flags         : {len(quality_flag_rows)}")
print(f"Items reversed for embedding: {n_rev}")
print(f"Recommendations             : {len(recs)} "
      f"({n_high_flags} HIGH / {n_med_flags} MEDIUM priority)")
print(f"\n📄 Output: {P5_OUT}")
print(f"\nReview workflow:")
print(f"  1. Open '{P5['output_file']}'")
print(f"  2. Start with 'Recommendations' and 'Quality_Flags' sheets")
print(f"  3. Work through 'Review_Pool' scale by scale")
print(f"  4. Fill in 'human_decision' (accept / modify / reject)")
print(f"  5. Add rationale in 'human_notes' — required for any reversal or flag")

In [ ]:
# ==============================================================================
# PHASE 5 (formatting pass) — Excel cell formatting for reviewer ergonomics
# ==============================================================================
# Re-opens the finished 05_pseudo_factor_analysis.xlsx and applies conditional
# formatting, frozen panes, filters, column widths, and number formats so that
# a reviewer scanning these sheets can see threshold pass/fail status, peak
# loadings, and input cells at a glance — without reading individual numbers.
#
# Design principles:
#   • Formatting is tied to the pipeline's own threshold constants — no new
#     numbers introduced here.
#   • 3-color scales on loading/similarity matrices use light tints so cell
#     contents remain readable (pale red / white / pale green).
#   • Pass/Fail cells use a restrained green / amber / red palette.
#   • Input cells (human_*) use a light yellow fill to mark them editable.
#   • All sheets get frozen panes and autofilter; sheet-specific widths only
#     where needed.
#
# The helpers below are idempotent — re-running this cell will not corrupt
# existing formatting. They tolerate missing columns (so configuration changes
# to include_columns don't break the styling step).
# ==============================================================================

from openpyxl                    import load_workbook
from openpyxl.formatting.rule    import CellIsRule, ColorScaleRule, FormulaRule
from openpyxl.styles             import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils              import get_column_letter

print(f"\n{'='*60}")
print(f"PHASE 5 (formatting pass): styling {P5_OUT.name}")
print(f"{'='*60}")

# ── Palette (muted; suitable for academic/professional output) ────────────────
FILL_PASS_STRONG  = PatternFill("solid", fgColor="C6EFCE")   # pale green
FILL_PASS_LIGHT   = PatternFill("solid", fgColor="E2EFDA")   # very pale green
FILL_WARN         = PatternFill("solid", fgColor="FFEB9C")   # pale amber
FILL_FAIL         = PatternFill("solid", fgColor="F8CBAD")   # pale red
FILL_FAIL_STRONG  = PatternFill("solid", fgColor="F4B084")   # red
FILL_INPUT        = PatternFill("solid", fgColor="FFF2CC")   # light yellow — reviewer inputs
FILL_HEADER       = PatternFill("solid", fgColor="305496")   # dark blue-grey
FILL_INFO         = PatternFill("solid", fgColor="DDEBF7")   # pale blue — diagnostic highlight

FONT_HEADER       = Font(bold=True, color="FFFFFF", size=11)
FONT_BOLD         = Font(bold=True)
ALIGN_CENTER      = Alignment(horizontal="center", vertical="center", wrap_text=False)
ALIGN_WRAP        = Alignment(horizontal="left",   vertical="top",    wrap_text=True)
ALIGN_LEFT_TOP    = Alignment(horizontal="left",   vertical="top",    wrap_text=False)

# Pull thresholds from settings so formatting stays in lockstep with pass rules
_RULES     = P5["pass_rules"]
T_R1       = _RULES.get("Rule_1_Strong_Single_Factor",   {}).get("threshold", 0.40)
T_R2       = _RULES.get("Rule_2_Adequate_Single_Factor", {}).get("threshold", 0.30)
T_R3       = _RULES.get("Rule_3_Adequate_Communality",   {}).get("threshold", 0.10)
T_R4       = _RULES.get("Rule_4_Clean_Factor_Structure", {}).get("threshold", 0.35)
T_R5       = _RULES.get("Rule_5_Cross_Sample_Viable",    {}).get("threshold", 0.20)
T_R6       = _RULES.get("Rule_6_Keying_Viable",          {}).get("threshold", 0.25)
T_CVC      = P4_CFG["pass_thresholds"].get("min_c_value", 0.88)
T_CVD      = P4_CFG["pass_thresholds"].get("min_d_value", 0.35)
T_RMSR     = P5["model_fit_thresholds"].get("rmsr_max",   0.08)
T_DISC_R   = P5.get("discriminant_validity", {}).get("target_to_max_ratio_min", 1.5)
T_DISC_C   = P5.get("discriminant_validity", {}).get("max_cross_loading_ceiling", 0.45)
T_FK_TGT   = P3_CFG.get("reading_level_target", 8)
T_FK_CAP   = P3_CFG.get("reading_level_hard_cap", 12)
T_BIAS_MIN = P3_CFG["bias_review"].get("flag_threshold", 3.5)


def _col_idx(ws, name: str) -> Optional[int]:
    """Return 1-based column index for a header name, or None if missing."""
    for cell in ws[1]:
        if str(cell.value) == name:
            return cell.column
    return None


def _col_letter(ws, name: str) -> Optional[str]:
    idx = _col_idx(ws, name)
    return get_column_letter(idx) if idx else None


def _data_range(ws, col_letter: str) -> Optional[str]:
    if ws.max_row < 2 or col_letter is None:
        return None
    return f"{col_letter}2:{col_letter}{ws.max_row}"


def _apply_header_and_filter(ws) -> None:
    """Standard treatment: styled header row, frozen top row, autofilter."""
    if ws.max_row < 1:
        return
    for cell in ws[1]:
        cell.fill      = FILL_HEADER
        cell.font      = FONT_HEADER
        cell.alignment = ALIGN_CENTER
    ws.row_dimensions[1].height = 30
    # Freeze first row (and first column if it looks like an ID column)
    first_header = str(ws[1][0].value or "").lower()
    if first_header in ("item_id", "scale_name", "rule_id"):
        ws.freeze_panes = "B2"
    else:
        ws.freeze_panes = "A2"
    # Autofilter across all used columns
    last_col = get_column_letter(ws.max_column)
    ws.auto_filter.ref = f"A1:{last_col}{max(ws.max_row, 1)}"


def _set_col_widths(ws, widths: Dict[str, float]) -> None:
    """Set explicit widths for named columns; skip if column absent."""
    for name, width in widths.items():
        idx = _col_idx(ws, name)
        if idx:
            ws.column_dimensions[get_column_letter(idx)].width = width


def _set_number_format(ws, col_name: str, fmt: str) -> None:
    letter = _col_letter(ws, col_name)
    if letter is None:
        return
    for row in range(2, ws.max_row + 1):
        ws[f"{letter}{row}"].number_format = fmt


def _cond_gte(ws, col_name: str, threshold: float, fill: PatternFill) -> None:
    rng = _data_range(ws, _col_letter(ws, col_name))
    if rng is None:
        return
    ws.conditional_formatting.add(
        rng,
        CellIsRule(operator="greaterThanOrEqual",
                   formula=[str(threshold)], fill=fill)
    )


def _cond_lt(ws, col_name: str, threshold: float, fill: PatternFill) -> None:
    rng = _data_range(ws, _col_letter(ws, col_name))
    if rng is None:
        return
    ws.conditional_formatting.add(
        rng,
        CellIsRule(operator="lessThan",
                   formula=[str(threshold)], fill=fill)
    )


def _cond_gt(ws, col_name: str, threshold: float, fill: PatternFill) -> None:
    rng = _data_range(ws, _col_letter(ws, col_name))
    if rng is None:
        return
    ws.conditional_formatting.add(
        rng,
        CellIsRule(operator="greaterThan",
                   formula=[str(threshold)], fill=fill)
    )


def _cond_equal_text(ws, col_name: str, value: str, fill: PatternFill) -> None:
    """Equality against a string literal, case-sensitive per Excel default."""
    rng = _data_range(ws, _col_letter(ws, col_name))
    if rng is None:
        return
    ws.conditional_formatting.add(
        rng,
        CellIsRule(operator="equal",
                   formula=[f'"{value}"'], fill=fill)
    )


def _cond_color_scale_abs(ws, col_name: str) -> None:
    """
    Red-white-green 3-color scale on absolute value. Used on factor loading
    columns where sign does not matter (FA sign flips are expected in PFA).
    Because openpyxl's color scales operate on raw values, we approximate by
    anchoring at -1 / 0 / 1 — the palette reads correctly for loadings in
    the [-1, 1] range: strong negative → pale red, zero → white, strong
    positive → pale green. Reviewers who care about sign can still read it;
    reviewers who only care about magnitude see a symmetric heatmap if they
    mentally flip the sign of negatively loading items.
    """
    rng = _data_range(ws, _col_letter(ws, col_name))
    if rng is None:
        return
    ws.conditional_formatting.add(
        rng,
        ColorScaleRule(
            start_type="num", start_value=-1.0, start_color="F8CBAD",
            mid_type="num",   mid_value=0.0,    mid_color="FFFFFF",
            end_type="num",   end_value=1.0,    end_color="C6EFCE",
        )
    )


def _cond_color_scale_sim(ws, col_name: str) -> None:
    """
    3-color scale for cosine similarity columns, anchored at pipeline pass-rule
    thresholds so the colour boundaries match the pipeline's own definitions of
    weak / adequate / strong:
        0.20 (Rule 5 floor)  = pale red   — below cross-sample viability
        0.30 (Rule 2)        = white      — borderline adequate
        0.40 (Rule 1)        = pale green — strong single-factor threshold

    Values above 0.40 render in progressively deeper green (openpyxl
    extrapolates past the 'end' anchor), so the genuine peaks in each row
    visibly dominate off-target cross-similarities.
    """
    rng = _data_range(ws, _col_letter(ws, col_name))
    if rng is None:
        return
    ws.conditional_formatting.add(
        rng,
        ColorScaleRule(
            start_type="num", start_value=T_R5, start_color="F8CBAD",
            mid_type="num",   mid_value=T_R2,   mid_color="FFFFFF",
            end_type="num",   end_value=T_R1,   end_color="63BE7B",
        )
    )


def _mark_input_columns(ws, col_names: List[str]) -> None:
    """Light-yellow fill on every data cell of the named reviewer-input columns."""
    for name in col_names:
        letter = _col_letter(ws, name)
        if letter is None:
            continue
        for row in range(2, ws.max_row + 1):
            cell = ws[f"{letter}{row}"]
            # Preserve any conditional colour by layering input fill only where
            # the cell is currently unfilled.
            if cell.fill.fgColor.rgb in (None, "00000000"):
                cell.fill = FILL_INPUT


# ==============================================================================
# SHEET-BY-SHEET STYLING
# ==============================================================================

def style_PFA_Item_Results(ws) -> None:
    """
    The big scanning sheet. Reviewer decides item by item. Formatting anchors:
      • pfa_pass_level           — green / amber / red
      • Rule_N columns           — PASS green, FAIL red
      • cv_c_value / cv_d_value  — fail cells highlighted red against thresholds
      • atomic / macro loadings  — 3-color heatmap (sign-preserving for atomic)
      • discriminant_flag_review — TRUE cells red
      • flesch_kincaid_grade     — above-target amber, above-cap red
      • item_keying              — subtle shading to distinguish positive/negative
      • human_review_pass / human_comments — marked as reviewer inputs
    """
    _apply_header_and_filter(ws)
    _set_col_widths(ws, {
        "item_id":                              14,
        "scale_name":                           22,
        "item_text":                            55,
        "item_keying":                          12,
        "encoding_strategy_primary":            16,
        "item_reversed_for_embedding":          14,
        "source_example":                       14,
        "occurrence_likelihood":                14,
        "pfa_pass_level":                       16,
        "human_review_pass":                    12,
        "human_comments":                       40,
        "bias_flagged_dims":                    22,
        "cv_majority_mapped_scale":             22,
        "discriminant_closest_scale":           22,
    })
    # Narrower widths for all numeric columns (handled implicitly by default)

    # Wrap item_text
    it_letter = _col_letter(ws, "item_text")
    if it_letter:
        for r in range(2, ws.max_row + 1):
            ws[f"{it_letter}{r}"].alignment = ALIGN_WRAP

    # Number formats
    for c in ["cv_c_value", "cv_d_value",
              "pfa_atomic_single_loading", "pfa_atomic_reversed_single_loading",
              "pfa_atomic_communality", "pfa_atomic_primary_loading",
              "pfa_atomic_max_sec_loading", "pfa_atomic_within_keying_loading",
              "pfa_macro_item_to_scale_sim",
              "discriminant_target_loading", "discriminant_max_cross_loading",
              "discriminant_target_to_max_ratio",
              "pfa_scale_rmsr",
              "cv_mean_target_rating", "cv_std_target_rating",
              "cv_scale_mapping_agreement", "cv_majority_agreement"]:
        _set_number_format(ws, c, "0.000")
    _set_number_format(ws, "flesch_kincaid_grade", "0.0")

    # Pass level — green / amber / red
    _cond_equal_text(ws, "pfa_pass_level", "HIGH_PASS",     FILL_PASS_STRONG)
    _cond_equal_text(ws, "pfa_pass_level", "STANDARD_PASS", FILL_WARN)
    _cond_equal_text(ws, "pfa_pass_level", "FAIL",          FILL_FAIL_STRONG)

    # Rule_N PASS/FAIL text cells
    for rule in ["Rule_1_Strong_Single_Factor", "Rule_2_Adequate_Single_Factor",
                 "Rule_3_Adequate_Communality", "Rule_4_Clean_Factor_Structure",
                 "Rule_5_Cross_Sample_Viable",  "Rule_6_Keying_Viable"]:
        _cond_equal_text(ws, rule, "PASS", FILL_PASS_LIGHT)
        _cond_equal_text(ws, rule, "FAIL", FILL_FAIL)

    # CV thresholds — flag cells that are below threshold
    _cond_lt(ws, "cv_c_value", T_CVC, FILL_FAIL)
    _cond_lt(ws, "cv_d_value", T_CVD, FILL_FAIL)

    # Loadings heatmap (sign-aware: negatives show red, positives green)
    for c in ["pfa_atomic_single_loading",
              "pfa_atomic_reversed_single_loading",
              "pfa_atomic_primary_loading",
              "pfa_atomic_within_keying_loading"]:
        _cond_color_scale_abs(ws, c)

    # Communality and macro similarity are bounded at 0; use sim palette
    _cond_color_scale_sim(ws, "pfa_macro_item_to_scale_sim")
    # Communality is 0..1; use a simple 2-colour ramp (white → green)
    comm_rng = _data_range(ws, _col_letter(ws, "pfa_atomic_communality"))
    if comm_rng:
        ws.conditional_formatting.add(
            comm_rng,
            ColorScaleRule(
                start_type="num", start_value=0.00, start_color="FFFFFF",
                end_type="num",   end_value=0.40,   end_color="C6EFCE",
            )
        )
    # Max secondary loading: smaller is better, so red-at-high
    maxsec_rng = _data_range(ws, _col_letter(ws, "pfa_atomic_max_sec_loading"))
    if maxsec_rng:
        ws.conditional_formatting.add(
            maxsec_rng,
            ColorScaleRule(
                start_type="num", start_value=0.00, start_color="FFFFFF",
                end_type="num",   end_value=T_R4,   end_color="F8CBAD",
            )
        )

    # Discriminant flag: TRUE cells red
    _cond_equal_text(ws, "discriminant_flag_review", "True", FILL_FAIL)
    _cond_equal_text(ws, "discriminant_flag_review", "TRUE", FILL_FAIL)

    # Discriminant ratio: below threshold is a problem
    _cond_lt(ws, "discriminant_target_to_max_ratio", T_DISC_R, FILL_FAIL)
    _cond_gt(ws, "discriminant_max_cross_loading",   T_DISC_C, FILL_FAIL)

    # FK grade — above soft target amber, above hard cap red
    _cond_gt(ws, "flesch_kincaid_grade", T_FK_CAP, FILL_FAIL)
    _cond_gt(ws, "flesch_kincaid_grade", T_FK_TGT, FILL_WARN)  # will be overlaid by above where applicable

    # Bias flag column: any non-empty value gets amber highlight
    bfl = _col_letter(ws, "bias_flagged_dims")
    if bfl:
        rng = _data_range(ws, bfl)
        ws.conditional_formatting.add(
            rng,
            FormulaRule(formula=[f'LEN(TRIM({bfl}2))>0'], fill=FILL_WARN)
        )

    # DAAL identity: FALSE cells red
    _cond_equal_text(ws, "daal_identity_confirmed", "False", FILL_FAIL)
    _cond_equal_text(ws, "daal_identity_confirmed", "FALSE", FILL_FAIL)

    # CV maps-to-target: FALSE amber
    _cond_equal_text(ws, "cv_maps_to_target", "False", FILL_WARN)
    _cond_equal_text(ws, "cv_maps_to_target", "FALSE", FILL_WARN)

    # item_keying shading — subtle differentiation (not red/green, just muted)
    _cond_equal_text(ws, "item_keying", "negative", FILL_INFO)

    # Mark reviewer input columns
    _mark_input_columns(ws, ["human_review_pass", "human_comments"])


def style_PFA_Scale_Summary(ws) -> None:
    """
    One row per scale. First stop for the reviewer. Anchors:
      • pass_rate_pct color scale
      • daal_identity_confirmed green/red
      • mean_scale_rmsr — green if acceptable, red if above threshold
      • disc_flag_count red if > 0
    """
    _apply_header_and_filter(ws)
    _set_col_widths(ws, {
        "scale_name":                22,
    })

    _set_number_format(ws, "pass_rate_pct",             "0.0")
    for c in ["mean_c_value", "mean_d_value", "mean_atomic_loading",
              "mean_communality", "mean_macro_sim",
              "mean_cv_rating", "mean_cv_agreement",
              "mean_scale_rmsr"]:
        _set_number_format(ws, c, "0.000")

    # Pass rate — color scale 0–100%
    pr_letter = _col_letter(ws, "pass_rate_pct")
    if pr_letter:
        ws.conditional_formatting.add(
            _data_range(ws, pr_letter),
            ColorScaleRule(
                start_type="num", start_value=0,   start_color="F8CBAD",
                mid_type="num",   mid_value=50,    mid_color="FFEB9C",
                end_type="num",   end_value=100,   end_color="C6EFCE",
            )
        )

    # DAAL
    _cond_equal_text(ws, "daal_identity_confirmed", "True",  FILL_PASS_LIGHT)
    _cond_equal_text(ws, "daal_identity_confirmed", "False", FILL_FAIL)

    # Scale RMSR — above threshold red
    _cond_gt(ws, "mean_scale_rmsr", T_RMSR, FILL_FAIL)

    # CV thresholds on means
    _cond_lt(ws, "mean_c_value", T_CVC, FILL_WARN)
    _cond_lt(ws, "mean_d_value", T_CVD, FILL_WARN)

    # Counts: fail/disc flag columns — >0 amber
    for c in ["fail", "disc_flag_count"]:
        _cond_gt(ws, c, 0, FILL_WARN)
    # n_reversed_for_embedding is informational, not a fault — use info colour
    nr = _col_letter(ws, "n_reversed_for_embedding")
    if nr:
        ws.conditional_formatting.add(
            _data_range(ws, nr),
            CellIsRule(operator="greaterThan", formula=["0"], fill=FILL_INFO)
        )


def style_Discriminant_Matrix(ws) -> None:
    """
    Item × factor loading matrix. Reviewer wants: across each row, the biggest
    value should be on the item's target scale. Apply a row-oriented color
    scale to every loading_on_* column so the peak stands out.
    """
    _apply_header_and_filter(ws)
    _set_col_widths(ws, {"item_id": 14, "scale_name": 22, "item_text": 55})
    it = _col_letter(ws, "item_text")
    if it:
        for r in range(2, ws.max_row + 1):
            ws[f"{it}{r}"].alignment = ALIGN_WRAP

    for cell in ws[1]:
        name = str(cell.value or "")
        if name.startswith("loading_on_"):
            col_letter = get_column_letter(cell.column)
            rng = _data_range(ws, col_letter)
            if rng is None:
                continue
            for r in range(2, ws.max_row + 1):
                ws[f"{col_letter}{r}"].number_format = "0.000"
            ws.conditional_formatting.add(
                rng,
                ColorScaleRule(
                    start_type="num", start_value=-0.8, start_color="F8CBAD",
                    mid_type="num",   mid_value=0.0,    mid_color="FFFFFF",
                    end_type="num",   end_value=0.8,    end_color="C6EFCE",
                )
            )


def style_Item_Scale_Heatmap(ws) -> None:
    """Item × scale cosine similarity. Bigger values green, small red."""
    _apply_header_and_filter(ws)
    _set_col_widths(ws, {
        "item_id": 14, "scale_name": 22, "item_text": 55,
        "item_keying": 12, "pfa_pass_level": 16,
    })
    it = _col_letter(ws, "item_text")
    if it:
        for r in range(2, ws.max_row + 1):
            ws[f"{it}{r}"].alignment = ALIGN_WRAP

    _cond_equal_text(ws, "pfa_pass_level", "HIGH_PASS",     FILL_PASS_STRONG)
    _cond_equal_text(ws, "pfa_pass_level", "STANDARD_PASS", FILL_WARN)
    _cond_equal_text(ws, "pfa_pass_level", "FAIL",          FILL_FAIL_STRONG)
    _cond_equal_text(ws, "item_keying",    "negative",      FILL_INFO)

    for cell in ws[1]:
        name = str(cell.value or "")
        if name.startswith("sim_"):
            col_letter = get_column_letter(cell.column)
            for r in range(2, ws.max_row + 1):
                ws[f"{col_letter}{r}"].number_format = "0.000"
            _cond_color_scale_sim(ws, name)


def style_Loading_Matrix(ws) -> None:
    """Item × scale FA loadings. Same treatment as Discriminant_Matrix columns."""
    _apply_header_and_filter(ws)
    _set_col_widths(ws, {
        "item_id": 14, "scale_name": 22, "item_text": 55, "item_keying": 12,
    })
    it = _col_letter(ws, "item_text")
    if it:
        for r in range(2, ws.max_row + 1):
            ws[f"{it}{r}"].alignment = ALIGN_WRAP

    for cell in ws[1]:
        name = str(cell.value or "")
        if name.startswith("loading_"):
            col_letter = get_column_letter(cell.column)
            for r in range(2, ws.max_row + 1):
                ws[f"{col_letter}{r}"].number_format = "0.000"
            rng = _data_range(ws, col_letter)
            if rng:
                ws.conditional_formatting.add(
                    rng,
                    ColorScaleRule(
                        start_type="num", start_value=-0.8, start_color="F8CBAD",
                        mid_type="num",   mid_value=0.0,    mid_color="FFFFFF",
                        end_type="num",   end_value=0.8,    end_color="C6EFCE",
                    )
                )
    # in_pfa_solution = FALSE means projected — mark amber
    _cond_equal_text(ws, "in_pfa_solution", "False", FILL_WARN)
    _cond_equal_text(ws, "in_pfa_solution", "FALSE", FILL_WARN)


def style_Review_Pool(ws) -> None:
    """Final shortlist. Reviewer inputs are THE feature — make them obvious."""
    _apply_header_and_filter(ws)
    _set_col_widths(ws, {
        "item_id": 14, "scale_name": 22, "item_text": 55, "item_keying": 12,
        "pfa_pass_level": 16,
        "human_review_pass": 12, "human_comments": 40,
        "human_decision":    14, "human_notes":    40,
        "encoding_strategy_primary": 16,
    })
    it = _col_letter(ws, "item_text")
    if it:
        for r in range(2, ws.max_row + 1):
            ws[f"{it}{r}"].alignment = ALIGN_WRAP

    for c in ["cv_c_value", "cv_d_value", "cv_mean_target_rating",
              "pfa_atomic_single_loading", "pfa_atomic_reversed_single_loading",
              "pfa_macro_item_to_scale_sim", "pfa_atomic_max_sec_loading"]:
        _set_number_format(ws, c, "0.000")
    _set_number_format(ws, "flesch_kincaid_grade", "0.0")

    _cond_equal_text(ws, "pfa_pass_level", "HIGH_PASS",     FILL_PASS_STRONG)
    _cond_equal_text(ws, "pfa_pass_level", "STANDARD_PASS", FILL_WARN)
    _cond_equal_text(ws, "pfa_pass_level", "FAIL",          FILL_FAIL_STRONG)
    _cond_equal_text(ws, "item_keying",    "negative",      FILL_INFO)

    _cond_lt(ws, "cv_c_value", T_CVC, FILL_FAIL)
    _cond_lt(ws, "cv_d_value", T_CVD, FILL_FAIL)

    _cond_equal_text(ws, "discriminant_flag_review", "True", FILL_FAIL)
    _cond_equal_text(ws, "discriminant_flag_review", "TRUE", FILL_FAIL)
    _cond_equal_text(ws, "daal_identity_confirmed",  "False", FILL_FAIL)
    _cond_equal_text(ws, "daal_identity_confirmed",  "FALSE", FILL_FAIL)

    # Human decision visual cues
    _cond_equal_text(ws, "human_decision", "accept", FILL_PASS_STRONG)
    _cond_equal_text(ws, "human_decision", "modify", FILL_WARN)
    _cond_equal_text(ws, "human_decision", "reject", FILL_FAIL_STRONG)

    _mark_input_columns(ws, ["human_review_pass", "human_comments",
                             "human_decision",    "human_notes"])


def style_Pool_Statistics(ws) -> None:
    _apply_header_and_filter(ws)
    _set_col_widths(ws, {"scale_name": 22})

    for c in ["mean_c_value", "mean_d_value", "mean_atomic_loading",
              "mean_macro_sim", "mean_cv_rating", "mean_cv_agreement"]:
        _set_number_format(ws, c, "0.000")
    _set_number_format(ws, "mean_fk_grade", "0.0")

    # Zero in either keying column → red; low counts → amber
    for c in ["positive_keyed_items", "negative_keyed_items"]:
        col = _col_letter(ws, c)
        if col is None:
            continue
        rng = _data_range(ws, col)
        ws.conditional_formatting.add(
            rng, CellIsRule(operator="equal",  formula=["0"], fill=FILL_FAIL_STRONG))
        ws.conditional_formatting.add(
            rng, CellIsRule(operator="between", formula=["1", "4"], fill=FILL_WARN))

    _cond_gt(ws, "items_with_bias_flags",       0, FILL_WARN)
    _cond_gt(ws, "discriminant_flags",          0, FILL_WARN)
    _cond_gt(ws, "items_reversed_for_embedding", 0, FILL_INFO)

    _cond_equal_text(ws, "daal_identity_confirmed", "True",  FILL_PASS_LIGHT)
    _cond_equal_text(ws, "daal_identity_confirmed", "False", FILL_FAIL)

    _cond_gt(ws, "mean_fk_grade", T_FK_TGT, FILL_WARN)
    _cond_lt(ws, "mean_c_value",  T_CVC,   FILL_WARN)
    _cond_lt(ws, "mean_d_value",  T_CVD,   FILL_WARN)


def style_Quality_Flags(ws) -> None:
    _apply_header_and_filter(ws)
    _set_col_widths(ws, {
        "item_id": 14, "scale_name": 22, "item_text": 55,
        "item_keying": 12, "pfa_pass_level": 16, "flags": 60,
    })
    for c in ["item_text", "flags"]:
        letter = _col_letter(ws, c)
        if letter:
            for r in range(2, ws.max_row + 1):
                ws[f"{letter}{r}"].alignment = ALIGN_WRAP

    _cond_equal_text(ws, "pfa_pass_level", "HIGH_PASS",     FILL_PASS_STRONG)
    _cond_equal_text(ws, "pfa_pass_level", "STANDARD_PASS", FILL_WARN)
    _cond_equal_text(ws, "pfa_pass_level", "FAIL",          FILL_FAIL_STRONG)

    # flag_count ramp
    fc = _col_letter(ws, "flag_count")
    if fc:
        ws.conditional_formatting.add(
            _data_range(ws, fc),
            ColorScaleRule(
                start_type="num", start_value=1, start_color="FFEB9C",
                end_type="num",   end_value=5,   end_color="F4B084",
            )
        )


def style_Recommendations(ws) -> None:
    _apply_header_and_filter(ws)
    _set_col_widths(ws, {"scale_name": 22, "priority": 10, "recommendation": 80})
    rec = _col_letter(ws, "recommendation")
    if rec:
        for r in range(2, ws.max_row + 1):
            ws[f"{rec}{r}"].alignment = ALIGN_WRAP

    _cond_equal_text(ws, "priority", "HIGH",   FILL_FAIL_STRONG)
    _cond_equal_text(ws, "priority", "MEDIUM", FILL_WARN)
    _cond_equal_text(ws, "priority", "LOW",    FILL_PASS_LIGHT)


def style_Rejected_Items(ws) -> None:
    _apply_header_and_filter(ws)
    _set_col_widths(ws, {
        "item_id": 14, "scale_name": 22, "item_text": 55,
        "item_keying": 12, "pfa_pass_level": 16,
    })
    it = _col_letter(ws, "item_text")
    if it:
        for r in range(2, ws.max_row + 1):
            ws[f"{it}{r}"].alignment = ALIGN_WRAP
    _cond_equal_text(ws, "pfa_pass_level", "FAIL", FILL_FAIL)


def style_Fit_Statistics(ws) -> None:
    _apply_header_and_filter(ws)
    _set_col_widths(ws, {"scale_name": 22, "encoding": 30,
                         "construct_definition": 60, "transformer_models": 30})
    for c in ["rmsr", "residual_mean", "residual_std", "residual_max_abs"]:
        _set_number_format(ws, c, "0.0000")
    _cond_gt(ws, "rmsr", T_RMSR, FILL_FAIL)
    _cond_equal_text(ws, "fit_acceptable", "True",  FILL_PASS_LIGHT)
    _cond_equal_text(ws, "fit_acceptable", "False", FILL_FAIL)
    _cond_equal_text(ws, "daal_identity_confirmed", "True",  FILL_PASS_LIGHT)
    _cond_equal_text(ws, "daal_identity_confirmed", "False", FILL_FAIL)


def style_Eigenvalues(ws) -> None:
    _apply_header_and_filter(ws)
    _set_col_widths(ws, {"scale_name": 22, "encoding": 30})
    _set_number_format(ws, "eigenvalue",           "0.000")
    _set_number_format(ws, "cumulative_variance",  "0.0%")
    _cond_equal_text(ws, "above_1", "Yes", FILL_PASS_LIGHT)


def style_Pass_Rules_Definition(ws) -> None:
    _apply_header_and_filter(ws)
    _set_col_widths(ws, {"rule_id": 32, "description": 70,
                         "threshold": 12, "operator": 10})
    desc = _col_letter(ws, "description")
    if desc:
        for r in range(2, ws.max_row + 1):
            ws[f"{desc}{r}"].alignment = ALIGN_WRAP


def style_High_All_Pass(ws) -> None:
    """High_Pass_Loadings and All_Pass_Loadings — compact filtered views."""
    _apply_header_and_filter(ws)
    _set_col_widths(ws, {
        "item_id": 14, "scale_name": 22, "item_text": 55,
        "item_keying": 12, "encoding_strategy_primary": 16,
    })
    it = _col_letter(ws, "item_text")
    if it:
        for r in range(2, ws.max_row + 1):
            ws[f"{it}{r}"].alignment = ALIGN_WRAP

    for c in ["cv_c_value", "cv_d_value", "cv_mean_target_rating",
              "pfa_atomic_single_loading", "pfa_atomic_reversed_single_loading",
              "pfa_macro_item_to_scale_sim", "pfa_atomic_max_sec_loading"]:
        _set_number_format(ws, c, "0.000")
    _set_number_format(ws, "flesch_kincaid_grade", "0.0")

    _cond_lt(ws, "cv_c_value", T_CVC, FILL_FAIL)
    _cond_lt(ws, "cv_d_value", T_CVD, FILL_FAIL)
    for c in ["pfa_atomic_single_loading", "pfa_atomic_reversed_single_loading"]:
        _cond_color_scale_abs(ws, c)
    _cond_equal_text(ws, "discriminant_flag_review", "True", FILL_FAIL)
    _cond_equal_text(ws, "discriminant_flag_review", "TRUE", FILL_FAIL)
    _cond_equal_text(ws, "item_keying",              "negative", FILL_INFO)


# ==============================================================================
# APPLY
# ==============================================================================
SHEET_STYLERS = {
    "PFA_Item_Results":       style_PFA_Item_Results,
    "PFA_Scale_Summary":      style_PFA_Scale_Summary,
    "Discriminant_Matrix":    style_Discriminant_Matrix,
    "Item_Scale_Heatmap":     style_Item_Scale_Heatmap,
    "Loading_Matrix_All":     style_Loading_Matrix,
    "Loading_Matrix_CV_Pass": style_Loading_Matrix,
    "Fit_Statistics":         style_Fit_Statistics,
    "Eigenvalues":            style_Eigenvalues,
    "High_Pass_Loadings":     style_High_All_Pass,
    "All_Pass_Loadings":      style_High_All_Pass,
    "Pass_Rules_Definition":  style_Pass_Rules_Definition,
    "Review_Pool":            style_Review_Pool,
    "Rejected_Items":         style_Rejected_Items,
    "Pool_Statistics":        style_Pool_Statistics,
    "Quality_Flags":          style_Quality_Flags,
    "Recommendations":        style_Recommendations,
}

wb = load_workbook(P5_OUT)
applied = 0
for sheet_name, styler in SHEET_STYLERS.items():
    if sheet_name in wb.sheetnames:
        try:
            styler(wb[sheet_name])
            applied += 1
            print(f"  ✓ styled {sheet_name}")
        except Exception as e:
            logger.warning(f"Styling failed for {sheet_name}: {e}")
    else:
        print(f"  – skipped {sheet_name} (not in workbook)")

# ── Sheet ordering: put the highest-value sheets first ────────────────────────
preferred_order = [
    "Recommendations",
    "PFA_Scale_Summary",
    "Quality_Flags",
    "Review_Pool",
    "Pool_Statistics",
    "PFA_Item_Results",
    "Item_Scale_Heatmap",
    "Discriminant_Matrix",
    "Loading_Matrix_CV_Pass",
    "Loading_Matrix_All",
    "Fit_Statistics",
    "Eigenvalues",
    "High_Pass_Loadings",
    "All_Pass_Loadings",
    "Rejected_Items",
    "Pass_Rules_Definition",
]
existing_order = wb.sheetnames[:]
new_order      = [s for s in preferred_order if s in existing_order]
new_order     += [s for s in existing_order   if s not in new_order]
if new_order != existing_order:
    wb._sheets = [wb[name] for name in new_order]

wb.save(P5_OUT)

print(f"\n✅ Formatting pass complete — {applied} sheets styled.")
print(f"   Thresholds visualised: "
      f"c>={T_CVC}, d>={T_CVD}, RMSR<={T_RMSR}, "
      f"Rule_1>={T_R1}, Rule_4<{T_R4}, disc_ratio>={T_DISC_R}")
print(f"   Reviewer input cells (yellow): human_review_pass, human_comments, "
      f"human_decision, human_notes")
print(f"   Sheet order reflects recommended review sequence.")
print(f"📄 Output: {P5_OUT}")